# FDS(이상거래탐지) 모델링 통합 노트북 — 팀 기존 작업과 개선 작업

## 이 노트북은 무엇입니까

이 노트북은 신용카드 이상거래탐지(FDS, Fraud Detection System) 팀 프로젝트의 모델링 과정을 **하나의 파일에서 처음부터 끝까지 재현**하는 문서입니다.
사용 데이터는 공개 데이터셋인 IEEE-CIS Fraud Detection이며, 재현하는 과정은 다음과 같습니다.
피처 엔지니어링, 월 단위 시간 기반 검증, XGBoost 모델 학습, 임계치와 비용 기준 결정.

프로젝트 종료 후 작성자(이수인)는 두 가지 출처에서 개선 항목을 도출했습니다.
하나는 프로젝트 발표 평가(심사위원 피드백)이고, 다른 하나는 팀 코드를 직접 검토하여 찾은 미흡한 점입니다.
각 개선 항목은 이 노트북에서 실제로 실행하여 검증했고, 성능이 오르지 않은 항목도 기각 사유와 함께 그대로 기록했습니다.

## 누가 무엇을 했습니까

| 구분 | 담당 | 내용 | 노트북 표시 |
|---|---|---|---|
| 팀 기존 부분 | 프로젝트 팀원 전체 | 팀이 프로젝트 기간에 작성한 데이터 로딩, 피처 엔지니어링, 기준 모델 학습 코드입니다. 로직을 바꾸지 않고 재현하며, 코드 이해를 위한 주석만 추가했습니다. | 🟦 파란색 |
| 개선 부분 | 이수인(작성자) | 심사 피드백과 팀 코드의 미흡한 점을 바탕으로 작성자가 새로 설계하고 실행한 모든 검증, 실험, 분석입니다. | 🟩 초록색 |
| 안내 | 이수인(작성자) | 설정, 실행 방법, 해석 안내입니다. | ⬜ 회색 |

팀 기존 부분과 개선 부분은 모든 섹션에서 아래 배너로 구분합니다. 모든 코드 셀의 첫 줄에도 같은 표시가 있습니다.
(`# 🟦 [팀 기존 …]`, `# 🟩 [수인 개선 · 개선 항목 이름]`, `# ⬜ [안내]`)

<div style="border-left:6px solid #1f77b4;background:#eef5fb;padding:8px 14px;margin:4px 0"><b>🟦 [팀 기존]</b> 팀이 프로젝트 기간에 이미 수행한 작업입니다. 로직을 바꾸지 않고 그대로 재현합니다. (출처 파일을 함께 표기)</div>

<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 개선 항목 이름]</b> 이수인이 이번에 새로 설계하고 실행한 개선입니다. 결과는 아래 <b>채택/기각 기준</b>으로 자동 판정합니다.</div>

<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 실행 방법, 설정, 해석 안내입니다.</div>

## 읽는 순서

1. 아래 **개선 항목 목록**에서 어떤 피드백과 미흡한 점을 어떻게 다뤘는지 확인합니다.
2. **채택/기각 기준**은 실행 전에 고정한 판정 규칙입니다. 결과를 본 뒤 변경하지 않았습니다.
3. 이후 섹션은 데이터 로딩, 팀 기존 피처 엔지니어링(🟦), 각 개선 항목 실험(🟩), 마지막 판정표 순서입니다.

## 개선 항목 목록 (전부 실행)

**심사 피드백 기반**

| 개선 항목 | 심사 피드백 (출처: 평가 취합 PDF) | 이 노트북의 개선 |
|---|---|---|
| 미래 정보 혼입 제거 (학습 구간 기준 피처 생성) | "학습 데이터 생성 과정에 미래 정보가 섞였을 가능성을 다시 확인하고 성능을 재제시" (개선·사전질문 4) | fold 내부 피처 생성(UID 집계·빈도 인코딩을 학습 구간 통계로만 계산) + 순방향 검증 |
| 임계치 통합표·권장 임계치 | "기준값이 0.0329 / 0.35 / 0.017로 다르게 제시, 하나로 정리" (개선·질문 3·총평) | 임계치 후보를 한 표에 정리하고 한 가지 평가표본·한 가지 비용식으로 권장 임계치 1개 확정 |
| 비용 가정 민감도 분석 | "$5.43 가정값에 대한 민감도 분석(2배·0.5배)" (개선) | 비용·회수율·용량 가정별 최적 임계치 민감도 |
| 변수 의미 사전·사유 코드·V 변수 제외 모델 | "핵심 근거 변수 의미를 알기 어려움 → 의미를 알 수 있는 변수 위주 설명" (개선·질문 5) | 변수 의미 사전 + 거래별 사유 코드 + 해석 가능 변수 모델 비교 |
| 3단계(자동 승인/추가 인증/수동 심사) 정책 시뮬레이션 | "중위험 구간 별도 확인 절차 구체화" (질문 1) | 3단계 정책(자동 승인/추가 인증/수동 심사) 비용 시뮬레이션과 손익분기 |
| 추론 지연시간·피처 축소(Top-K) | "321개 피처의 실시간 계산 지연시간 검토" (질문 2) | 추론 지연시간 측정 + 피처 축소(Top-K) 성능·지연 비교 |
| 지역·결제환경 의존 피처 제거 실험 | "국내 환경 적용 시 달라질 점 논의" (개선) | 지역·결제환경 의존 피처 그룹 제거 실험 + 논의 문단 |
| 룰 선처리 + 모델 후처리 | "룰로 먼저 처리하고 나머지를 AI로 넘기는 시도" (현장 질문 5) | 학습 구간에서 룰 추출 → 룰 선처리 + 모델 후처리 비교 |
| 상대 시각·요일 피처 추가 / 시간 의존 피처 제거 영향 분석 | "시간 변수를 채택한 고려" (현장 질문 4) | 시간 파생 피처 추가 실험과 시간 의존 피처 제거 실험 |
| 변수 선택 단계별 근거표 | "변수 선택에 대한 충분한 설명" (현장 질문 2) | 변수 선택 단계별 근거표(자동 집계) |
| 데이터 분할 표와 사용 방식 명시 | "Test set을 어떻게 분류해 사용했는가" (현장 질문 3) | 데이터 분할 표와 사용 방식 명시 |
| 가설과 모델 근거 연결 (접속·기기 변수군 제거 실험) | "가설 기각·채택의 근거" (현장 질문 1) | 통합 EDA 표시본의 가설 판정 셀과 이 노트북의 모델 검증을 연결하는 대응표 |

**팀 코드에서 직접 찾은 미흡한 점**

| 개선 항목 | 발견한 미흡한 점 (근거 파일) | 이 노트북의 개선 |
|---|---|---|
| 내부 검증 기반 early stopping | `baseline_groupkfold_tc.py`는 early stopping에 **검증 fold 자체**를 사용하므로 검증 성능이 낙관적일 수 있음 | 학습 구간의 마지막 달을 내부 검증으로 쓰는 early stopping |
| 예측 확률 교차 보정 | 비용 기반 임계치는 예측 확률이 보정되어 있어야 신뢰할 수 있으나, 최종 모델은 보정을 하지 않음 (SUMMARY_DRAFT.md에서도 한계로 언급) | fold 교차 보정(isotonic) 후 Brier·ECE와 임계치 영향 비교 |
| 과거 이력 기반 속도 피처 | UID 집계는 통계적 평균 위주이고 **거래 순서·속도 정보(직전 거래와의 간격, 최근 1h/24h 건수)** 가 없음 | 과거 이력만 사용하는 속도(velocity) 피처 추가 |
| LightGBM 단독 / XGBoost+LightGBM 평균 앙상블 | MODELING_REPORT.md 한계: "단일 모델(XGBoost)만 검증, 앙상블 미시도" | LightGBM 추가 + XGBoost 평균 앙상블 |
| 하이퍼파라미터 재검토 | 하이퍼파라미터 튜닝이 경량 설정(500트리)으로만 이루어져 기각됨 | 같은 스크리닝 설정으로 기준 모델과 후보를 공정 비교하고, 채택 시 공식 설정으로 재검증 |

<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 채택/기각 기준 (결과를 보기 전에 고정)]</b> 아래 기준은 실행 전에 정한 것이며, 결과를 본 뒤 바꾸지 않습니다. 마지막 셀의 판정표는 이 기준을 코드로 적용한 결과입니다.</div>

**공통 원칙**: 주 지표는 OOF PR-AUC(보조: ROC-AUC)입니다. 비교는 반드시 **같은 fold, 같은 설정의 기준 모델**과 합니다.
개선이 성능을 올리지 못하면 **"개선 시도, 기각"** 으로 그대로 기록합니다.

| 판정 종류 | 해당 개선 항목 | 판정 기준 |
|---|---|---|
| 평가 타당성 수정 | 미래 정보 혼입 제거, 내부 검증 기반 early stopping, 예측 확률 교차 보정 | 성능이 떨어져도 **채택**합니다. 평가 방식을 더 정직하게 바꾸는 것이 목적이므로 하락폭을 그대로 보고합니다. |
| 성능 개선 후보 | 과거 이력 기반 속도 피처, LightGBM 단독, XGBoost+LightGBM 평균 앙상블, 하이퍼파라미터 재검토, 상대 시각·요일 피처 추가 | ΔPR-AUC ≥ **+0.003** 이고 비교 fold의 **2/3 이상에서 PR-AUC가 개선**되면 **채택**, 아니면 **기각**합니다. |
| 경량화 | 추론 지연시간·피처 축소(Top-K) | ΔPR-AUC ≥ **−0.005** 이고 단건 추론 지연시간이 **30% 이상 감소**하면 **채택**합니다. |
| 분석·보고 | 임계치 통합표, 비용 가정 민감도 분석, 변수 의미 사전·사유 코드, 3단계 정책 시뮬레이션, 지역·결제환경 의존 피처 제거 실험, 시간 의존 피처 제거 영향 분석, 변수 선택 근거표, 데이터 분할 명시, 가설과 모델 근거 연결 | 채택/기각이 아니라 **수행 결과 보고**입니다. 수치는 모두 실행 결과이며 가정은 가정으로 표시합니다. |
| 룰 선처리 | 룰 선처리 + 모델 후처리 | 같은 Recall에서 경고 건수가 **3% 이상 감소**하고 Recall 손실이 없으면 **채택**, 아니면 **기각**합니다. |

<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 실행 방법</div>

1. 이 파일은 `FDS 수인 개선` 폴더에 두고 **프로젝트 `.venv`(Python 3.13)** 커널로 실행합니다. 데이터 경로는 아래 설정 셀에서 자동 확인합니다.
2. **위에서 아래로 순서대로** 실행합니다. 오래 걸리는 단계는 결과를 `수인_개선_결과/` 폴더에 저장하므로 **중간에 멈춰도 다시 실행하면 이어서** 진행됩니다.
3. 먼저 `SMOKE = True`로 한 번 실행하여 오류가 없는지 확인하고(수 분), 이후 `SMOKE = False`로 본 실행을 합니다.
4. 예상 소요 시간(CPU 코어 수·메모리에 따라 크게 달라짐): 피처 엔지니어링 약 10~30분, 공식 설정 6-fold 1회 약 30분~2시간, 스크리닝(가벼운 설정) 변형 1개 약 10~30분.
   전체 실행은 수 시간~반나절 이상 걸릴 수 있으므로, 설정 셀의 `RUN` 스위치로 필요한 항목만 켜고 끌 수 있습니다.
5. 메모리가 부족하면 `LOW_MEMORY = True`로 설정합니다(float32로 읽어 메모리를 줄이며, 수치가 아주 약간 달라질 수 있습니다).

<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 0. 설정과 경로</div>

In [1]:
# ⬜ [안내] 설정 · 경로 · 스위치
import os  # 환경 변수 조회
import sys  # 파이썬 실행 환경 정보
import gc  # 가비지 컬렉션 수동 호출(메모리 회수)
import time  # 단계별 소요 시간 측정
import json  # 로그·설정의 JSON 저장/로딩
import math  # 수학 함수
import warnings  # 경고 메시지 제어
import datetime as dt  # 날짜 처리
from pathlib import Path  # 경로 객체
import numpy as np  # 수치 배열 연산
import pandas as pd  # 표 형식 데이터 처리
import xgboost as xgb  # 그래디언트 부스팅 모델
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss  # 평가 지표
from sklearn.model_selection import GroupKFold  # 그룹 단위 교차 검증 분할
warnings.filterwarnings("ignore")  # 경고 메시지 숨김
pd.set_option("display.width", 200)  # 출력 표 너비
pd.set_option("display.max_columns", 60)  # 출력 최대 컬럼 수
pd.set_option("display.max_colwidth", 80)  # 셀 값 최대 표시 길이

SMOKE = False  # True: 소규모·가벼운 설정으로 코드 점검만 수행(수 분)
LOW_MEMORY = False  # True: float32로 읽어 메모리 절약
SEED = 42  # 난수 시드(재현성)

# 실행 스위치: False로 끄면 해당 섹션을 건너뜀(이전 결과가 저장되어 있으면 불러옴)
RUN = dict(  # 섹션별 실행 여부 딕셔너리
    baseline=True,  # 팀 기존: 기준 모델 재현
    fold_features=True,  # 미래 정보 혼입 제거와 순방향 검증
    inner_early_stopping=True,  # 내부 검증 기반 early stopping
    velocity=True,  # 과거 이력 기반 속도 피처
    time_features=True,  # 시간 관련 피처 실험
    lgb_ensemble=True,  # LightGBM 단독, XGBoost+LightGBM 앙상블
    tuning=True,  # 하이퍼파라미터 재검토(스크리닝 설정 내 비교)
    explainability=True,  # 설명 가능성(변수 의미 사전, 사유 코드)
    region_dependence=True,  # 지역·결제환경 의존 피처 제거 실험
    latency_topk=True,  # 추론 지연시간과 피처 축소(Top-K)
    final=True,  # 채택 항목을 합친 최종 모델(공식 설정 6-fold)
    threshold_policy=True,  # 임계치·비용 민감도·확률 보정·3단계 정책
    rule_prefilter=True,  # 룰 선처리 + 모델 후처리
)  # RUN 딕셔너리 끝

# 경로: 이 노트북이 있는 폴더 또는 아래 절대경로에서 data/train_transaction.csv를 탐색
_env = os.environ.get("FDS_ROOT")  # 환경 변수로 지정한 프로젝트 루트(없으면 None)
CANDIDATES = ([Path(_env)] if _env else []) + [  # 후보 경로: 환경 변수 경로, 현재 폴더, 상위 폴더, 고정 경로
    Path.cwd(),  # 현재 작업 폴더
    Path.cwd().parent,  # 현재 작업 폴더의 상위 폴더
]  # 후보 경로 목록 끝
ROOT = next((p for p in CANDIDATES if (p / "data" / "train_transaction.csv").exists()), None)  # 데이터가 있는 첫 후보
assert ROOT is not None, "data/train_transaction.csv를 찾지 못했습니다. 노트북이 'FDS 수인 개선' 폴더에 있는지 확인하십시오."  # 경로 확인
DATA = ROOT / "data"  # 원본 데이터 폴더
TEAM_DIR = ROOT / "ieee-cis-modeling-main"  # 팀 코드·문서 폴더
TEAM_OOF = TEAM_DIR / "data" / "final_oof_predictions.parquet"  # 팀 최종 OOF 예측(팀 산출물)
TEAM_TC = TEAM_DIR / "data" / "time_consistency_results.csv"  # 팀 Time Consistency 결과(팀 산출물)
OUT = ROOT / ("수인_개선_결과_SMOKE" if SMOKE else "수인_개선_결과")  # 결과 저장 폴더(SMOKE 모드는 별도 폴더)
CK = OUT / "ckpt"  # 체크포인트 폴더
CK.mkdir(parents=True, exist_ok=True)  # 체크포인트 폴더 생성(있으면 유지)
print("ROOT :", ROOT)  # 프로젝트 루트 출력
print("DATA :", DATA)  # 데이터 폴더 출력
print("OUT  :", OUT)  # 결과 폴더 출력
print("팀 OOF 파일 존재:", TEAM_OOF.exists(), "| 팀 TC 결과 존재:", TEAM_TC.exists())  # 팀 산출물 존재 여부 출력
print("xgboost", xgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)  # 라이브러리 버전 출력

ROOT : <작업 폴더>/FDS 수인 개선
DATA : <작업 폴더>/FDS 수인 개선\data
OUT  : <작업 폴더>/FDS 수인 개선\수인_개선_결과
팀 OOF 파일 존재: True | 팀 TC 결과 존재: True
xgboost 3.4.1 | pandas 3.0.5 | numpy 2.5.2


In [2]:
# ⬜ [안내] 공통 상수: 공식 설정(팀 설정 그대로)과 스크리닝 설정(가벼운 비교용)
# 🟦 [팀 기존 · 출처: src/baseline_groupkfold_tc.py] 공식 하이퍼파라미터
XGB_OFFICIAL = dict(  # 팀 공식 XGBoost 파라미터
    objective="binary:logistic",  # 이진 분류 확률 출력
    eval_metric="auc",  # 조기 종료에 쓰는 평가 지표
    max_depth=12,  # 트리 최대 깊이
    eta=0.02,  # 학습률
    subsample=0.8,  # 트리별 행 샘플링 비율
    colsample_bytree=0.4,  # 트리별 열 샘플링 비율
    tree_method="hist",  # 히스토그램 기반 분할 탐색
    max_bin=256,  # 히스토그램 구간 수
    nthread=-1,  # 가용 CPU 코어 전부 사용
    seed=0,  # 모델 난수 시드
)  # 공식 파라미터 끝
ROUNDS_OFFICIAL, ES_OFFICIAL = 2000, 100  # 최대 부스팅 횟수, early stopping 대기 횟수
# 🟩 [수인 개선 · 하이퍼파라미터 재검토] 스크리닝 설정: 같은 구조에서 학습률을 키우고 트리 수를 줄여 여러 변형을 같은 조건으로 비교
XGB_SCREEN = dict(XGB_OFFICIAL, max_depth=8, eta=0.08)  # 스크리닝용 파라미터(깊이 8, eta 0.08)
ROUNDS_SCREEN, ES_SCREEN = 500, 50  # 스크리닝용 최대 트리 수, early stopping 대기 횟수
if SMOKE:  # 코드 점검 모드이면 아주 가벼운 설정으로 교체
    XGB_OFFICIAL = dict(XGB_OFFICIAL, max_depth=4, eta=0.3)  # 공식 설정을 얕은 트리·큰 학습률로 축소
    ROUNDS_OFFICIAL, ES_OFFICIAL = 40, 10  # 공식 설정의 트리 수와 대기 횟수 축소
    XGB_SCREEN = dict(XGB_SCREEN, max_depth=4, eta=0.3)  # 스크리닝 설정을 얕은 트리·큰 학습률로 축소
    ROUNDS_SCREEN, ES_SCREEN = 30, 10  # 스크리닝 설정의 트리 수와 대기 횟수 축소

TC_FAILED = ["D1_was_missing", "V286", "V107", "V305"]  # 🟦 팀 Time Consistency Test 탈락 피처
START_DATE = pd.Timestamp("2017-11-30")  # 🟦 팀 코드와 동일한 월 계산 기준일
N_SPLITS = 6  # 교차 검증 fold 수
SCREEN_FOLDS = [1, 3, 5]  # 스크리닝 비교에 쓰는 fold(팀 OOF 기준 월 15, 17, 16). 공식 평가는 6개 모두 사용
FORWARD_MONTHS = [14, 15, 16, 17]  # 순방향 검증에서 평가할 월
print("공식 설정:", XGB_OFFICIAL, ROUNDS_OFFICIAL, ES_OFFICIAL)  # 공식 설정 출력
print("스크리닝 설정:", XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN)  # 스크리닝 설정 출력

공식 설정: {'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 12, 'eta': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.4, 'tree_method': 'hist', 'max_bin': 256, 'nthread': -1, 'seed': 0} 2000 100
스크리닝 설정: {'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 8, 'eta': 0.08, 'subsample': 0.8, 'colsample_bytree': 0.4, 'tree_method': 'hist', 'max_bin': 256, 'nthread': -1, 'seed': 0} 500 50


<div style="border-left:6px solid #1f77b4;background:#eef5fb;padding:8px 14px;margin:4px 0"><b>🟦 [팀 기존]</b> 1. 데이터 로딩 — 출처: <code>src/baseline_groupkfold.py</code>의 <code>load()</code> (경로만 수인 폴더로 지정)</div>

In [3]:
# 🟦 [팀 기존 · 출처: src/baseline_groupkfold.py load()] — 로직 동일, 경로만 DATA 변수로 지정
def load():  # 거래 테이블과 identity 테이블을 읽어 병합
    kw = dict(dtype_backend=None)  # 팀 원본에 있던 미사용 변수(로직 영향 없음)
    tr = pd.read_csv(DATA / "train_transaction.csv", index_col="TransactionID")  # 거래 테이블 로딩
    ti = pd.read_csv(DATA / "train_identity.csv", index_col="TransactionID")  # identity 테이블 로딩
    if LOW_MEMORY:  # 메모리 절약 모드이면
        for d in (tr, ti):  # 두 테이블 각각에 대해
            for c in d.select_dtypes("float64").columns: d[c] = d[c].astype("float32")  # float64 컬럼을 float32로 변환
    df = tr.merge(ti, how="left", left_index=True, right_index=True)  # 거래 기준 left join(TransactionID 인덱스)
    del tr, ti; gc.collect()  # 원본 테이블 삭제 후 메모리 회수
    return df  # 병합 결과 반환

t0 = time.time()  # 로딩 시작 시각
df_raw = load()  # 원본 데이터 로딩·병합
if SMOKE: pass  # 팀 원본 구조 유지(동작 없음)
y_all = df_raw["isFraud"].astype("int8").values  # 전체 정답 라벨 배열
df_raw = df_raw.drop(columns=["isFraud"])  # 입력 데이터에서 정답 컬럼 제거
print("원본 병합 shape:", df_raw.shape, f"({time.time()-t0:.0f}s) | 사기 비율 {y_all.mean():.4%}")  # 데이터 크기·사기 비율 출력

원본 병합 shape: (590540, 432) (14s) | 사기 비율 3.4990%


<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 월(DT_M) 계산과 fold 구성. 팀의 <code>GroupKFold(6, groups=월)</code>과 같은 분할을 쓰기 위해, 팀 OOF 파일이 있으면 그 파일의 월→fold 대응을 그대로 사용합니다.</div>

In [4]:
# 🟦 [팀 기존 · 출처: baseline_groupkfold_tc.py] 월 그룹 계산 (벡터화만 다름, 결과 동일)
_dt = START_DATE + pd.to_timedelta(df_raw["TransactionDT"].astype("int64"), unit="s")  # 기준일 + 경과 초 = 거래 일시
DT_M = ((_dt.dt.year - 2017) * 12 + _dt.dt.month).to_numpy()  # 2017년 기준 월 번호 배열
del _dt  # 임시 변수 삭제
MONTHS = sorted(set(DT_M.tolist()))  # 데이터에 존재하는 월 번호 목록
print("월 분포:", pd.Series(DT_M).value_counts().sort_index().to_dict())  # 월별 거래 건수 출력

# fold 대응: 팀 OOF 파일이 있으면 그대로 사용, 없으면 팀과 같은 GroupKFold 호출
if TEAM_OOF.exists() and not SMOKE:  # 팀 OOF 파일이 있고 점검 모드가 아니면
    _t = pd.read_parquet(TEAM_OOF)  # 팀 OOF 예측 파일 로딩
    month_to_fold = _t.groupby("month")["fold"].first().to_dict()  # 월별 fold 번호 대응표
    print("팀 OOF의 월→fold 대응 사용:", month_to_fold)  # 사용한 대응표 출력
else:  # 팀 OOF를 쓸 수 없으면
    _gkf = GroupKFold(n_splits=N_SPLITS)  # 팀과 같은 월 그룹 분할기
    month_to_fold = {}  # 월→fold 대응표 초기화
    for f, (_, iv) in enumerate(_gkf.split(np.zeros(len(DT_M)), y_all, groups=DT_M)):  # fold별 검증 인덱스 순회
        month_to_fold[int(DT_M[iv[0]])] = f  # 검증 인덱스의 월을 해당 fold에 대응
    print("GroupKFold로 계산한 월→fold 대응:", month_to_fold)  # 계산한 대응표 출력
FOLD_OF_ROW = np.array([month_to_fold[int(m)] for m in DT_M], dtype="int8")  # 행별 fold 번호
fold_to_month = {f: m for m, f in month_to_fold.items()}  # fold→월 역대응표
FOLD_IDX = {f: np.where(FOLD_OF_ROW == f)[0] for f in range(N_SPLITS)}  # fold별 행 인덱스

월 분포: {12: 137321, 13: 92585, 14: 86021, 15: 101632, 16: 83655, 17: 89326}
팀 OOF의 월→fold 대응 사용: {12: 0, 13: 2, 14: 4, 15: 1, 16: 5, 17: 3}


<div style="border-left:6px solid #1f77b4;background:#eef5fb;padding:8px 14px;margin:4px 0"><b>🟦 [팀 기존]</b> 2. 팀 피처 엔지니어링 (원본 그대로) — 출처: <code>src/baseline_groupkfold_tc.py</code> <code>engineer_features()</code>. 로직·순서를 바꾸지 않았고, 단계별 피처 수 기록용 <code>LOG</code>와 한국어 주석만 추가했습니다.</div>

팀 방식의 핵심은 **UID 집계 피처와 빈도 인코딩을 전체 데이터에서 한 번에 계산**한다는 점입니다. 이 점이 심사위원이 지적한 "학습 데이터 생성 과정의 미래 정보 혼입 가능성"과 직접 관련됩니다. (뒤의 '미래 정보 혼입 제거' 항목에서 검증합니다.)

In [5]:
# 🟦 [팀 기존 · 출처: src/baseline_groupkfold_tc.py engineer_features()] — 로직 동일 (LOG 기록만 추가)
LOG = {}  # 단계별 컬럼 수 기록용 딕셔너리(수인 추가)
def team_engineer_features(df):  # 팀 피처 엔지니어링 함수
    LOG["입력 컬럼 수"] = df.shape[1]  # 입력 컬럼 수 기록
    # 1. identity 결측 지표
    df["has_identity"] = df["id_01"].notna().astype("int8")  # identity 정보 존재 여부 지표
    id_cols = [c for c in df.columns if c.startswith("id_")]  # identity 컬럼 목록
    for c in id_cols:  # identity 컬럼별로
        miss_rate = df[c].isna().mean()  # 결측 비율
        if 0.2 <= miss_rate <= 0.8:  # 결측 비율이 20~80%이면 결측 여부 자체가 정보
            df[c + "_is_missing"] = df[c].isna().astype("int8")  # 결측 지표 컬럼 추가
    # 2. D 컬럼 정규화
    for i in range(1, 16):  # D1~D15에 대해
        if i in (1, 2, 3, 5, 9):  # 정규화에서 제외하는 컬럼
            continue  # 다음 컬럼으로
        c = f"D{i}"  # 컬럼 이름
        if c in df.columns:  # 컬럼이 존재하면
            df[c] = df[c] - df["TransactionDT"] / np.float32(24 * 60 * 60)  # 거래 시점(일)을 빼서 시간 추세 제거
    # 3. TransactionAmt cents
    df["cents"] = (df["TransactionAmt"] - np.floor(df["TransactionAmt"])).astype("float32")  # 거래 금액의 소수부
    # 4. 범주형 라벨 인코딩
    obj_cols = [c for c in df.columns if df[c].dtype == "object" or df[c].dtype.name == "str"]  # 문자열 컬럼 목록
    for c in obj_cols:  # 문자열 컬럼별로
        df[c], _ = df[c].factorize(sort=True)  # 정렬된 정수 코드로 변환(결측은 -1)
        df[c] = df[c].astype("int32")  # int32로 변환
    # 5. 빈도 인코딩
    for c in ["addr1", "card1", "card2", "card3", "P_emaildomain"]:  # 빈도 인코딩 대상 컬럼
        vc = df[c].value_counts(dropna=True, normalize=True)  # 값별 상대 빈도(전체 데이터 기준)
        df[c + "_FE"] = df[c].map(vc).astype("float32")  # 상대 빈도 피처 추가
    # 6. UID 생성 -> 집계 피처 생성 -> UID 원본은 드롭
    day = df["TransactionDT"] / np.float32(24 * 60 * 60)  # 경과 일수
    df["D1_was_missing"] = df["D1"].isna().astype("int8")  # D1 결측 지표
    d1_filled = df["D1"].fillna(0)  # D1 결측을 0으로 대체
    df["uid"] = (  # 카드·주소·최초 거래일 추정값을 결합한 사용자 식별자
        df["card1"].astype(str) + "_" +  # card1 문자열
        df["addr1"].fillna(-999).astype(str) + "_" +  # addr1 문자열(결측은 -999)
        np.floor(day - d1_filled).astype(str)  # 거래일에서 D1을 뺀 추정 최초 거래일
    )  # uid 정의 끝
    for col in ["TransactionAmt", "D4", "D9", "D10", "D15"]:  # 평균·표준편차 집계 대상 컬럼
        if col not in df.columns:  # 컬럼이 없으면
            continue  # 건너뜀
        grp = df.groupby("uid")[col]  # uid별 그룹
        df[f"{col}_uid_mean"] = grp.transform("mean").astype("float32")  # uid별 평균(전체 데이터 기준)
        df[f"{col}_uid_std"] = grp.transform("std").astype("float32")  # uid별 표준편차(전체 데이터 기준)
    for col in [f"C{i}" for i in range(1, 15) if i != 3]:  # 평균 집계 대상 C 컬럼(C3 제외)
        if col not in df.columns:  # 컬럼이 없으면
            continue  # 건너뜀
        df[f"{col}_uid_mean"] = df.groupby("uid")[col].transform("mean").astype("float32")  # uid별 평균
    df["uid_FE"] = df["uid"].map(df["uid"].value_counts(normalize=True)).astype("float32")  # uid 상대 빈도
    df["uid_tx_count"] = df.groupby("uid")["uid"].transform("count").astype("float32")  # uid별 거래 건수
    df = df.drop(columns=["uid"])  # uid 원본 컬럼 제거
    LOG["UID·빈도 집계 후 컬럼 수"] = df.shape[1]  # 집계 후 컬럼 수 기록
    # 7. V 컬럼 정제
    v_cols = [c for c in df.columns if c.startswith("V")]  # V 컬럼 목록
    if v_cols:  # V 컬럼이 있으면
        v_sample = df[v_cols].fillna(-1)  # 결측을 -1로 대체
        corr = v_sample.corr().abs()  # 컬럼 간 상관계수 절댓값
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))  # 상삼각 행렬만 남겨 중복 쌍 제거
        to_drop = [col for col in upper.columns if (upper[col] > 0.95).any()]  # 상관 0.95 초과 컬럼 목록
        df = df.drop(columns=to_drop)  # 상관이 높은 중복 V 컬럼 제거
        LOG["V 컬럼 수(전)"] = len(v_cols)  # 제거 전 V 컬럼 수 기록
        LOG["V 상관 0.95+ 제거 수"] = len(to_drop)  # 제거한 V 컬럼 수 기록
        LOG["V 컬럼 유지 수"] = len(v_cols) - len(to_drop)  # 유지한 V 컬럼 수 기록
        print(f"V 컬럼 {len(v_cols)}개 중 상관관계 0.95+ 중복 {len(to_drop)}개 제거, {len(v_cols) - len(to_drop)}개 유지")  # 'V 컬럼 …개 중 상관관계 0.95+ 중복 …개 제거, …개 유지' 출력
    df = df.replace([np.inf, -np.inf], np.nan)  # 무한대 값을 결측으로 변환
    return df  # 피처 엔지니어링 결과 반환

In [6]:
# 🟦 [팀 기존] 팀 피처 엔지니어링 실행 (결과는 체크포인트 저장 → 다음 실행부터 즉시 불러옴)
_ck_team = CK / "team_features.parquet"  # 팀 피처 체크포인트 파일 경로
if _ck_team.exists():  # 체크포인트가 있으면
    TEAM_DF = pd.read_parquet(_ck_team)  # 저장된 팀 피처 로딩
    LOG.update(json.loads((CK / "team_features_log.json").read_text(encoding="utf-8")))  # 저장된 단계별 로그 복원
    print("체크포인트에서 팀 피처 로딩:", TEAM_DF.shape)  # 로딩한 shape 출력
else:  # 체크포인트가 없으면
    t0 = time.time()  # 피처 엔지니어링 시작 시각
    TEAM_DF = team_engineer_features(df_raw.copy())  # 원본 복사본에 팀 피처 엔지니어링 적용
    TEAM_DF = TEAM_DF.drop(columns=[c for c in TC_FAILED if c in TEAM_DF.columns])  # Time Consistency 탈락 피처 제거
    TEAM_DF.to_parquet(_ck_team)  # 팀 피처 체크포인트 저장
    (CK / "team_features_log.json").write_text(json.dumps(LOG, ensure_ascii=False), encoding="utf-8")  # 단계별 로그 저장
    print(f"팀 피처 엔지니어링 완료 ({time.time()-t0:.0f}s):", TEAM_DF.shape)  # 완료 시간·shape 출력
TEAM_FEATURES = [c for c in TEAM_DF.columns if c not in ("TransactionDT", "DT_M")]  # 모델 입력 피처 이름 목록
print("팀 최종 피처 수:", len(TEAM_FEATURES), "(팀 보고서: 321개)")  # 피처 수 출력(팀 보고서 값과 대조)
X_TEAM = TEAM_DF[TEAM_FEATURES].to_numpy(dtype=np.float32, copy=True)  # 이후 모든 변형의 기준 행렬(float32)

체크포인트에서 팀 피처 로딩: (590540, 322)
팀 최종 피처 수: 321 (팀 보고서: 321개)


<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 3. 공통 학습·검증 함수 — 모든 변형이 동일한 함수와 동일한 fold로 비교되도록 하나로 통합하였습니다. 학습 방식은 팀 코드(XGBoost, <code>missing=-1</code>, 검증 AUC 기준 early stopping)와 동일하며, 메모리 사용량을 줄이기 위해 <code>xgb.train</code>과 <code>QuantileDMatrix</code>를 사용합니다(팀 <code>_stage_c.py</code>와 동일한 방식).</div>

In [7]:
# ⬜ [안내] 공통 함수 (학습 · 평가 · 체크포인트 · 판정 기록)
import pickle  # 체크포인트(fold별 결과) 직렬화에 사용


def _qdm(X, y, names, mb, ref=None):  # QuantileDMatrix 생성 헬퍼
    # 결측값은 -1로 지정하고, 검증용 행렬은 학습용 행렬의 분위수 경계(ref)를 공유한다
    return xgb.QuantileDMatrix(X, label=y, missing=-1, feature_names=names, max_bin=mb, ref=ref)


def fit_predict_xgb(Xtr, ytr, Xva, yva, names: list, params: dict, rounds: int, es: int,
                    es_mode: str = "valid", mtr=None, keep_model: bool = False):  # XGBoost 학습 후 검증 구간 예측
    # es_mode: 'valid'    = 팀 방식(검증 fold로 early stopping),
    # 'inner'    = 내부 검증 기반 early stopping: 학습 구간의 마지막 달을 내부 검증으로 사용해
    # 트리 수를 정한 뒤 전체 학습 구간으로 재학습,
    # 'innerfit' = 'inner'에서 재학습 없이 내부 검증 모델을 그대로 예측에 사용(스크리닝용, 비교 일관성 목적)
    p = dict(params)  # 호출자의 파라미터 딕셔너리를 변경하지 않도록 복사
    mb = p.get("max_bin", 256)  # 분위수 경계 수(지정하지 않으면 256)
    if es_mode == "valid":  # 팀 방식: 검증 fold를 early stopping에 사용
        dtr = _qdm(Xtr, ytr, names, mb)  # 학습용 행렬
        dva = _qdm(Xva, yva, names, mb, ref=dtr)  # 검증용 행렬(학습용과 동일한 분위수 경계)
        bst = xgb.train(p, dtr, rounds, evals=[(dva, "valid")], early_stopping_rounds=es, verbose_eval=False)  # 학습
        best = bst.best_iteration + 1  # 검증 AUC가 최고인 트리 수
        pred = bst.predict(dva, iteration_range=(0, best))  # 최적 트리 수까지만 사용해 예측
    else:  # 내부 검증 방식
        inner = mtr == mtr.max()  # 학습 구간의 마지막 달 행을 내부 검증 행으로 지정
        d_fit = _qdm(Xtr[~inner], ytr[~inner], names, mb)  # 내부 학습용 행렬
        d_es = _qdm(Xtr[inner], ytr[inner], names, mb, ref=d_fit)  # 내부 검증용 행렬
        bst0 = xgb.train(p, d_fit, rounds, evals=[(d_es, "es")], early_stopping_rounds=es, verbose_eval=False)  # 학습
        best0 = bst0.best_iteration + 1  # 내부 검증 AUC가 최고인 트리 수
        if es_mode == "innerfit":  # 재학습 없이 내부 검증 모델을 그대로 사용
            bst, best = bst0, best0  # 내부 검증 모델과 트리 수 채택
            dva = _qdm(Xva, yva, names, mb, ref=d_fit)  # 검증용 행렬(내부 학습용과 동일한 분위수 경계)
            pred = bst.predict(dva, iteration_range=(0, best))  # 최적 트리 수까지만 사용해 예측
        else:  # 전체 학습 구간으로 재학습
            best = int(best0 * 1.1) + 1  # 학습 데이터가 늘어난 만큼 트리 수를 10% 늘려 적용
            dtr = _qdm(Xtr, ytr, names, mb)  # 전체 학습 구간 행렬
            bst = xgb.train(p, dtr, best, verbose_eval=False)  # 정해진 트리 수로 재학습
            dva = _qdm(Xva, yva, names, mb, ref=dtr)  # 검증용 행렬
            pred = bst.predict(dva)  # 재학습 모델로 예측
    imp = bst.get_score(importance_type="gain")  # 피처별 gain 중요도
    return pred, best, imp, (bst if keep_model else None)  # 예측값, 트리 수, 중요도, (선택) 모델


def cv_run(tag: str, specs: list, params: dict, rounds: int, es: int, build, es_mode: str = "valid",
           force: bool = False, fit_fn=None):  # fold 목록을 순회하며 학습·평가하고 체크포인트에 저장
    # specs: [{'key','tr','va'}] — 결과는 CK/cv_{tag}.pkl에 fold 단위로 저장(중단 후 재실행하면 이어서 진행).
    path = CK / f"cv_{tag}.pkl"  # 태그별 체크포인트 파일 경로
    if path.exists() and not force:  # 기존 체크포인트가 있으면 이어서 진행
        with open(path, "rb") as f:  # 체크포인트 파일 열기
            store = pickle.load(f)  # 저장된 fold 결과 불러오기
    else:  # 체크포인트가 없거나 강제 재실행
        store = {"folds": {}}  # 빈 결과 저장소 생성
    for sp in specs:  # fold별 반복
        k = str(sp["key"])  # fold 식별자(문자열)
        if k in store["folds"]:  # 이미 계산된 fold는 건너뜀
            continue  # 다음 fold로 이동
        t0 = time.time()  # fold 소요 시간 측정 시작
        Xtr, ytr, Xva, yva, names, mtr = build(sp)  # 이 fold의 학습·검증 행렬 생성
        if fit_fn is None:  # 기본 학습 함수 사용
            pred, best, imp, _ = fit_predict_xgb(Xtr, ytr, Xva, yva, names, params, rounds, es, es_mode, mtr)  # XGBoost
        else:  # 사용자 지정 학습 함수 사용(예: LightGBM)
            pred, best, imp = fit_fn(Xtr, ytr, Xva, yva, names, mtr)  # 지정된 함수로 학습·예측
        store["folds"][k] = dict(  # fold 결과 기록
            va=sp["va"], pred=pred.astype("float32"), best=int(best), imp=imp,  # 검증 행 인덱스, 예측값, 트리 수, 중요도
            sec=time.time() - t0, n_feat=len(names),  # 소요 시간(초), 피처 수
            roc=roc_auc_score(yva, pred), pr=average_precision_score(yva, pred),  # ROC-AUC, PR-AUC
        )
        with open(path, "wb") as f:  # fold 하나를 마칠 때마다 체크포인트 저장
            pickle.dump(store, f)  # 직렬화하여 기록
        r = store["folds"][k]  # 방금 저장한 fold 결과
        print(f"[{tag}] fold {k}: ROC {r['roc']:.4f} PR {r['pr']:.4f} | 트리 {r['best']} | "
              f"피처 {r['n_feat']} | {r['sec']:.0f}s", flush=True)  # fold별 결과 출력
        del Xtr, Xva  # 대용량 행렬 참조 해제
        gc.collect()  # 메모리 회수
    RESULTS[tag] = store  # 전역 결과 저장소에 등록
    return store  # 전체 fold 결과 반환


def summarize(tag: str, keys=None) -> dict:  # 지정한 fold의 예측을 합쳐 전체 성능을 계산
    st = RESULTS[tag]["folds"]  # 해당 태그의 fold별 결과
    ks = [k for k in st if keys is None or k in keys]  # 대상 fold 선택(지정하지 않으면 전체)
    va = np.concatenate([st[k]["va"] for k in ks])  # 대상 fold의 검증 행 인덱스 결합
    pr_ = np.concatenate([st[k]["pred"] for k in ks])  # 대상 fold의 예측값 결합
    return dict(  # 요약 결과
        tag=tag, n_folds=len(ks),  # 태그와 fold 수
        roc=roc_auc_score(y_all[va], pr_), pr=average_precision_score(y_all[va], pr_),  # 결합 ROC-AUC, PR-AUC
        fold_pr={k: st[k]["pr"] for k in ks}, fold_roc={k: st[k]["roc"] for k in ks},  # fold별 PR-AUC, ROC-AUC
        sec=sum(st[k]["sec"] for k in ks),  # 총 학습 시간(초)
    )


def compare(tag: str, ref: str, quiet: bool = False) -> dict:  # 같은 fold에서 tag를 기준(ref)과 비교
    # 같은 fold에서 tag를 ref(기준)와 비교. ΔPR-AUC, ΔROC-AUC, PR 개선 fold 비율.
    common = [k for k in RESULTS[tag]["folds"] if k in RESULTS[ref]["folds"]]  # 두 결과에 모두 있는 fold
    a, b = summarize(tag, common), summarize(ref, common)  # 비교 대상과 기준의 요약
    wins = sum(a["fold_pr"][k] > b["fold_pr"][k] for k in common) / len(common)  # PR-AUC가 개선된 fold 비율
    out = dict(  # 비교 결과
        tag=tag, ref=ref, n_folds=len(common), d_pr=a["pr"] - b["pr"], d_roc=a["roc"] - b["roc"], win_frac=wins,  # 차이
        pr=a["pr"], roc=a["roc"], ref_pr=b["pr"], ref_roc=b["roc"],  # 비교 대상과 기준의 절대 성능
    )
    if not quiet:  # 출력 억제 옵션이 꺼져 있으면
        print(f"{tag} vs {ref} ({len(common)} fold) | PR {a['pr']:.4f} vs {b['pr']:.4f} (Δ{out['d_pr']:+.4f}) | "
              f"ROC {a['roc']:.4f} vs {b['roc']:.4f} (Δ{out['d_roc']:+.4f}) | PR 개선 fold {wins:.0%}")  # 비교 요약 출력
    return out  # 비교 결과 반환


# ── 채택/기각 판정 기록 (노트북 마지막의 판정표에서 사용). 판정 기준은 노트북 앞부분에서 고정한 값입니다.
RESULTS = {}  # 태그별 교차 검증 결과
VERDICTS = {}  # 항목 id별 판정 기록


def add_verdict(id_: str, name: str, typ: str, verdict: str, detail: str, **metrics):  # 판정 한 건을 기록하고 저장
    VERDICTS[id_] = {"id": id_, "항목": name, "판정 종류": typ, "판정": verdict, "근거": detail, **metrics}  # 판정 기록
    verdicts_json = json.dumps(VERDICTS, ensure_ascii=False, indent=1, default=float)  # JSON 직렬화(numpy 수치는 float 변환)
    (OUT / "verdicts.json").write_text(verdicts_json, encoding="utf-8")  # 판정 기록 파일 갱신
    print(f"▶ {name}: **{verdict}** — {detail}")  # 판정 결과 출력


def judge_B(cmp: dict) -> str:  # 성능 개선 후보 기준: ΔPR-AUC ≥ +0.003 이고 fold의 2/3 이상에서 개선
    return "채택" if (cmp["d_pr"] >= 0.003 and cmp["win_frac"] >= 2 / 3) else "기각"  # 기준 충족 여부로 판정


# 학습·검증 fold 정의 (월 단위 그룹 분할)
SPEC_GROUP = {f: dict(key=f, tr=np.where(FOLD_OF_ROW != f)[0], va=FOLD_IDX[f]) for f in range(N_SPLITS)}  # fold별 행
SPECS_ALL = [SPEC_GROUP[f] for f in range(N_SPLITS)]  # 전체 fold 목록
SPECS_SCREEN = [SPEC_GROUP[f] for f in SCREEN_FOLDS]  # 스크리닝에 사용하는 fold 목록


def spec_forward(m: int) -> dict:  # 순방향 검증 fold: 월 m 미만으로 학습하고 월 m을 검증
    return dict(key=f"fw{m}", tr=np.where(DT_M < m)[0], va=np.where(DT_M == m)[0])  # 과거 월은 학습, 해당 월은 검증


SPECS_FWD = [spec_forward(m) for m in FORWARD_MONTHS]  # 순방향 검증 fold 목록
COL = {n: i for i, n in enumerate(TEAM_FEATURES)}  # 피처 이름 → 열 위치
screen_months = [fold_to_month[f] for f in SCREEN_FOLDS]  # 스크리닝 fold에 해당하는 월
print("fold→월:", fold_to_month, "| 스크리닝 fold:", SCREEN_FOLDS, "→ 월", screen_months)  # fold 구성 출력

fold→월: {0: 12, 2: 13, 4: 14, 1: 15, 5: 16, 3: 17} | 스크리닝 fold: [1, 3, 5] → 월 [15, 17, 16]


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 미래 정보 혼입 제거]</b> 학습 구간 기준 피처 생성 — 분할 전 전체 데이터 통계의 영향 확인</div>

**확인한 문제**: 팀 코드는 UID 집계(`*_uid_mean/std`, `uid_FE`, `uid_tx_count`)와 빈도 인코딩(`*_FE`)을 **fold 분할 전 전체 데이터**로 계산합니다(위 `team_engineer_features`의 5·6단계).
이 때문에 검증 월(미래)의 거래가 학습 행의 집계 피처 값에 반영되고, 검증 행의 피처도 자기 자신과 미래 거래를 포함하여 계산됩니다. 심사 과정에서 미래 정보 혼입 가능성으로 지적된 부분입니다.
(팀 `SUMMARY_DRAFT.md`에도 동일한 한계가 기재되어 있습니다: "일부 빈도 및 UID 집계 피처도 fold 분할 전에 계산되어 검증 월의 분포가 피처에 반영될 수 있다".)

**개선**: 해당 7종류(빈도 인코딩 5개, UID 집계·빈도·건수)를 **학습 구간 행만으로 계산한 통계**로 대체합니다. 검증 행은 학습 구간에서 계산된 통계를 조회만 하며, 학습 구간에 없던 UID는 결측으로 처리합니다.
그 밖의 피처(행 단위 변환, 결측 지표, 소수점 금액 등)와 하이퍼파라미터는 팀 설정과 완전히 동일합니다.

In [8]:
# 🟩 [수인 개선 · 미래 정보 혼입 제거] fold 내부 피처 생성 함수 (팀 5·6단계와 같은 수식, 통계 계산 구간만 학습 행으로 제한)
RAW_COLS = ["card1", "addr1", "card2", "card3", "P_emaildomain", "D1", "TransactionDT", "TransactionAmt"]  # 원본 값이 필요한 열
RAW = {c: TEAM_DF[c].to_numpy(dtype="float64") for c in RAW_COLS}  # 열별 원본 값 배열
_day = RAW["TransactionDT"] / np.float32(24 * 60 * 60)  # 기준 시점 이후 경과 일수(팀과 동일한 계산)
_uid_str = (  # UID 문자열: card1 + addr1 + 추정 기준일
    pd.Series(RAW["card1"]).astype(str) + "_"  # 카드 관련 코드(익명화된 값)
    + pd.Series(RAW["addr1"]).fillna(-999).astype(str) + "_"  # 청구지 주소(결측은 -999)
    + np.floor(_day - pd.Series(RAW["D1"]).fillna(0)).astype(str)  # 거래일에서 D1을 뺀 추정 기준일
)
UID_CODE, _uniq = pd.factorize(_uid_str)  # UID 문자열을 정수 코드로 변환
K_UID = len(_uniq)  # 서로 다른 UID의 수
del _uid_str, _uniq  # 중간 객체 해제
AGG_COLS = [c for c in ["TransactionAmt", "D4", "D9", "D10", "D15"] if c in TEAM_DF.columns]  # 평균·표준편차 집계 대상
AGG_STD = set(AGG_COLS)  # 표준편차까지 계산하는 열의 집합
C_COLS = [f"C{i}" for i in range(1, 15) if i != 3 and f"C{i}" in TEAM_DF.columns]  # 평균만 집계하는 C 계열 열
AGG_VALUES = {c: TEAM_DF[c].to_numpy(dtype="float64") for c in AGG_COLS + C_COLS}  # 집계 대상 열의 원본 값
FE_BASE = ["addr1", "card1", "card2", "card3", "P_emaildomain"]  # 빈도 인코딩 대상 열
CROSS_NAMES = (  # fold 내부에서 다시 계산해야 하는 피처 이름 전체
    [c + "_FE" for c in FE_BASE]  # 빈도 인코딩 피처
    + [f"{c}_uid_{s}" for c in AGG_COLS for s in ("mean", "std")]  # UID별 평균·표준편차 피처
    + [f"{c}_uid_mean" for c in C_COLS]  # C 계열 UID별 평균 피처
    + ["uid_FE", "uid_tx_count"]  # UID 빈도, UID 거래 건수
)
CROSS_NAMES = [n for n in CROSS_NAMES if n in COL]  # 팀 최종 피처에 실제로 있는 것만 남김
print("fold 내부 계산 대상 피처:", len(CROSS_NAMES), "개 |", CROSS_NAMES[:6], "...")  # 대상 피처 수와 일부 이름 출력


def _grp_mean_std(codes, vals, K):  # 그룹별 평균·표준편차 계산
    # 그룹 평균·표준편차(ddof=1, pandas groupby.std와 동일).
    # 2-pass(편차 제곱합)로 계산하여 값이 모두 같은 그룹에서 표준편차가 정확히 0이 되도록 한다.
    m = ~np.isnan(vals)  # 결측이 아닌 값의 마스크
    c = codes[m]  # 결측 제외 그룹 코드
    v = vals[m]  # 결측 제외 값
    cnt = np.bincount(c, minlength=K).astype("float64")  # 그룹별 유효 값 개수
    s1 = np.bincount(c, weights=v, minlength=K)  # 그룹별 합
    with np.errstate(invalid="ignore", divide="ignore"):  # 0으로 나누기 경고 억제(결과는 NaN 처리)
        mean = np.where(cnt > 0, s1 / cnt, np.nan)  # 그룹 평균(값이 없는 그룹은 NaN)
        dev = v - mean[c]  # 각 값의 그룹 평균 대비 편차
        ss = np.bincount(c, weights=dev * dev, minlength=K)  # 그룹별 편차 제곱합
        std = np.where(cnt > 1, np.sqrt(ss / (cnt - 1)), np.nan)  # 표본 표준편차(값이 2개 미만이면 NaN)
    return mean, std  # 그룹별 평균, 표준편차


def cross_row_stats(train_idx) -> dict:  # 학습 행만으로 통계를 계산해 전체 행의 피처 값을 생성
    # train_idx 행만으로 통계를 계산해 '전체 행'의 7종류 피처 값을 반환 (dict name → float32[N]).
    out = {}  # 피처 이름 → 값 배열
    n_tr = len(train_idx)  # 학습 행 수
    for c in FE_BASE:  # 빈도 인코딩 열별 반복
        vc = pd.Series(RAW[c][train_idx]).value_counts(dropna=True, normalize=True)  # 학습 행 기준 값별 비율
        out[c + "_FE"] = pd.Series(RAW[c]).map(vc).to_numpy(dtype="float32")  # 전체 행에 비율 매핑(학습에 없던 값은 NaN)
    ctr = UID_CODE[train_idx]  # 학습 행의 UID 코드
    for c in AGG_COLS:  # 평균·표준편차 집계 열별 반복
        mean, std = _grp_mean_std(ctr, AGG_VALUES[c][train_idx], K_UID)  # 학습 행 기준 UID별 평균·표준편차
        out[f"{c}_uid_mean"] = mean[UID_CODE].astype("float32")  # 전체 행에 UID별 평균 조회
        out[f"{c}_uid_std"] = std[UID_CODE].astype("float32")  # 전체 행에 UID별 표준편차 조회
    for c in C_COLS:  # 평균만 집계하는 열별 반복
        mean, _ = _grp_mean_std(ctr, AGG_VALUES[c][train_idx], K_UID)  # 학습 행 기준 UID별 평균
        out[f"{c}_uid_mean"] = mean[UID_CODE].astype("float32")  # 전체 행에 UID별 평균 조회
    cnt = np.bincount(ctr, minlength=K_UID).astype("float64")  # 학습 행 기준 UID별 거래 건수
    seen = cnt[UID_CODE] > 0  # 학습 구간에 등장한 UID인지 여부
    out["uid_FE"] = np.where(seen, cnt[UID_CODE] / n_tr, np.nan).astype("float32")  # UID 빈도(비율), 미등장 UID는 NaN
    out["uid_tx_count"] = np.where(seen, cnt[UID_CODE], np.nan).astype("float32")  # UID 거래 건수, 미등장 UID는 NaN
    return {k: v for k, v in out.items() if k in COL}  # 팀 최종 피처에 있는 것만 반환


# 동등성 검증: 학습 구간을 전체 데이터로 지정하면 팀 피처와 같은 값이 나와야 한다(구현 오류 점검)
_chk = cross_row_stats(np.arange(len(DT_M)))  # 전체 행을 학습 구간으로 간주하여 계산
_bad = []  # 불일치 항목 목록
for n, v in _chk.items():  # 피처별 비교
    t = X_TEAM[:, COL[n]]  # 팀 피처 값
    same_nan = np.array_equal(np.isnan(v), np.isnan(t))  # 결측 위치가 동일한지 여부
    both = ~np.isnan(v) & ~np.isnan(t)  # 양쪽 모두 값이 있는 행
    close = np.allclose(v[both], t[both], rtol=1e-3, atol=1e-3)  # float32 저장 오차를 허용한 값 일치 여부
    if not (same_nan and close):  # 결측 위치 또는 값이 다르면
        err = np.abs(v[both] - t[both])  # 절대 오차
        n_over = int((err > 1e-3 + 1e-3 * np.abs(t[both])).sum())  # 허용 오차를 초과한 행 수
        _bad.append((n, bool(same_nan), float(err.max()) if err.size else 0.0, n_over))  # 불일치 정보 기록
print("동등성 검증:", "통과 (팀 피처와 일치)" if not _bad else
      f"불일치 (이름, NaN 위치 일치, 최대 절대 오차, 허용 초과 행 수): {_bad[:8]}")  # 검증 결과 출력
assert not _bad, "fold 내부 피처 구현이 팀 피처와 다릅니다. 위 목록을 확인하십시오."  # 불일치 시 실행 중단
del _chk  # 검증용 배열 해제
gc.collect()  # 메모리 회수

fold 내부 계산 대상 피처: 30 개 | ['addr1_FE', 'card1_FE', 'card2_FE', 'card3_FE', 'P_emaildomain_FE', 'TransactionAmt_uid_mean'] ...
동등성 검증: 통과 (팀 피처와 일치)


0

In [9]:
# 🟩 [수인 개선 · 미래 정보 혼입 제거] 빌더: 팀 피처 행렬을 기반으로 (1) fold 내부 통계로 교체 (2) 열 제거 (3) 추가 피처 결합
CROSS_CACHE = {}  # fold별 fold 내부 통계 캐시(같은 fold에서 반복 계산 방지)


def make_builder(fold_safe: bool = False, drop=(), extra=None, only=None):  # fold별 학습·검증 행렬 생성 함수를 만들어 반환
    # extra: [(names, array[N,k])]  /  only: 지정하면 이 이름들만 사용
    drop_set = set(drop)  # 제거할 피처 이름 집합
    keep = [i for i, n in enumerate(TEAM_FEATURES) if n not in drop_set and (only is None or n in only)]  # 사용할 열 위치
    base_names = [TEAM_FEATURES[i] for i in keep]  # 사용할 팀 피처 이름
    ex_names = [n for ns, _ in (extra or []) for n in ns]  # 추가 피처 이름
    names = base_names + ex_names  # 최종 피처 이름(팀 피처 + 추가 피처)
    pos = {n: j for j, n in enumerate(names)}  # 피처 이름 → 최종 행렬의 열 위치

    def build(sp):  # fold 하나의 학습·검증 데이터를 생성
        tr, va = sp["tr"], sp["va"]  # 학습·검증 행 인덱스
        Xtr = X_TEAM[np.ix_(tr, keep)]  # 학습 행렬(선택된 열만)
        Xva = X_TEAM[np.ix_(va, keep)]  # 검증 행렬(선택된 열만)
        if extra:  # 추가 피처가 있으면
            Xtr = np.hstack([Xtr] + [a[tr] for _, a in extra])  # 학습 행렬에 추가 피처 결합
            Xva = np.hstack([Xva] + [a[va] for _, a in extra])  # 검증 행렬에 추가 피처 결합
        if fold_safe:  # 학습 구간 기준 통계로 교체
            key = str(sp["key"])  # fold 식별자
            if key not in CROSS_CACHE:  # 캐시에 없으면 새로 계산
                if len(CROSS_CACHE) >= 6:  # 캐시가 가득 차면
                    CROSS_CACHE.pop(next(iter(CROSS_CACHE)))  # 가장 오래된 항목 제거(메모리 제한)
                CROSS_CACHE[key] = cross_row_stats(tr)  # 학습 행 기준 통계 계산 후 캐시에 저장
            cr = CROSS_CACHE[key]  # 이 fold의 fold 내부 통계
            for n, v in cr.items():  # 교체 대상 피처별 반복
                if n in pos:  # 현재 행렬에 포함된 피처만
                    Xtr[:, pos[n]] = v[tr]  # 학습 행의 값 교체
                    Xva[:, pos[n]] = v[va]  # 검증 행의 값 교체(학습 구간 통계 조회)
        return Xtr, y_all[tr], Xva, y_all[va], names, DT_M[tr]  # 학습·검증 행렬, 정답, 피처 이름, 학습 행의 월

    build.names = names  # 피처 이름을 빌더의 속성으로 노출
    return build  # 빌더 반환


BUILD_TEAM = make_builder(fold_safe=False)  # 🟦 팀 방식 (전체 데이터 통계)
BUILD_FS = make_builder(fold_safe=True)  # 🟩 학습 구간 통계 (미래 정보 혼입 제거)
print("피처 수:", len(BUILD_TEAM.names))  # 피처 수 출력

피처 수: 321


<div style="border-left:6px solid #1f77b4;background:#eef5fb;padding:8px 14px;margin:4px 0"><b>🟦 [팀 기존]</b> 4. 팀 기준 모델 재현 — 팀 설정(<code>max_depth=12, eta=0.02, subsample=0.8, colsample_bytree=0.4</code>, 최대 2000트리, 검증 fold AUC 기준 early stopping)을 그대로 사용</div>

- **재현 검증**: 공식 설정으로 **fold 1(월 15)** 한 개만 다시 학습하여 팀 보고서의 값(ROC 0.9537 / PR 0.7390, `PROGRESS.md` 3-2절)과 비교합니다. 값이 일치하면 이 노트북의 데이터·피처·학습 경로가 팀과 동일하다고 판단할 수 있습니다.
- 팀 최종 OOF(ROC 0.9493 / PR 0.7079, `final_oof_predictions.parquet`)는 팀 산출물 파일에서 불러옵니다. 6개 fold 전체를 다시 학습하여 확인하려면 아래 `FULL_BASELINE_RERUN = True`로 변경하십시오(수 시간 소요).

In [10]:
# 🟦 [팀 기존 · 출처: src/baseline_groupkfold_tc.py main()] 기준 모델 재현 (학습 로직 동일)
FULL_BASELINE_RERUN = False  # True이면 6개 fold 전체를 다시 학습(수 시간 소요)
if RUN["baseline"]:  # 기준 모델 재현 실행 여부
    spec_ref = [SPEC_GROUP[1]]  # 재현 검증 대상 fold(fold 1)
    cv_run("team_official_fold1", spec_ref, XGB_OFFICIAL, ROUNDS_OFFICIAL, ES_OFFICIAL, BUILD_TEAM,
           es_mode="valid")  # fold 1만 팀 공식 설정으로 학습
    if FULL_BASELINE_RERUN:  # 전체 재학습 옵션이 켜져 있으면
        cv_run("team_official", SPECS_ALL, XGB_OFFICIAL, ROUNDS_OFFICIAL, ES_OFFICIAL, BUILD_TEAM,
               es_mode="valid")  # 6개 fold 전체 학습
    r = RESULTS["team_official_fold1"]["folds"]["1"]  # fold 1 재현 결과
    print(f"\n재현 검증 — fold 1(월 {fold_to_month[1]}): 이 노트북 ROC {r['roc']:.4f} / PR {r['pr']:.4f}   "
          f"(팀 보고서: 0.9537 / 0.7390)")  # 팀 보고서 값과 나란히 출력
    if not SMOKE:  # 축소 실행(스모크 테스트)이 아닐 때만 판정
        print("판정:", "재현 성공(차이 0.005 이내)" if abs(r["pr"] - 0.7390) < 0.005
              else "차이가 큽니다 — 데이터/환경 확인 필요")  # PR-AUC 차이 기준 재현 여부 출력

# 팀 최종 OOF 불러오기 (팀 산출물)
if TEAM_OOF.exists() and not SMOKE:  # 팀 OOF 파일이 있고 정식 실행인 경우
    TEAM_OOF_DF = pd.read_parquet(TEAM_OOF).set_index("TransactionID").reindex(df_raw.index)  # 거래 ID 순서에 맞춰 정렬
    OOF_TEAM = TEAM_OOF_DF["y_prob"].to_numpy()  # 팀 OOF 예측 확률
    assert (TEAM_OOF_DF["y_true"].to_numpy() == y_all).all(), "팀 OOF의 정답이 이 노트북 데이터와 다릅니다."  # 정답 일치 확인
else:  # 파일이 없거나 축소 실행인 경우
    OOF_TEAM = None  # 팀 OOF 없음
if OOF_TEAM is not None:  # 팀 OOF가 있으면 성능 출력
    print(f"팀 최종 OOF: ROC {roc_auc_score(y_all, OOF_TEAM):.4f} / PR {average_precision_score(y_all, OOF_TEAM):.4f}"
          f"  (팀 보고서: 0.9493 / 0.7079)")  # 전체 OOF 성능과 팀 보고서 값
    print(pd.DataFrame({m: [roc_auc_score(y_all[DT_M == m], OOF_TEAM[DT_M == m]),
                            average_precision_score(y_all[DT_M == m], OOF_TEAM[DT_M == m])]
                        for m in MONTHS}, index=["ROC-AUC", "PR-AUC"]).round(4).T.rename_axis("월"))  # 월별 OOF 성능표 출력

[team_official_fold1] fold 1: ROC 0.9542 PR 0.7456 | 트리 828 | 피처 321 | 643s

재현 검증 — fold 1(월 15): 이 노트북 ROC 0.9542 / PR 0.7456   (팀 보고서: 0.9537 / 0.7390)
판정: 차이가 큽니다 — 데이터/환경 확인 필요
팀 최종 OOF: ROC 0.9492 / PR 0.7082  (팀 보고서: 0.9493 / 0.7079)
    ROC-AUC  PR-AUC
월                  
12   0.9190  0.5674
13   0.9553  0.7531
14   0.9613  0.7742
15   0.9542  0.7456
16   0.9671  0.7603
17   0.9477  0.6715


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 미래 정보 혼입 제거 · 내부 검증 기반 early stopping]</b> 스크리닝 설정의 비교 기준 모델 3종</div>

변형을 여러 개 비교하려면 모델을 수십 회 학습해야 하므로, **가벼운 스크리닝 설정**(max_depth 8, eta 0.08, 최대 500트리)으로 동일한 fold(월 15·17·16)에서 비교합니다. 기준 모델 3종은 다음과 같습니다.

| 이름 | 피처 생성 | early stopping | 의미 |
|---|---|---|---|
| `screen_team` | 전체 데이터 통계(팀 방식) | 검증 fold(팀 방식) | 팀 방식을 스크리닝 설정으로 실행한 결과 |
| `screen_foldsafe` | **학습 구간 통계** | 검증 fold(팀 방식) | `screen_team` 대비 차이 = 미래 정보 혼입의 영향 |
| `screen_clean` | 학습 구간 통계 | **내부 검증** | `screen_foldsafe` 대비 차이 = early stopping 편향의 영향. **누수 제거 기준 모델이며, 이후 모든 개선 후보의 비교 기준** |

In [11]:
# 🟩 [수인 개선 · 미래 정보 혼입 제거 / 내부 검증 기반 early stopping] 스크리닝 기준 모델 3종 실행
if RUN["fold_features"] or RUN["inner_early_stopping"]:  # 두 항목 중 하나라도 실행 대상이면
    cv_run("screen_team", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_TEAM, es_mode="valid")  # 팀 방식
    cv_run("screen_foldsafe", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_FS, es_mode="valid")  # 학습 구간 통계
    cv_run("screen_clean", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_FS, es_mode="innerfit")  # 누수 제거 기준
    print()  # 출력 구분용 빈 줄
    c_fb1 = compare("screen_foldsafe", "screen_team")  # 미래 정보 혼입 제거의 효과
    c_sf1 = compare("screen_clean", "screen_foldsafe")  # early stopping 편향의 효과
    c_all = compare("screen_clean", "screen_team")  # 두 효과를 합친 누적 변화

[screen_team] fold 1: ROC 0.9480 PR 0.7270 | 트리 452 | 피처 321 | 77s
[screen_team] fold 3: ROC 0.9363 PR 0.6505 | 트리 290 | 피처 321 | 55s
[screen_team] fold 5: ROC 0.9627 PR 0.7404 | 트리 476 | 피처 321 | 117s
[screen_foldsafe] fold 1: ROC 0.9336 PR 0.6731 | 트리 411 | 피처 321 | 124s
[screen_foldsafe] fold 3: ROC 0.9276 PR 0.6111 | 트리 496 | 피처 321 | 205s
[screen_foldsafe] fold 5: ROC 0.9463 PR 0.6713 | 트리 486 | 피처 321 | 252s
[screen_clean] fold 1: ROC 0.9336 PR 0.6588 | 트리 337 | 피처 321 | 257s
[screen_clean] fold 3: ROC 0.9097 PR 0.5642 | 트리 499 | 피처 321 | 322s
[screen_clean] fold 5: ROC 0.9279 PR 0.6127 | 트리 260 | 피처 321 | 140s

screen_foldsafe vs screen_team (3 fold) | PR 0.6493 vs 0.7065 (Δ-0.0572) | ROC 0.9341 vs 0.9488 (Δ-0.0146) | PR 개선 fold 0%
screen_clean vs screen_foldsafe (3 fold) | PR 0.6153 vs 0.6493 (Δ-0.0340) | ROC 0.9231 vs 0.9341 (Δ-0.0111) | PR 개선 fold 0%
screen_clean vs screen_team (3 fold) | PR 0.6153 vs 0.7065 (Δ-0.0912) | ROC 0.9231 vs 0.9488 (Δ-0.0257) | PR 개선 fold 0%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 미래 정보 혼입 제거 (순방향 검증)]</b> 과거 데이터만으로 미래 월을 예측하는 검증</div>

팀의 `GroupKFold(월)`은 검증 월보다 **미래**의 월도 학습에 사용하는 구조입니다(예: 월 15를 검증할 때 월 16·17이 학습에 포함됩니다). 실제 운영에서는 과거 데이터로 미래를 예측하므로, **월 *m*을 검증할 때 월 *m* 미만의 데이터만 학습에 사용하는** 순방향 검증을 함께 수행합니다.
팀 방식 피처(전체 데이터 통계, 미래 정보 포함)와 학습 구간 기준 피처(미래 정보 제외)를 동일한 순방향 fold에서 비교하여, **미래 정보를 제거한 뒤 성능이 얼마나 달라지는지**를 직접 측정합니다. 이는 사전 검토 과정에서 제기된 질문(미래 정보 제거 후 성능 변화)에 대한 답변입니다. (스크리닝 설정, 검증 월 `FORWARD_MONTHS`)

In [12]:
# 🟩 [수인 개선 · 미래 정보 혼입 제거 (순방향 검증)] 같은 순방향 fold에서 팀 방식과 학습 구간 기준 피처를 비교
if RUN["fold_features"]:  # 미래 정보 혼입 제거 항목을 실행하는 경우
    cv_run("fwd_team", SPECS_FWD, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_TEAM, es_mode="innerfit")  # 팀 방식 피처
    cv_run("fwd_foldsafe", SPECS_FWD, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_FS, es_mode="innerfit")  # 학습 구간 기준 피처
    c_fw = compare("fwd_foldsafe", "fwd_team")  # 순방향 fold에서의 성능 차이
    fwd_keys = list(RESULTS["fwd_team"]["folds"])  # 순방향 fold 식별자 목록
    tbl = pd.DataFrame({  # 월별 비교표
        "월": [int(k[2:]) for k in fwd_keys],  # 검증 월(식별자 'fw{월}'에서 추출)
        "팀 방식 PR": [RESULTS["fwd_team"]["folds"][k]["pr"] for k in fwd_keys],  # 팀 방식 PR-AUC
        "학습 구간 기준 PR": [RESULTS["fwd_foldsafe"]["folds"][k]["pr"] for k in fwd_keys],  # 학습 구간 기준 PR-AUC
    })
    tbl["차이"] = tbl["학습 구간 기준 PR"] - tbl["팀 방식 PR"]  # 월별 PR-AUC 변화량
    display(tbl.round(4))  # 비교표 출력
    add_verdict(  # 판정 기록: 평가 타당성 수정이므로 성능과 무관하게 채택
        "fold_features", "미래 정보 혼입 제거(학습 구간 기준 피처 생성)", "평가 타당성 수정", "채택(평가 타당성 수정)",
        f"GroupKFold 스크리닝: PR {c_fb1['ref_pr']:.4f}→{c_fb1['pr']:.4f} (Δ{c_fb1['d_pr']:+.4f}), "
        f"ROC Δ{c_fb1['d_roc']:+.4f} | "
        f"순방향: PR {c_fw['ref_pr']:.4f}→{c_fw['pr']:.4f} (Δ{c_fw['d_pr']:+.4f})",  # 근거 문자열
        d_pr=c_fb1["d_pr"], d_pr_forward=c_fw["d_pr"],  # 판정표에 함께 저장할 수치
    )

[fwd_team] fold fw14: ROC 0.8868 PR 0.5388 | 트리 191 | 피처 321 | 27s
[fwd_team] fold fw15: ROC 0.9094 PR 0.5924 | 트리 320 | 피처 321 | 166s
[fwd_team] fold fw16: ROC 0.9279 PR 0.5662 | 트리 424 | 피처 321 | 252s
[fwd_team] fold fw17: ROC 0.9189 PR 0.5853 | 트리 274 | 피처 321 | 236s
[fwd_foldsafe] fold fw14: ROC 0.8743 PR 0.5183 | 트리 184 | 피처 321 | 87s
[fwd_foldsafe] fold fw15: ROC 0.8683 PR 0.5179 | 트리 238 | 피처 321 | 50s
[fwd_foldsafe] fold fw16: ROC 0.9001 PR 0.4755 | 트리 333 | 피처 321 | 232s
[fwd_foldsafe] fold fw17: ROC 0.9097 PR 0.5642 | 트리 499 | 피처 321 | 309s
fwd_foldsafe vs fwd_team (4 fold) | PR 0.5081 vs 0.5684 (Δ-0.0603) | ROC 0.8843 vs 0.9086 (Δ-0.0243) | PR 개선 fold 0%


,월,팀 방식 PR,학습 구간 기준 PR,차이
0,14,0.5388,0.5183,-0.0205
1,15,0.5924,0.5179,-0.0745
2,16,0.5662,0.4755,-0.0907
3,17,0.5853,0.5642,-0.0210


▶ 미래 정보 혼입 제거(학습 구간 기준 피처 생성): **채택(평가 타당성 수정)** — GroupKFold 스크리닝: PR 0.7065→0.6493 (Δ-0.0572), ROC Δ-0.0146 | 순방향: PR 0.5684→0.5081 (Δ-0.0603)


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 내부 검증 기반 early stopping]</b> early stopping 편향 확인</div>

팀 코드는 `eval_set=[(검증 fold)]`, `early_stopping_rounds=100`으로 **검증 fold의 정답을 보고 트리 수를 선택**합니다. 검증 fold가 모델 선택에 사용되므로 보고된 검증 성능이 낙관적일 수 있습니다.
개선: 학습 구간의 **마지막 달**을 내부 검증으로 사용하여 트리 수를 정하고, 검증 fold는 평가에만 사용합니다. 위 `screen_clean`(내부 검증)과 `screen_foldsafe`(검증 fold 사용)의 차이가 편향의 크기입니다.

In [13]:
# 🟩 [수인 개선 · 내부 검증 기반 early stopping] 편향 판정 (평가 타당성 수정: 성능과 무관하게 채택, 하락폭 보고)
if RUN["inner_early_stopping"]:  # 내부 검증 early stopping 항목을 실행하는 경우
    add_verdict(  # 판정 기록
        "inner_early_stopping", "내부 검증 기반 early stopping", "평가 타당성 수정", "채택(평가 타당성 수정)",
        f"스크리닝 3 fold: PR {c_sf1['ref_pr']:.4f}→{c_sf1['pr']:.4f} (Δ{c_sf1['d_pr']:+.4f}), ROC Δ{c_sf1['d_roc']:+.4f}. "
        f"(내부 검증 모델은 학습 구간 마지막 1개월을 학습에 덜 사용하므로 하락분에 이 효과도 일부 포함됩니다.)",  # 근거 문자열
        d_pr=c_sf1["d_pr"],  # 판정표에 함께 저장할 수치
    )
    print("팀 방식(screen_team) → 누수 제거 기준 모델(screen_clean) 누적 변화: PR Δ%+.4f, ROC Δ%+.4f"
          % (c_all["d_pr"], c_all["d_roc"]))  # 누적 변화 출력

▶ 내부 검증 기반 early stopping: **채택(평가 타당성 수정)** — 스크리닝 3 fold: PR 0.6493→0.6153 (Δ-0.0340), ROC Δ-0.0111. (내부 검증 모델은 학습 구간 마지막 1개월을 학습에 덜 사용하므로 하락분에 이 효과도 일부 포함됩니다.)
팀 방식(screen_team) → 누수 제거 기준 모델(screen_clean) 누적 변화: PR Δ-0.0912, ROC Δ-0.0257


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 과거 이력 기반 속도 피처]</b> 과거 이력 기반 속도(velocity) 피처</div>

**확인한 미흡한 점**: 팀의 UID 집계는 그룹 평균·표준편차와 같은 **정적 통계**입니다. 사기 거래의 전형적 패턴인 *짧은 시간에 집중된 연속 거래*와 *직전 거래 대비 급격한 금액 변화*는 거래의 **순서와 시간 간격** 정보가 있어야 포착할 수 있습니다.

**개선**: 동일 UID(및 `card1+addr1`)의 **해당 거래보다 과거의 거래만** 사용하여 아래 피처를 생성합니다. 미래 거래와 정답 라벨을 사용하지 않으므로 미래 정보 혼입 문제를 다시 만들지 않으며, 운영 환경에서도 계산할 수 있는 정보입니다.

`gap_prev`(직전 거래와의 간격, 초) · `n_prev_1h` / `n_prev_24h`(직전 1시간/24시간 내 거래 수) · `n_prev_all`(과거 거래 누적 수) · `amt_ratio_prev`(직전 거래 대비 금액 비) · `amt_z_past`(과거 금액 분포 대비 z-score)

In [14]:
# 🟩 [수인 개선 · 과거 이력 기반 속도 피처] 속도 피처 계산 (과거 거래만 사용, 정렬·누적합·searchsorted 기반 벡터화)
def velocity_feats(codes, t, amt):  # 그룹 코드·시각·금액으로 과거 이력 기반 피처 6종을 계산
    n = len(t)  # 전체 거래 수
    order = np.lexsort((t, codes))  # 그룹 코드, 시각 순으로 정렬한 위치
    c = codes[order]  # 정렬된 그룹 코드
    tt = t[order].astype("float64")  # 정렬된 시각
    aa = amt[order].astype("float64")  # 정렬된 금액
    start = np.r_[True, c[1:] != c[:-1]]  # 각 그룹의 첫 거래 표시
    seg = np.cumsum(start) - 1  # 거래별 그룹 번호
    seg_starts = np.flatnonzero(start)  # 그룹별 시작 위치
    pos = np.arange(n) - seg_starts[seg]  # 그룹 내 과거 거래 수(자기 자신 제외)
    gap = np.full(n, np.nan)  # 직전 거래와의 시간 간격 배열
    gap[1:] = tt[1:] - tt[:-1]  # 인접 거래 간 시간 차
    gap[start] = np.nan  # 그룹의 첫 거래는 직전 거래가 없으므로 결측
    prev_a = np.full(n, np.nan)  # 직전 거래 금액 배열
    prev_a[1:] = aa[:-1]  # 한 칸 밀어서 직전 거래 금액 대입
    prev_a[start] = np.nan  # 그룹의 첫 거래는 결측
    with np.errstate(invalid="ignore", divide="ignore"):  # 0으로 나누기 경고 억제(결과는 NaN 처리)
        ratio = aa / np.where(prev_a > 0, prev_a, np.nan)  # 직전 거래 대비 금액 비(직전 금액이 0 이하이면 결측)
        cs = np.r_[0.0, np.cumsum(aa)[:-1]]  # 자기 자신을 제외한 금액 누적합
        cs2 = np.r_[0.0, np.cumsum(aa * aa)[:-1]]  # 자기 자신을 제외한 금액 제곱 누적합
        ps = cs - cs[seg_starts][seg]  # 그룹 내 과거 금액 합
        ps2 = cs2 - cs2[seg_starts][seg]  # 그룹 내 과거 금액 제곱 합
        mean = np.where(pos >= 1, ps / np.maximum(pos, 1), np.nan)  # 과거 금액 평균(과거 거래가 없으면 결측)
        var = np.where(pos >= 2, (ps2 - pos * mean ** 2) / np.maximum(pos - 1, 1), np.nan)  # 과거 금액 표본 분산
        z = (aa - mean) / np.sqrt(np.maximum(var, 0) + 1.0)  # 과거 분포 대비 z-score(분모에 1을 더해 안정화)
    tg = tt + seg * 1e10  # 그룹 간 시간 범위가 겹치지 않도록 그룹별 오프셋 추가
    n1h = np.arange(n) - np.searchsorted(tg, tg - 3600, side="left")  # 직전 1시간 내 과거 거래 수
    n24 = np.arange(n) - np.searchsorted(tg, tg - 86400, side="left")  # 직전 24시간 내 과거 거래 수
    out = np.empty((n, 6), dtype="float32")  # 결과 행렬(원래 행 순서)
    for j, v in enumerate([gap, n1h, n24, pos, ratio, z]):  # 피처 6종을 열로 배치
        out[order, j] = v  # 정렬 이전의 원래 행 위치로 되돌려 저장
    return out  # 속도 피처 행렬


_cache = CK / "velocity.npy"  # 속도 피처 캐시 파일 경로
VEL_NAMES = [f"{k}_{s}" for k in ("uid", "c1a1")  # 그룹 기준(UID, card1+addr1)별로
             for s in ("gap_prev", "n_prev_1h", "n_prev_24h", "n_prev_all", "amt_ratio_prev", "amt_z_past")]  # 피처 6종 이름
if _cache.exists():  # 캐시가 있으면
    VEL = np.load(_cache)  # 캐시에서 불러오기
else:  # 캐시가 없으면 새로 계산
    _c1a1_str = (pd.Series(RAW["card1"]).astype(str) + "_"
                 + pd.Series(RAW["addr1"]).fillna(-999).astype(str))  # card1+addr1 문자열 키
    _c1a1, _ = pd.factorize(_c1a1_str)  # 문자열 키를 정수 코드로 변환
    VEL = np.hstack([  # UID 기준과 card1+addr1 기준 피처를 열 방향으로 결합
        velocity_feats(UID_CODE, RAW["TransactionDT"], RAW["TransactionAmt"]),  # UID 기준
        velocity_feats(_c1a1, RAW["TransactionDT"], RAW["TransactionAmt"]),  # card1+addr1 기준
    ])
    np.save(_cache, VEL)  # 다음 실행을 위해 캐시에 저장
print("속도 피처:", VEL.shape, "| 결측 비율(첫 거래 등):", np.isnan(VEL).mean(axis=0).round(2)[:6])  # 형태와 결측 비율 출력

# 인과성 점검: 어떤 거래의 속도 피처가 그 거래 이후의 거래에 영향받지 않는지 — 무작위 5개 UID에서 직접 재계산하여 비교
_rng = np.random.default_rng(0)  # 재현 가능한 난수 생성기
_bad = 0  # 불일치 횟수
for u in _rng.choice(np.unique(UID_CODE), 5, replace=False):  # 무작위 UID 5개
    ix = np.where(UID_CODE == u)[0]  # 해당 UID의 거래 위치
    ix = ix[np.argsort(RAW["TransactionDT"][ix], kind="stable")]  # 시각 순으로 정렬
    for r in range(1, len(ix)):  # 두 번째 거래부터 점검
        prev_t = RAW["TransactionDT"][ix[:r]]  # 과거 거래 시각
        me = RAW["TransactionDT"][ix[r]]  # 현재 거래 시각
        exp_1h = int((prev_t >= me - 3600).sum())  # 직접 센 직전 1시간 내 과거 거래 수
        if exp_1h != int(VEL[ix[r], 1]) and prev_t.max() != me:  # 계산값과 다르고 동시각 거래가 아닌 경우
            _bad += 1  # 불일치 횟수 증가
print("인과성 점검 불일치 수(0이어야 함):", _bad)  # 점검 결과 출력

속도 피처: (590540, 12) | 결측 비율(첫 거래 등): [0.37 0.   0.   0.   0.37 0.53]
인과성 점검 불일치 수(0이어야 함): 0


In [15]:
# 🟩 [수인 개선 · 과거 이력 기반 속도 피처] 누수 제거 기준 모델(screen_clean) 대비 비교 — 성능 개선 후보 기준 적용
BUILD_VEL = make_builder(fold_safe=True, extra=[(VEL_NAMES, VEL)])  # 학습 구간 기준 피처에 속도 피처를 추가한 빌더
if RUN["velocity"]:  # 속도 피처 항목을 실행하는 경우
    cv_run("screen_velocity", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_VEL, es_mode="innerfit")  # 학습
    c = compare("screen_velocity", "screen_clean")  # 누수 제거 기준 모델과 비교
    add_verdict(  # 판정 기록
        "velocity", "과거 이력 기반 속도 피처", "성능 개선 후보", judge_B(c),
        f"PR {c['ref_pr']:.4f}→{c['pr']:.4f} (Δ{c['d_pr']:+.4f}), ROC Δ{c['d_roc']:+.4f}, "
        f"PR 개선 fold {c['win_frac']:.0%} "
        f"(기준: Δ≥+0.003 & fold 2/3)",  # 근거 문자열
        d_pr=c["d_pr"], win_frac=c["win_frac"],  # 판정표에 함께 저장할 수치
    )
    last_fold = str(SCREEN_FOLDS[-1])  # 마지막 스크리닝 fold 식별자
    imp = pd.Series({k: v for k, v in RESULTS["screen_velocity"]["folds"][last_fold]["imp"].items()
                     if k in VEL_NAMES}).sort_values(ascending=False)  # 속도 피처의 gain 중요도(내림차순)
    print("속도 피처 gain 중요도(마지막 fold):")  # 제목 출력
    display(imp.round(1).to_frame("gain"))  # 중요도 표 출력

[screen_velocity] fold 1: ROC 0.9298 PR 0.6515 | 트리 228 | 피처 333 | 174s
[screen_velocity] fold 3: ROC 0.8986 PR 0.5431 | 트리 323 | 피처 333 | 241s
[screen_velocity] fold 5: ROC 0.9215 PR 0.5693 | 트리 170 | 피처 333 | 187s
screen_velocity vs screen_clean (3 fold) | PR 0.5903 vs 0.6153 (Δ-0.0250) | ROC 0.9155 vs 0.9231 (Δ-0.0075) | PR 개선 fold 0%
▶ 과거 이력 기반 속도 피처: **기각** — PR 0.6153→0.5903 (Δ-0.0250), ROC Δ-0.0075, PR 개선 fold 0% (기준: Δ≥+0.003 & fold 2/3)
속도 피처 gain 중요도(마지막 fold):


,gain
c1a1_n_prev_all,8.6
uid_gap_prev,7.8
c1a1_n_prev_1h,6.5
c1a1_gap_prev,6.4
c1a1_amt_z_past,6.4
uid_n_prev_1h,6.2
c1a1_n_prev_24h,6.0
uid_amt_ratio_prev,5.8
uid_n_prev_24h,5.2
c1a1_amt_ratio_prev,4.8


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 상대 시각·요일 피처 추가 / 시간 의존 피처 제거 영향 분석]</b> 시간 변수 채택 근거 — 심사 과정에서 제기된 질문 "시간에 대한 변수를 사용했는데 어떤 고려가 있어서 변수로 채택되었는가"에 대한 검증</div>

팀 최종 피처에는 `TransactionDT` 원본이 제외되어 있으나, **D 컬럼 정규화(`Dn = 거래일 − Dn`)와 UID 생성은 시간 정보를 이용**합니다. 또한 EDA(가설 7)에서 상대 시간대별 사기율 차이(최고 11.21% / 최저 2.31%)가 확인되었습니다.
이 노트북은 시간 정보의 역할을 두 방향으로 검증합니다.

- **(a) 추가**: 상대 시각(`hour_rel`)·상대 요일(`dow_rel`) 피처를 추가하면 성능이 향상되는가? → 성능 개선 후보 기준(PR-AUC +0.003 이상 개선, 개선 fold 비율 2/3 이상)으로 채택 여부 판정
- **(b) 제거**: 시간에 의존하는 D 정규화 피처(D4·D10~D15)를 모두 제거하면 성능이 얼마나 하락하는가? → 시간 정보를 **유지한 근거**로 활용(분석·보고 항목이며 채택/기각 판정은 하지 않음)

(`TransactionDT`는 실제 날짜가 아니라 기준 시점 이후의 경과 초이므로 "상대" 시각·요일로 표기합니다.)

In [16]:
# 🟩 [수인 개선 · 상대 시각·요일 피처 추가 / 시간 의존 피처 제거 영향 분석] 시간 피처 추가(a) / 시간 의존 피처 제거(b)
TIME_NEW = np.c_[(RAW["TransactionDT"] // 3600) % 24,  # 상대 시각(0~23)
                 (RAW["TransactionDT"] // 86400) % 7].astype("float32")  # 상대 요일(0~6)
TIME_NAMES = ["hour_rel", "dow_rel"]  # 추가 피처 이름
TIME_DEP = [n for n in TEAM_FEATURES if n in ("D4", "D10", "D11", "D12", "D13", "D14", "D15")]  # 시간에 의존하는 D 정규화 피처
if RUN["time_features"]:  # 시간 피처 항목을 실행하는 경우
    BUILD_TIME_ADD = make_builder(fold_safe=True, extra=[(TIME_NAMES, TIME_NEW)])  # (a) 시간 피처를 추가한 빌더
    BUILD_TIME_DROP = make_builder(fold_safe=True, drop=TIME_DEP)  # (b) 시간 의존 피처를 제거한 빌더
    cv_run("screen_time_add", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_TIME_ADD,
           es_mode="innerfit")  # (a) 시간 피처 추가 모델 학습
    cv_run("screen_time_drop", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, BUILD_TIME_DROP,
           es_mode="innerfit")  # (b) 시간 의존 피처 제거 모델 학습
    ca = compare("screen_time_add", "screen_clean")  # (a) 누수 제거 기준 모델과 비교
    cb = compare("screen_time_drop", "screen_clean")  # (b) 누수 제거 기준 모델과 비교
    add_verdict(  # (a) 판정 기록
        "time_add", "상대 시각·요일 피처 추가", "성능 개선 후보", judge_B(ca),
        f"PR Δ{ca['d_pr']:+.4f}, ROC Δ{ca['d_roc']:+.4f}, PR 개선 fold {ca['win_frac']:.0%}",  # 근거 문자열
        d_pr=ca["d_pr"], win_frac=ca["win_frac"],  # 판정표에 함께 저장할 수치
    )
    add_verdict(  # (b) 판정 기록(분석·보고 항목)
        "time_drop", "시간 의존 피처 제거 영향 분석", "분석·보고", "수행 완료",
        f"제거 피처 {TIME_DEP}: PR Δ{cb['d_pr']:+.4f}, ROC Δ{cb['d_roc']:+.4f} → 시간 정보를 유지한 근거로 사용",  # 근거 문자열
        d_pr=cb["d_pr"],  # 판정표에 함께 저장할 수치
    )

[screen_time_add] fold 1: ROC 0.9339 PR 0.6546 | 트리 328 | 피처 323 | 261s
[screen_time_add] fold 3: ROC 0.9088 PR 0.5563 | 트리 438 | 피처 323 | 332s
[screen_time_add] fold 5: ROC 0.9206 PR 0.5592 | 트리 164 | 피처 323 | 99s
[screen_time_drop] fold 1: ROC 0.9240 PR 0.6448 | 트리 476 | 피처 314 | 283s
[screen_time_drop] fold 3: ROC 0.9050 PR 0.5408 | 트리 411 | 피처 314 | 292s
[screen_time_drop] fold 5: ROC 0.9048 PR 0.5425 | 트리 184 | 피처 314 | 171s
screen_time_add vs screen_clean (3 fold) | PR 0.5959 vs 0.6153 (Δ-0.0194) | ROC 0.9205 vs 0.9231 (Δ-0.0026) | PR 개선 fold 0%
screen_time_drop vs screen_clean (3 fold) | PR 0.5789 vs 0.6153 (Δ-0.0364) | ROC 0.9093 vs 0.9231 (Δ-0.0138) | PR 개선 fold 0%
▶ 상대 시각·요일 피처 추가: **기각** — PR Δ-0.0194, ROC Δ-0.0026, PR 개선 fold 0%
▶ 시간 의존 피처 제거 영향 분석: **수행 완료** — 제거 피처 ['D4', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15']: PR Δ-0.0364, ROC Δ-0.0138 → 시간 정보를 유지한 근거로 사용


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · LightGBM 단독, XGBoost+LightGBM 평균 앙상블]</b> 모델 다양성 — LightGBM 추가와 평균 앙상블</div>

**발견한 미흡한 점**: 팀 `MODELING_REPORT.md` 한계 항목에 "단일 모델(XGBoost)만 검증, 앙상블 미시도"가 있습니다.

**개선**: 같은 피처·같은 fold·같은 내부 검증 방식으로 LightGBM을 학습하고, XGBoost와의 **단순 평균**(가중치 튜닝 없음 → 과적합 여지 최소화)을 비교합니다. 비교 기준은 누수 제거 기준 모델(`screen_clean`)이며, 성능 개선 후보 기준(ΔPR-AUC ≥ +0.003, 비교 fold의 2/3 이상에서 개선)을 적용합니다.

In [17]:
# 🟩 [수인 개선 · LightGBM 단독, XGBoost+LightGBM 평균 앙상블]
try:  # lightgbm 설치 여부를 확인
    import lightgbm as lgb  # LightGBM 라이브러리를 불러옴
except ImportError as e:  # 설치되어 있지 않은 경우
    raise ImportError("lightgbm이 필요합니다. 이 셀 위에서 `%pip install lightgbm` 을 한 번 실행하세요.") from e  # 안내 후 중단

# 스크리닝용 LightGBM 파라미터 (XGBoost 스크리닝 설정과 학습률을 맞춤)
LGB_SCREEN = dict(objective="binary", metric="auc", learning_rate=0.08, num_leaves=63, feature_fraction=0.4,
                  bagging_fraction=0.8, bagging_freq=1, min_child_samples=50, verbose=-1, num_threads=-1, seed=0)
# 공식 설정용 LightGBM 파라미터 (학습률을 낮추고 잎 수를 늘림)
LGB_OFFICIAL = dict(LGB_SCREEN, learning_rate=0.03, num_leaves=127)
if SMOKE:  # 코드 점검용 실행에서는 모델을 작게 줄임
    LGB_SCREEN = dict(LGB_SCREEN, num_leaves=15, learning_rate=0.3, min_child_samples=20)  # 잎 수·학습률·최소 표본 축소
    LGB_OFFICIAL = dict(LGB_SCREEN)  # 공식 설정도 같은 경량 설정으로 대체


def make_lgb_fit(params, rounds, es):  # cv_run의 fit_fn 규격에 맞는 LightGBM 학습·예측 함수를 만든다.
    to_nan = lambda X: np.where(X == -1, np.nan, X)  # XGBoost의 missing=-1 규칙과 같게 결측 표기를 NaN으로 변환

    def fit(Xtr, ytr, Xva, yva, names, mtr):  # 한 번의 fold 학습·예측을 수행하는 함수
        inner = mtr == mtr.max()  # 학습 구간의 마지막 달을 내부 검증 구간으로 지정
        d_fit = lgb.Dataset(to_nan(Xtr[~inner]), ytr[~inner], feature_name=list(names), free_raw_data=True)  # 학습용 데이터셋
        d_es = lgb.Dataset(to_nan(Xtr[inner]), ytr[inner], reference=d_fit, free_raw_data=True)  # early stopping용 데이터셋
        # 내부 검증 AUC 기준 early stopping으로 학습
        b0 = lgb.train(params, d_fit, rounds, valid_sets=[d_es], callbacks=[lgb.early_stopping(es, verbose=False)])
        best = b0.best_iteration or rounds  # 최적 반복 수 (없으면 최대 반복 수)
        pred = b0.predict(to_nan(Xva), num_iteration=best)  # 검증 fold 예측 확률
        imp = dict(zip(names, b0.feature_importance("gain")))  # 변수별 gain 중요도
        return pred, best, imp  # cv_run이 기대하는 (예측, 트리 수, 중요도) 반환

    return fit  # 학습 함수 반환


def blend_store(tag, tags, weights=None):  # tags의 fold별 예측 확률을 (가중)평균해 새 결과를 만든다.
    folds = {}  # fold별 결과 저장 딕셔너리
    for k in RESULTS[tags[0]]["folds"]:  # 첫 번째 결과의 fold 키를 순회
        if not all(k in RESULTS[t]["folds"] for t in tags):  # 모든 입력 결과에 존재하는 fold만 사용
            continue  # 일부 결과에 없는 fold는 건너뜀
        w = weights or [1 / len(tags)] * len(tags)  # 가중치가 없으면 동일 가중 평균
        # fold별 예측 확률의 가중합 (float64로 계산 후 float32로 저장)
        p = sum(wi * RESULTS[t]["folds"][k]["pred"].astype("float64") for wi, t in zip(w, tags)).astype("float32")
        va = RESULTS[tags[0]]["folds"][k]["va"]  # 검증 fold 행 인덱스
        yv = y_all[va]  # 검증 fold 정답 라벨
        folds[k] = dict(va=va, pred=p, best=0, imp={}, sec=0.0, roc=roc_auc_score(yv, p),
                        pr=average_precision_score(yv, p), n_feat=0)  # 지표를 포함한 fold 결과
    RESULTS[tag] = dict(folds=folds)  # 앙상블 결과를 등록
    return RESULTS[tag]  # 등록한 결과 반환


if RUN["lgb_ensemble"]:  # 실행 스위치가 켜져 있을 때만 수행
    # LightGBM을 스크리닝 fold에서 학습 (누수 제거 기준과 같은 fold 내부 피처)
    fit_lgb = make_lgb_fit(LGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN)  # 스크리닝 설정의 LightGBM 학습 함수
    # LightGBM을 스크리닝 fold에서 학습 (누수 제거 기준과 같은 fold 내부 피처)
    cv_run("screen_lgb", SPECS_SCREEN, None, None, None, BUILD_FS, fit_fn=fit_lgb)
    blend_store("screen_ensemble", ["screen_clean", "screen_lgb"])  # XGBoost와 LightGBM 예측 확률의 단순 평균
    c_l = compare("screen_lgb", "screen_clean")  # LightGBM 단독 vs 누수 제거 기준 모델
    c_e = compare("screen_ensemble", "screen_clean")  # 앙상블 vs 누수 제거 기준 모델
    # LightGBM 단독 판정 기록
    add_verdict("lgb", "LightGBM 단독", "성능 개선 후보", judge_B(c_l),
                f"PR Δ{c_l['d_pr']:+.4f}, ROC Δ{c_l['d_roc']:+.4f}, PR 개선 fold {c_l['win_frac']:.0%}",
                d_pr=c_l["d_pr"], win_frac=c_l["win_frac"])
    # 앙상블 판정 기록
    add_verdict("ensemble", "XGBoost+LightGBM 평균 앙상블", "성능 개선 후보", judge_B(c_e),
                f"PR Δ{c_e['d_pr']:+.4f}, ROC Δ{c_e['d_roc']:+.4f}, PR 개선 fold {c_e['win_frac']:.0%}",
                d_pr=c_e["d_pr"], win_frac=c_e["win_frac"])

[screen_lgb] fold 1: ROC 0.9407 PR 0.6757 | 트리 399 | 피처 321 | 97s
[screen_lgb] fold 3: ROC 0.9018 PR 0.5258 | 트리 425 | 피처 321 | 104s
[screen_lgb] fold 5: ROC 0.9292 PR 0.6125 | 트리 172 | 피처 321 | 59s
screen_lgb vs screen_clean (3 fold) | PR 0.6049 vs 0.6153 (Δ-0.0104) | ROC 0.9246 vs 0.9231 (Δ+0.0015) | PR 개선 fold 33%
screen_ensemble vs screen_clean (3 fold) | PR 0.6331 vs 0.6153 (Δ+0.0178) | ROC 0.9315 vs 0.9231 (Δ+0.0084) | PR 개선 fold 100%
▶ LightGBM 단독: **기각** — PR Δ-0.0104, ROC Δ+0.0015, PR 개선 fold 33%
▶ XGBoost+LightGBM 평균 앙상블: **채택** — PR Δ+0.0178, ROC Δ+0.0084, PR 개선 fold 100%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 하이퍼파라미터 재검토]</b> 누수 제거 기준 모델 대비 공정 비교</div>

**발견한 미흡한 점**: 팀 보고서의 "튜닝 기각"은 경량 설정(500트리)과 검증 fold 기준 early stopping 조건에서 내려진 결정입니다. 기준 자체가 낙관적이었으므로 같은 질문을 **누수 제거 기준 모델(`screen_clean`) 대비**로 다시 확인합니다.

후보 5개(트리 깊이 2종, 열 샘플링, 규제·최소 자식 가중치, `scale_pos_weight`)를 같은 fold에서 비교합니다. 후보가 여러 개이므로 **가장 좋은 1개가 성능 개선 후보 기준을 통과할 때만 채택**하고, 선택 편향을 줄이기 위해 아래 최종 공식 6-fold 실행에서 다시 확인합니다.
(`scale_pos_weight`는 팀이 이미 시도해 기각한 항목이며, 같은 기준에서 재확인하는 것이므로 팀 결과를 부정하는 것은 아닙니다.)

In [18]:
# 🟩 [수인 개선 · 하이퍼파라미터 재검토] 후보 5개 스크리닝
TUNE_CANDS = {  # 후보 이름 → XGBoost 파라미터 변경값
    "depth12": dict(max_depth=12),  # 트리 깊이를 12로 증가
    "depth6": dict(max_depth=6),  # 트리 깊이를 6으로 감소
    "colsample07": dict(colsample_bytree=0.7),  # 트리별 열 샘플링 비율을 0.7로 설정
    "regularized": dict(min_child_weight=5, reg_lambda=5.0),  # 최소 자식 가중치와 L2 규제를 강화
    # 양성 가중치를 클래스 비율의 제곱근으로 설정
    "pos_weight": dict(scale_pos_weight=float(math.sqrt((1 - y_all.mean()) / y_all.mean()))),
}
TUNE_OVER = {}  # 채택된 파라미터 변경값 (기본: 없음)
if RUN["tuning"]:  # 실행 스위치가 켜져 있을 때만 수행
    rows = []  # 후보별 비교 결과 목록
    for name, over in TUNE_CANDS.items():  # 후보를 하나씩 평가
        p = dict(XGB_SCREEN, **over)  # 스크리닝 설정에 후보 변경값을 덮어씀
        if SMOKE and "max_depth" in over:  # 코드 점검 실행에서는 깊이를 제한
            p["max_depth"] = min(over["max_depth"], 5)  # 최대 깊이를 5로 제한
        cv_run("screen_tune_" + name, SPECS_SCREEN, p, ROUNDS_SCREEN, ES_SCREEN, BUILD_FS, es_mode="innerfit")  # 후보 학습
        c = compare("screen_tune_" + name, "screen_clean", quiet=True)  # 누수 제거 기준 모델과 비교
        c["name"] = name  # 후보 이름을 결과에 기록
        rows.append(c)  # 결과 목록에 추가
    tune_tbl = pd.DataFrame(rows)[["name", "pr", "ref_pr", "d_pr", "d_roc", "win_frac"]]  # 비교 표 구성
    display(tune_tbl.round(4))  # 비교 표 출력
    best = max(rows, key=lambda r: r["d_pr"])  # PR-AUC 개선폭이 가장 큰 후보 선택
    TUNE_OVER = TUNE_CANDS[best["name"]] if judge_B(best) == "채택" else {}  # 기준 통과 시에만 파라미터 채택
    # 최선 후보의 판정 기록
    add_verdict("tuning", "하이퍼파라미터 재검토(후보 5개 중 최선)", "성능 개선 후보", judge_B(best),
                f"최선 후보 {best['name']}: PR Δ{best['d_pr']:+.4f}, ROC Δ{best['d_roc']:+.4f}, "
                f"PR 개선 fold {best['win_frac']:.0%} (후보 5개 중 선택 → 최종 6-fold에서 재확인)",
                d_pr=best["d_pr"], win_frac=best["win_frac"], over=json.dumps(TUNE_CANDS[best["name"]], default=float))
    print("채택된 파라미터 변경:", TUNE_OVER or "없음")  # 채택 결과 출력

[screen_tune_depth12] fold 1: ROC 0.9334 PR 0.6746 | 트리 179 | 피처 321 | 209s
[screen_tune_depth12] fold 3: ROC 0.9110 PR 0.5469 | 트리 107 | 피처 321 | 144s
[screen_tune_depth12] fold 5: ROC 0.9253 PR 0.5906 | 트리 80 | 피처 321 | 131s
[screen_tune_depth6] fold 1: ROC 0.9230 PR 0.6259 | 트리 498 | 피처 321 | 324s
[screen_tune_depth6] fold 3: ROC 0.8923 PR 0.5045 | 트리 499 | 피처 321 | 300s
[screen_tune_depth6] fold 5: ROC 0.9022 PR 0.5103 | 트리 332 | 피처 321 | 230s
[screen_tune_colsample07] fold 1: ROC 0.9183 PR 0.6199 | 트리 262 | 피처 321 | 185s
[screen_tune_colsample07] fold 3: ROC 0.8968 PR 0.5359 | 트리 354 | 피처 321 | 247s
[screen_tune_colsample07] fold 5: ROC 0.9210 PR 0.5712 | 트리 231 | 피처 321 | 180s
[screen_tune_regularized] fold 1: ROC 0.9314 PR 0.6704 | 트리 477 | 피처 321 | 287s
[screen_tune_regularized] fold 3: ROC 0.9089 PR 0.5629 | 트리 500 | 피처 321 | 226s
[screen_tune_regularized] fold 5: ROC 0.9297 PR 0.6119 | 트리 332 | 피처 321 | 64s
[screen_tune_pos_weight] fold 1: ROC 0.9308 PR 0.6437 | 트리 184 | 피처 3

,name,pr,ref_pr,d_pr,d_roc,win_frac
0,depth12,0.6085,0.6153,-0.0068,-0.0035,0.3333
1,depth6,0.5504,0.6153,-0.0649,-0.0173,0.0000
2,colsample07,0.5786,0.6153,-0.0367,-0.0120,0.0000
3,regularized,0.6182,0.6153,0.0029,0.0000,0.3333
4,pos_weight,0.5891,0.6153,-0.0262,-0.0113,0.0000


▶ 하이퍼파라미터 재검토(후보 5개 중 최선): **기각** — 최선 후보 regularized: PR Δ+0.0029, ROC Δ+0.0000, PR 개선 fold 33% (후보 5개 중 선택 → 최종 6-fold에서 재확인)
채택된 파라미터 변경: 없음


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 변수 의미 사전 · 사유 코드 · V 변수 제외 모델]</b> 핵심 근거 변수의 의미 — 심사 개선의견·사전질문: "핵심 근거 변수의 의미를 알기 어려움 → 의미를 알 수 있는 변수 위주로 설명"</div>

팀 최종 모델의 중요 변수는 `C13`, `V…`, `D…`처럼 **익명화된 이름**이라 현업 담당자가 이유를 설명하기 어렵습니다. 세 가지를 수행합니다.

1. **변수 의미 사전**: 변수군별로 의미를 정리하고(IEEE-CIS 대회 공개 설명 기준), 모델이 의존하는 정도를 변수군 단위로 집계합니다. 대회 공개 설명에 없는 내용은 "익명화된 값"으로만 적었습니다. 해석 가능성 등급(높음/중간/낮음)은 **수인의 판단 기준**이며, 모델 성능과는 무관한 분류입니다.
2. **거래별 사유 코드**: XGBoost 내장 SHAP(`pred_contribs`)으로 의심 거래마다 위험을 높인 상위 변수를 선정하고, 의미 사전과 연결해 문장으로 보여줍니다.
3. **V 변수 제외 모델**: 의미가 공개되지 않은 V 변수를 모두 제외한 모델의 성능을 측정해, 설명 가능성을 얻는 대가를 수치로 보여줍니다.

In [19]:
# 🟩 [수인 개선 · 변수 의미 사전] 변수군 규칙 기반 의미 사전
import re  # 정규식 매칭용 모듈
GROUP_MEANING = [  # (정규식, 변수군, 의미, 해석 가능성 등급 ← 수인 판단). 위에서부터 처음 일치한 규칙을 적용
    (r"^TransactionAmt$", "금액", "거래 금액(USD)", "높음"),  # 대회 공개 설명: 결제 금액(USD)
    (r"^cents$", "금액", "[팀 생성] 거래 금액의 소수부", "높음"),  # 팀이 생성한 피처
    (r"^ProductCD", "상품", "거래 상품 코드(범주)", "높음"),  # 대회 공개 설명: 거래별 상품 코드
    (r"^card4", "카드", "카드 정보(범주형, 값은 카드 브랜드 유형)", "높음"),  # 데이터에 나타나는 값 기준
    (r"^card6", "카드", "카드 정보(범주형, 값은 직불/신용 유형)", "높음"),  # 데이터에 나타나는 값 기준
    (r"^card[1235]", "카드", "카드 관련 정보(익명화된 값)", "낮음"),  # 대회 공개 설명: 카드 종류·발급 은행·국가 등, 세부 의미 비공개
    (r"^addr1|^addr2", "지역", "주소 관련 코드(익명화된 값)", "중간"),  # 세부 의미는 확인 필요
    (r"^dist", "지역", "거리 관련 값(익명화된 값)", "중간"),  # 대회 공개 설명: 거리
    (r"^[PR]_emaildomain", "이메일", "구매자(P)/수신자(R) 이메일 도메인", "높음"),  # 대회 공개 설명
    (r"^C\d+", "카운팅(C)", "카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)", "중간"),  # 대회 공개 설명
    (r"^D\d+", "시간간격(D)", "시간 간격 변수(예: 이전 거래와의 일수 등)", "중간"),  # 대회 공개 설명
    (r"^M\d+", "일치여부(M)", "일치 여부 변수(예: 카드·주소의 이름 일치 등, 값은 T/F)", "높음"),  # 대회 공개 설명
    (r"^V\d+", "V(Vesta)", "Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등)", "낮음"),  # 대회 공개 설명
    (r"^id_\d+|^Device", "접속·기기(id)", "접속 환경·기기·브라우저 관련 정보(익명화된 값)", "중간"),  # 대회 공개 설명
    (r"_FE$|^uid_FE$", "팀생성-빈도", "[팀 생성] 해당 값(또는 UID)의 출현 빈도", "높음"),  # 팀이 생성한 빈도 인코딩
    (r"_uid_(mean|std)$", "팀생성-UID집계", "[팀 생성] UID(card1+addr1+추정 첫 거래일) 단위 평균/표준편차", "높음"),  # 팀 UID 집계
    (r"^uid_tx_count$", "팀생성-UID집계", "[팀 생성] UID의 거래 건수", "높음"),  # 팀 UID 집계
    (r"_isna$|_was_missing$|missing", "결측지표", "[팀 생성] 결측 여부 지표", "높음"),  # 팀이 생성한 결측 지표
    (r"^(uid|c1a1)_(gap_prev|n_prev|amt_)", "수인생성-속도", "[수인 생성] 과거 거래 기반 속도 피처", "높음"),  # 개선 단계에서 생성
    (r"^(hour_rel|dow_rel)$", "수인생성-시간", "[수인 생성] 상대 시각/요일", "높음"),  # 개선 단계에서 생성
]


def meaning(n):  # 변수 이름에 처음 일치하는 규칙의 (변수군, 의미, 등급)을 반환한다.
    for pat, g, m, lvl in GROUP_MEANING:  # 규칙을 순서대로 확인
        if re.search(pat, n):  # 정규식이 일치하면
            return g, m, lvl  # 해당 규칙의 값을 반환
    return "기타", "의미 사전에 없음(확인 필요)", "낮음"  # 일치하는 규칙이 없는 경우


# 전체 변수(팀 + 속도 + 시간)에 대한 의미 사전 표
DICT = pd.DataFrame([(n, *meaning(n)) for n in TEAM_FEATURES + VEL_NAMES + TIME_NAMES],
                    columns=["변수", "변수군", "의미", "해석가능성(수인판단)"])
unmapped = DICT.loc[DICT["변수군"] == "기타", "변수"].tolist()  # 의미 사전에 없는 변수 목록
print("의미 사전에 없는 변수:", len(unmapped), "개 →", unmapped[:15])  # 미분류 변수 확인
display(DICT.groupby(["변수군", "해석가능성(수인판단)"]).size().rename("변수 수").reset_index())  # 변수군별 변수 수 표
DICT.to_csv(OUT / "변수_의미_사전.csv", index=False, encoding="utf-8-sig")  # 의미 사전을 CSV로 저장

의미 사전에 없는 변수: 1 개 → ['has_identity']


,변수군,해석가능성(수인판단),변수 수
0,V(Vesta),낮음,178
1,금액,높음,2
2,기타,낮음,1
3,상품,높음,1
4,수인생성-속도,높음,12
5,수인생성-시간,높음,2
6,시간간격(D),중간,23
7,이메일,높음,3
8,일치여부(M),높음,9
9,접속·기기(id),중간,59


In [20]:
# 🟩 [수인 개선 · 변수 의미 사전] 변수군별 모델 의존도 (누수 제거 기준 모델 gain 평균) + 상위 변수의 의미
if RUN["explainability"]:  # 실행 스위치가 켜져 있을 때만 수행
    folds_clean = RESULTS["screen_clean"]["folds"]  # 누수 제거 기준 모델의 fold별 결과
    imps = [pd.Series(f["imp"]) for f in folds_clean.values()]  # fold별 gain 중요도 시리즈
    imp_mean = pd.concat(imps, axis=1).fillna(0).mean(axis=1)  # fold 평균 gain (없는 변수는 0)
    imp_mean = imp_mean / imp_mean.sum()  # 합이 1이 되도록 정규화
    gtab = DICT.set_index("변수").join(imp_mean.rename("gain비중"), how="left").fillna({"gain비중": 0})  # 사전에 gain 비중 결합
    grp = (gtab.groupby("변수군")["gain비중"].agg(["sum", "count"])  # 변수군별 gain 합과 변수 수
           .sort_values("sum", ascending=False).rename(columns={"sum": "gain 비중 합", "count": "변수 수"}))  # 정렬·열 이름
    GROUP_IMP = grp  # 다른 셀에서 참조할 수 있도록 전역에 저장
    display(grp.round(4))  # 변수군별 의존도 표 출력
    top = gtab.sort_values("gain비중", ascending=False).head(15)[["변수군", "의미", "해석가능성(수인판단)", "gain비중"]]  # 상위 15개 변수
    display(top.round(4))  # 상위 변수 표 출력
    low_share = gtab.loc[gtab["해석가능성(수인판단)"] == "낮음", "gain비중"].sum()  # 해석 가능성 낮음 변수의 gain 비중 합
    print(f"해석 가능성 '낮음'(V·카드 익명 코드) 변수가 모델 gain에서 차지하는 비중: {low_share:.1%}")  # 비중 출력

,gain 비중 합,변수 수
변수군,,
V(Vesta),0.6824,178
카운팅(C),0.1367,27
접속·기기(id),0.0609,59
시간간격(D),0.0393,23
카드,0.0287,9
일치여부(M),0.0161,9
지역,0.0147,5
이메일,0.0065,3
팀생성-UID집계,0.0057,3


,변수군,의미,해석가능성(수인판단),gain비중
변수,,,,
V200,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.1576
V243,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0503
V258,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0460
C4,카운팅(C),"카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)",중간,0.0361
V205,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0312
V294,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0266
V148,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0259
V283,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0150
V274,V(Vesta),Vesta가 생성한 익명화 파생변수(순위·카운트·개체 관계 등),낮음,0.0150


해석 가능성 '낮음'(V·카드 익명 코드) 변수가 모델 gain에서 차지하는 비중: 70.2%


In [21]:
# 🟩 [수인 개선 · 사유 코드] 거래별 사유 코드 — XGBoost 내장 SHAP(pred_contribs)
if RUN["explainability"]:  # 실행 스위치가 켜져 있을 때만 수행
    f_last = SCREEN_FOLDS[-1]  # 스크리닝 fold 중 마지막 fold 사용
    sp = SPEC_GROUP[f_last]  # 해당 fold의 학습·검증 행 정의
    Xtr, ytr, Xva, yva, names, mtr = BUILD_FS(sp)  # fold 내부 통계로 피처 행렬 생성
    # 사유 코드 산출용 모델 학습 (모델 객체를 함께 반환)
    pred_rc, best_rc, _, bst_rc = fit_predict_xgb(Xtr, ytr, Xva, yva, names, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN,
                                                  "innerfit", mtr, keep_model=True)
    order = np.argsort(-pred_rc)[:300]  # 예측 확률 상위 300건의 검증 fold 내 위치
    dm = xgb.DMatrix(Xva[order], feature_names=names, missing=-1)  # 상위 300건을 DMatrix로 변환
    contrib = bst_rc.predict(dm, pred_contribs=True, iteration_range=(0, best_rc))[:, :-1]  # 변수별 SHAP 기여도 (편향항 제외)
    LV = DICT.set_index("변수")["해석가능성(수인판단)"].to_dict()  # 변수 → 해석 가능성 등급
    MN = DICT.set_index("변수")["의미"].to_dict()  # 변수 → 의미
    GP = DICT.set_index("변수")["변수군"].to_dict()  # 변수 → 변수군

    def reason(i, k=3, only_interpretable=True):  # i번째 거래에서 위험을 높인 상위 k개 변수를 의미와 함께 문장으로 만든다.
        c = contrib[i]  # i번째 거래의 변수별 기여도
        idx = np.argsort(-c)  # 기여도 내림차순 변수 위치
        out = []  # 사유 문장 목록
        for j in idx:  # 기여도가 큰 변수부터 확인
            if c[j] <= 0:  # 위험을 높이지 않는 변수가 나오면
                break  # 이후 변수도 모두 해당하지 않으므로 종료
            if only_interpretable and LV.get(names[j]) == "낮음":  # 해석 가능성이 낮은 변수는
                continue  # 사유에서 제외
            out.append(f"{names[j]}={Xva[order[i], j]:.4g} ({GP.get(names[j])}: {MN.get(names[j])})")  # 변수명=값 (변수군: 의미)
            if len(out) == k:  # 필요한 개수를 채우면
                break  # 종료
        return " / ".join(out) if out else "해석 가능한 변수로 설명되지 않음(익명 변수 위주)"  # 사유가 없으면 안내 문구

    show = [0, 1, 2, 3, 4]  # 예시로 보여줄 상위 5건의 위치
    ex = pd.DataFrame({"거래ID": df_raw.index.to_numpy()[sp["va"]][order][show],  # 거래 ID
                       "예측확률": pred_rc[order][show].round(3),  # 예측 확률
                       "실제 사기": yva[order][show],  # 실제 라벨
                       "해석 가능한 상위 사유(최대 3)": [reason(i) for i in show]})  # 사유 문장
    display(ex)  # 예시 표 출력
    cover = np.mean([reason(i) != "해석 가능한 변수로 설명되지 않음(익명 변수 위주)" for i in range(len(order))])  # 사유 제시 비율
    print(f"상위 300건 중 해석 가능한 사유 1개 이상을 제시할 수 있는 비율: {cover:.1%}")  # 비율 출력
    del Xtr, Xva  # 큰 행렬을 해제해 메모리 확보

,거래ID,예측확률,실제 사기,해석 가능한 상위 사유(최대 3)
0,3409595,0.999,1,"C1=325 (카운팅(C): 카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)) / D8=-113 (시간간격(D)..."
1,3409591,0.999,1,"C1=325 (카운팅(C): 카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)) / D8=-113 (시간간격(D)..."
2,3409589,0.999,1,"C1=325 (카운팅(C): 카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)) / C4=87 (카운팅(C): 카..."
3,3415461,0.998,1,"C1=116 (카운팅(C): 카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)) / C12=83 (카운팅(C): ..."
4,3438306,0.998,1,"C13=0 (카운팅(C): 카운팅 변수(예: 결제 카드에 연결된 주소 수 등, 실제 의미는 비공개)) / TransactionAmt_ui..."


상위 300건 중 해석 가능한 사유 1개 이상을 제시할 수 있는 비율: 100.0%


In [22]:
# 🟩 [수인 개선 · V 변수 제외 모델] 설명 가능성을 얻는 대가 (분석, 판정 없음)
if RUN["explainability"]:  # 실행 스위치가 켜져 있을 때만 수행
    V_NAMES = [n for n in TEAM_FEATURES if re.match(r"^V\d+$", n)]  # V1~V339 변수 이름 목록
    # V 변수를 제외한 피처 구성으로 스크리닝 학습
    builder_no_v = make_builder(True, drop=V_NAMES)  # V 변수를 제외하는 피처 빌더
    # V 변수를 제외한 피처 구성으로 스크리닝 학습
    cv_run("screen_no_v", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, builder_no_v, es_mode="innerfit")
    c = compare("screen_no_v", "screen_clean")  # 누수 제거 기준 모델과 비교
    # 분석 결과 기록 (채택/기각 판정 없음)
    add_verdict("explainability", "변수 의미 사전 · 사유 코드 · V 변수 제외 모델", "분석·보고", "수행 완료",
                f"V {len(V_NAMES)}개 제외 시 PR Δ{c['d_pr']:+.4f}, ROC Δ{c['d_roc']:+.4f}. "
                f"상위 300건 중 해석 가능한 사유 제시 비율 {cover:.0%}", d_pr=c["d_pr"])

[screen_no_v] fold 1: ROC 0.9281 PR 0.6124 | 트리 212 | 피처 143 | 20s
[screen_no_v] fold 3: ROC 0.9010 PR 0.5058 | 트리 483 | 피처 143 | 87s
[screen_no_v] fold 5: ROC 0.9266 PR 0.5949 | 트리 263 | 피처 143 | 105s
screen_no_v vs screen_clean (3 fold) | PR 0.5685 vs 0.6153 (Δ-0.0467) | ROC 0.9185 vs 0.9231 (Δ-0.0046) | PR 개선 fold 0%
▶ 변수 의미 사전 · 사유 코드 · V 변수 제외 모델: **수행 완료** — V 178개 제외 시 PR Δ-0.0467, ROC Δ-0.0046. 상위 300건 중 해석 가능한 사유 제시 비율 100%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 지역·결제환경 의존 피처 제거 실험]</b> 국내 환경 적용 시 달라질 점 — 심사 개선의견: "국내 결제 환경에 적용할 때 달라질 점 논의"</div>

IEEE-CIS 데이터는 해외(주로 미국) 카드 결제 데이터로 알려져 있으므로, 지역·결제환경 의존 변수는 국내에서 그대로 쓰이지 않을 수 있습니다. 데이터로 확인할 수 있는 범위는 **"이 변수군을 제외해도 성능이 유지되는가"** 입니다.

- 지역 의존: `addr1`, `addr2`, `dist1`, `dist2`, `addr1_FE`
- 결제환경 의존: `P_emaildomain`, `R_emaildomain`, `P_emaildomain_FE`

이 변수들을 제거한 모델의 성능 하락폭이 작으면 모델이 환경 의존적 정보에 덜 기대고 있다는 뜻이고, 크면 국내 적용 시 **해당 변수를 국내 데이터로 다시 설계·재학습**해야 한다는 뜻입니다. (국내 데이터가 없으므로 국내 성능은 이 노트북에서 알 수 없습니다.)

In [23]:
# 🟩 [수인 개선 · 지역·결제환경 의존 피처 제거 실험] (분석, 판정 없음)
REGION = [n for n in ["addr1", "addr2", "dist1", "dist2", "addr1_FE"] if n in COL]  # 지역 의존 변수 중 존재하는 것
ENV = [n for n in ["P_emaildomain", "R_emaildomain", "P_emaildomain_FE"] if n in COL]  # 결제환경 의존 변수 중 존재하는 것
if RUN["region_dependence"]:  # 실행 스위치가 켜져 있을 때만 수행
    # 실험 이름 → (결과 태그, 제거할 변수 목록)
    region_exps = {"지역 제거": ("screen_no_region", REGION),
                   "이메일 제거": ("screen_no_env", ENV),
                   "둘 다 제거": ("screen_no_region_env", REGION + ENV)}
    cs = {}  # 실험별 기준 모델 대비 비교 결과
    for label, (tag, dropped) in region_exps.items():  # 실험을 하나씩 수행
        # 해당 변수를 제외한 피처 구성으로 스크리닝 학습
        builder = make_builder(True, drop=dropped)  # 해당 변수를 제외하는 피처 빌더
        # 해당 변수를 제외한 피처 구성으로 스크리닝 학습
        cv_run(tag, SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, builder, es_mode="innerfit")
        cs[label] = compare(tag, "screen_clean", quiet=True)  # 누수 제거 기준 모델과 비교
    region_tbl = pd.DataFrame({label: {"제거 변수": len(region_exps[label][1]), "PR-AUC": v["pr"], "기준 PR-AUC": v["ref_pr"],
                                       "ΔPR": v["d_pr"], "ΔROC": v["d_roc"]} for label, v in cs.items()}).T  # 실험별 비교 표
    display(region_tbl.round(4))  # 비교 표 출력
    # 분석 결과 기록 (채택/기각 판정 없음)
    add_verdict("region_dependence", "지역·결제환경 의존 피처 제거 실험", "분석·보고", "수행 완료",
                f"지역 {REGION} 제거 PR Δ{cs['지역 제거']['d_pr']:+.4f} / 이메일 {ENV} 제거 PR Δ{cs['이메일 제거']['d_pr']:+.4f} / "
                f"둘 다 제거 PR Δ{cs['둘 다 제거']['d_pr']:+.4f}", d_pr=cs["둘 다 제거"]["d_pr"])

[screen_no_region] fold 1: ROC 0.9341 PR 0.6581 | 트리 461 | 피처 316 | 264s
[screen_no_region] fold 3: ROC 0.8855 PR 0.5309 | 트리 371 | 피처 316 | 200s
[screen_no_region] fold 5: ROC 0.9167 PR 0.5805 | 트리 156 | 피처 316 | 147s
[screen_no_env] fold 1: ROC 0.9270 PR 0.6359 | 트리 274 | 피처 318 | 144s
[screen_no_env] fold 3: ROC 0.8888 PR 0.5298 | 트리 383 | 피처 318 | 186s
[screen_no_env] fold 5: ROC 0.9210 PR 0.5784 | 트리 174 | 피처 318 | 140s
[screen_no_region_env] fold 1: ROC 0.9204 PR 0.6125 | 트리 163 | 피처 313 | 154s
[screen_no_region_env] fold 3: ROC 0.8863 PR 0.4982 | 트리 395 | 피처 313 | 280s
[screen_no_region_env] fold 5: ROC 0.9066 PR 0.5202 | 트리 214 | 피처 313 | 188s


,제거 변수,PR-AUC,기준 PR-AUC,ΔPR,ΔROC
지역 제거,5.0,0.5941,0.6153,-0.0212,-0.0119
이메일 제거,3.0,0.5810,0.6153,-0.0343,-0.0099
둘 다 제거,8.0,0.5463,0.6153,-0.0690,-0.0195


▶ 지역·결제환경 의존 피처 제거 실험: **수행 완료** — 지역 ['addr1', 'addr2', 'dist1', 'dist2', 'addr1_FE'] 제거 PR Δ-0.0212 / 이메일 ['P_emaildomain', 'R_emaildomain', 'P_emaildomain_FE'] 제거 PR Δ-0.0343 / 둘 다 제거 PR Δ-0.0690


<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 지역·결제환경 의존 피처 제거 실험 해석 (위 결과를 읽는 법)</div>

- 하락폭이 **작으면**: 해당 변수군은 모델의 핵심 의존 대상이 아니므로 국내 적용 시 제외하거나 국내 대응 변수로 교체해도 부담이 적습니다.
- 하락폭이 **크면**: 해당 변수군이 성능에 중요하므로, 국내 데이터에서는 같은 역할을 하는 변수(예: 국내 지역·발급사·결제수단 정보)를 새로 정의하고 재학습·재검증해야 합니다.
- 어느 쪽이든 **국내 성능은 이 데이터로는 검증할 수 없다**는 점은 한계로 남습니다. 국내 적용 시 필요한 조치는 ① 변수 재설계 ② 국내 라벨 기반 재학습 ③ 임계치·비용 가정 재산정입니다.

<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 가설과 모델 근거 연결(접속·기기 변수군 제거 실험)]</b> 가설 기각·채택 근거 — 심사 현장 질문: "가설을 어떤 근거로 기각/채택했는가"</div>

EDA 가설 판정은 통합 EDA 표시본(`IEEE_CIS_EDA_조원전체통합_수인표시본.ipynb`)에 기록되어 있습니다. 이 노트북은 **가설이 모델링 단계에서도 지지되는지**를 모델 실험으로 연결합니다. 아래 대응표의 가설 번호·이름은 통합 EDA의 제목을 그대로 가져왔고, 오른쪽의 모델 근거는 이 노트북에서 실제 계산한 값입니다.

| EDA 가설 (통합 EDA 제목) | 모델링에서 확인하는 방법 | 이 노트북의 근거 |
|---|---|---|
| 가설 2 — 상품군·거래금액·카테고리 조합에 따라 사기 위험이 달라질 것이다 | 상품·금액·카드 변수군 gain 비중 | 위 변수군별 모델 의존도 표 |
| 가설 6 — 피처의 실전 유효성은 시간순 검증에서 확인해야 한다 | 시간순 검증 + fold 내부 피처 | 미래 정보 혼입 제거 실험 (GroupKFold vs 순방향 검증) |
| 가설 7 — 상대 시간 신호 | 시간 피처 추가·제거 | 상대 시각·요일 피처 추가 / 시간 의존 피처 제거 영향 분석 |
| 가설 9 — 기기 정보에 따라 사기율이 달라질 것이다 / 가설 10 — Identity 정보 | 접속·기기(id) 변수군 제거 | 아래 `screen_no_identity` 실험 |

In [24]:
# 🟩 [수인 개선 · 가설과 모델 근거 연결] 접속·기기(id) 변수군 제거 → 가설 9·10의 모델 근거 (분석, 판정 없음)
ID_NAMES = [n for n in TEAM_FEATURES if re.match(r"^id_\d+|^Device", n)]  # id_*, Device* 변수 이름 목록
if RUN["explainability"] or RUN["region_dependence"]:  # 관련 실행 스위치 중 하나라도 켜져 있으면 수행
    # id/Device 변수를 제외한 피처 구성으로 스크리닝 학습
    cv_run("screen_no_identity", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, make_builder(True, drop=ID_NAMES),
           es_mode="innerfit")
    c_id = compare("screen_no_identity", "screen_clean")  # 누수 제거 기준 모델과 비교
    # 분석 결과 기록 (채택/기각 판정 없음)
    add_verdict("hypothesis_link", "가설과 모델 근거 연결(접속·기기 변수군 제거 실험)", "분석·보고", "수행 완료",
                f"id/Device {len(ID_NAMES)}개 제거 PR Δ{c_id['d_pr']:+.4f}, ROC Δ{c_id['d_roc']:+.4f}", d_pr=c_id["d_pr"])
    try:  # 변수군 의존도 표가 계산되어 있으면 함께 출력
        display(GROUP_IMP.round(4))  # 변수군별 모델 의존도 표 출력
    except NameError:  # 해당 셀이 실행되지 않은 경우
        pass  # 출력을 생략

[screen_no_identity] fold 1: ROC 0.9326 PR 0.6530 | 트리 316 | 피처 262 | 225s
[screen_no_identity] fold 3: ROC 0.8928 PR 0.5174 | 트리 380 | 피처 262 | 224s
[screen_no_identity] fold 5: ROC 0.9089 PR 0.5446 | 트리 232 | 피처 262 | 101s
screen_no_identity vs screen_clean (3 fold) | PR 0.5704 vs 0.6153 (Δ-0.0449) | ROC 0.9117 vs 0.9231 (Δ-0.0113) | PR 개선 fold 0%
▶ 가설과 모델 근거 연결(접속·기기 변수군 제거 실험): **수행 완료** — id/Device 59개 제거 PR Δ-0.0449, ROC Δ-0.0113


,gain 비중 합,변수 수
변수군,,
V(Vesta),0.6824,178
카운팅(C),0.1367,27
접속·기기(id),0.0609,59
시간간격(D),0.0393,23
카드,0.0287,9
일치여부(M),0.0161,9
지역,0.0147,5
이메일,0.0065,3
팀생성-UID집계,0.0057,3


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 추론 지연시간·피처 축소(Top-K)]</b> 실시간 적용성 — 심사 현장 질문: "321개 피처를 실시간으로 만들 때 지연시간은?"</div>

두 가지를 측정합니다.

1. **모델 추론 지연시간**: 거래 1건을 `inplace_predict`로 예측하는 시간(스레드 1개, 반복 측정의 p50/p95).
2. **피처 축소(Top-K)**: 학습 fold의 gain 중요도 상위 K개만 사용했을 때의 성능 변화와 지연시간 변화. (중요도는 각 fold의 **학습 구간 모델**에서만 가져와 검증 정보가 선택에 쓰이지 않습니다.)

**측정 범위의 한계**: 지연시간은 모델 추론 구간만 측정한 값입니다. 실제 서비스에서는 UID 집계·속도 피처를 **저장소(예: 키-값 DB)에서 조회**하는 시간이 별도로 소요되며, 이 노트북은 그 시간을 측정하지 않습니다. 또한 측정은 스크리닝 설정 모델 기준이며, 공식 설정(깊이 12, 최대 2000트리)의 모델은 더 느립니다.

경량화 기준: ΔPR-AUC ≥ −0.005 이고 지연시간 30% 이상 감소 → 채택.

In [25]:
# 🟩 [수인 개선 · 추론 지연시간·피처 축소(Top-K)] 지연시간 측정 + Top-K 피처 축소
def build_topk(K):  # fold별 누수 제거 기준 모델의 gain 상위 K개 변수만 사용하는 피처 빌더를 만든다.
    def b(sp):  # fold 정의를 받아 Top-K 피처 행렬을 만드는 함수
        imp = RESULTS["screen_clean"]["folds"][str(sp["key"])]["imp"]  # 해당 fold 학습 구간 모델의 gain 중요도
        top = set(sorted(imp, key=imp.get, reverse=True)[:K])  # gain 상위 K개 변수
        return make_builder(True, only=top)(sp)  # 상위 K개 변수만 포함한 피처 행렬 생성
    return b  # 빌더 함수 반환


def latency_ms(bst, X, best, n=200):  # 단건 추론 지연시간(ms)의 p50, p95를 반환한다 (스레드 1개, 첫 5회는 예열로 제외).
    bst.set_param({"nthread": 1})  # 단일 스레드로 고정
    ts = []  # 측정 시간 목록(ms)
    for i in range(min(n, len(X))):  # 최대 n건을 1건씩 예측
        x = np.ascontiguousarray(X[i:i + 1])  # 1행 입력을 연속 메모리 배열로 변환
        t = time.perf_counter()  # 측정 시작 시각
        bst.inplace_predict(x, iteration_range=(0, best), missing=-1, validate_features=False)  # 단건 예측
        ts.append((time.perf_counter() - t) * 1000)  # 경과 시간을 ms로 기록
    ts = np.array(ts[5:])  # 예열 구간 5회 제외
    return float(np.percentile(ts, 50)), float(np.percentile(ts, 95))  # p50, p95 반환


if RUN["latency_topk"]:  # 실행 스위치가 켜져 있을 때만 수행
    f_last = SCREEN_FOLDS[-1]  # 스크리닝 fold 중 마지막 fold 사용
    sp = SPEC_GROUP[f_last]  # 해당 fold의 학습·검증 행 정의
    nfeat_full = len(BUILD_FS.names)  # 전체 피처 수
    KS = [k for k in (30, 60, 100, 150) if k < nfeat_full]  # 전체 피처 수보다 작은 K 후보
    Xtr, ytr, Xva, yva, names, mtr = BUILD_FS(sp)  # 전체 피처 행렬 생성
    # 전체 피처 모델 학습 (모델 객체를 함께 반환)
    _, b_full, _, bst_full = fit_predict_xgb(Xtr, ytr, Xva, yva, names, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN,
                                             "innerfit", mtr, keep_model=True)
    lat_full = latency_ms(bst_full, Xva, b_full)  # 전체 피처 모델의 지연시간
    rows = [dict(K=nfeat_full, 트리수=b_full, p50_ms=lat_full[0], p95_ms=lat_full[1])]  # 결과 표의 첫 행(전체 피처)
    cmp_k = {}  # K별 성능·지연 비교 결과
    for K in KS:  # K 후보를 하나씩 평가
        # Top-K 피처 구성으로 스크리닝 학습
        cv_run(f"screen_top{K}", SPECS_SCREEN, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN, build_topk(K), es_mode="innerfit")
        cmp_k[K] = compare(f"screen_top{K}", "screen_clean", quiet=True)  # 누수 제거 기준 모델과 성능 비교
        imp0 = RESULTS["screen_clean"]["folds"][str(f_last)]["imp"]  # 마지막 fold 학습 구간 모델의 gain 중요도
        top = set(sorted(imp0, key=imp0.get, reverse=True)[:K])  # gain 상위 K개 변수
        Xt, yt, Xv, yv, nm, mt = make_builder(True, only=top)(sp)  # 상위 K개 변수 피처 행렬 생성
        # 지연시간 측정용 Top-K 모델 학습
        _, b_k, _, bst_k = fit_predict_xgb(Xt, yt, Xv, yv, nm, XGB_SCREEN, ROUNDS_SCREEN, ES_SCREEN,
                                           "innerfit", mt, keep_model=True)
        lat_k = latency_ms(bst_k, Xv, b_k)  # Top-K 모델의 지연시간
        rows.append(dict(K=K, 트리수=b_k, p50_ms=lat_k[0], p95_ms=lat_k[1]))  # 결과 표에 행 추가
        cmp_k[K].update(p50=lat_k[0], red=1 - lat_k[0] / lat_full[0])  # p50 지연 감소율 기록
        del Xt, Xv  # 큰 행렬을 해제해 메모리 확보
    tbl = pd.DataFrame(rows)  # 결과 표 생성
    tbl["지연 감소(p50)"] = 1 - tbl["p50_ms"] / tbl["p50_ms"].iloc[0]  # 전체 피처 대비 p50 지연 감소율
    tbl["ΔPR"] = [0.0] + [cmp_k[K]["d_pr"] for K in KS]  # 전체 피처 대비 PR-AUC 변화
    display(tbl.round(4))  # 결과 표 출력
    ok = [K for K in KS if cmp_k[K]["d_pr"] >= -0.005 and cmp_k[K]["red"] >= 0.30]  # 경량화 기준을 충족하는 K 목록
    if ok:  # 기준을 충족하는 K가 있으면
        Kbest = min(ok)  # 가장 작은 K 선택
        # 채택 판정 기록
        add_verdict("latency_topk", f"Top-{Kbest} 피처 축소(지연시간)", "경량화", "채택",
                    f"ΔPR {cmp_k[Kbest]['d_pr']:+.4f}, 지연(p50) {cmp_k[Kbest]['red']:.0%} 감소 "
                    f"(전체 {nfeat_full}개 → {Kbest}개)",
                    d_pr=cmp_k[Kbest]["d_pr"])
    else:  # 기준을 충족하는 K가 없으면
        kk = KS[-1]  # 예시로 가장 큰 K를 보고
        # 기각 판정 기록
        add_verdict("latency_topk", "Top-K 피처 축소(지연시간)", "경량화", "기각",
                    f"기준(ΔPR≥-0.005 & 지연 30%↓) 충족 K 없음. 예: K={kk}: ΔPR {cmp_k[kk]['d_pr']:+.4f}, "
                    f"지연 {cmp_k[kk]['red']:.0%} 감소", d_pr=cmp_k[kk]["d_pr"])
    print(f"전체 {nfeat_full}개 모델 단건 추론: p50 {lat_full[0]:.3f} ms / p95 {lat_full[1]:.3f} ms (스레드 1, 모델 추론만)")  # 지연 요약
    del Xtr, Xva  # 큰 행렬을 해제해 메모리 확보

[screen_top30] fold 1: ROC 0.8735 PR 0.5093 | 트리 98 | 피처 30 | 17s
[screen_top30] fold 3: ROC 0.8631 PR 0.4469 | 트리 61 | 피처 30 | 11s
[screen_top30] fold 5: ROC 0.8631 PR 0.4446 | 트리 63 | 피처 30 | 9s
[screen_top60] fold 1: ROC 0.8887 PR 0.5670 | 트리 162 | 피처 60 | 32s
[screen_top60] fold 3: ROC 0.8524 PR 0.4706 | 트리 244 | 피처 60 | 36s
[screen_top60] fold 5: ROC 0.8729 PR 0.4720 | 트리 67 | 피처 60 | 17s
[screen_top100] fold 1: ROC 0.9089 PR 0.6156 | 트리 475 | 피처 100 | 43s
[screen_top100] fold 3: ROC 0.8534 PR 0.4982 | 트리 267 | 피처 100 | 29s
[screen_top100] fold 5: ROC 0.8993 PR 0.5605 | 트리 277 | 피처 100 | 63s
[screen_top150] fold 1: ROC 0.9300 PR 0.6532 | 트리 380 | 피처 150 | 112s
[screen_top150] fold 3: ROC 0.8977 PR 0.5522 | 트리 371 | 피처 150 | 99s
[screen_top150] fold 5: ROC 0.9008 PR 0.5238 | 트리 151 | 피처 150 | 50s


,K,트리수,p50_ms,p95_ms,지연 감소(p50),ΔPR
0,321,260,1.0590,1.4116,0.0000,0.0000
1,30,63,0.2242,0.3402,0.7883,-0.1457
2,60,67,0.7497,1.0772,0.2921,-0.1093
3,100,277,1.0492,1.3279,0.0093,-0.0537
4,150,151,0.8432,1.0261,0.2038,-0.0462


▶ Top-K 피처 축소(지연시간): **기각** — 기준(ΔPR≥-0.005 & 지연 30%↓) 충족 K 없음. 예: K=150: ΔPR -0.0462, 지연 20% 감소
전체 321개 모델 단건 추론: p50 1.059 ms / p95 1.412 ms (스레드 1, 모델 추론만)


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 최종 모델]</b> 채택된 개선 항목만 조합해 공식 설정(팀과 같은 하이퍼파라미터, 6-fold)으로 최종 검증</div>

스크리닝에서 **채택**된 성능 개선 항목과 평가 타당성 수정 항목(학습 구간 기준 피처 생성, 내부 검증 기반 early stopping)을 합쳐 팀 공식 설정으로 6개 fold를 모두 학습합니다. 비교 대상은 다음 세 가지입니다.

| 이름 | 내용 |
|---|---|
| **팀 OOF** | 팀 최종 결과 파일(전체 데이터 통계 피처 + 검증 fold early stopping) |
| **누수 제거 기준 모델** (`final_clean`) | 팀 피처·팀 하이퍼파라미터를 사용하되 학습 구간 통계 피처 + 내부 검증 early stopping을 적용 (평가 타당성 수정만 반영) |
| **최종 모델** (`final_model`) | 누수 제거 기준 모델 + 스크리닝에서 채택된 성능 개선 항목 |

해석 순서: **팀 OOF → 누수 제거 기준 모델**의 차이는 평가 방식을 바로잡은 효과이며 성능이 내려가도 그대로 보고합니다. **누수 제거 기준 모델 → 최종 모델**의 차이가 수인이 추가한 성능 개선분입니다. 실행 시간이 가장 긴 구간이며, 중간에 멈춰도 fold 단위로 이어서 실행됩니다.

In [26]:
# 🟩 [수인 개선 · 최종 모델] 채택 항목 조합 → 공식 설정 6-fold
def adopted(id_):  # 판정 id가 "채택"으로 기록되었는지 확인하는 함수
    return VERDICTS.get(id_, {}).get("판정") == "채택"  # 판정 미기록이면 빈 dict로 처리해 False 반환
FINAL_EXTRA = []  # 최종 모델에 추가할 (피처 이름 목록, 피처 행렬) 쌍의 목록
FINAL_PARTS = ["학습 구간 기준 피처", "내부 검증 기반 early stopping"]  # 최종 모델 구성 요소 설명 목록
if adopted("velocity"):  # 과거 이력 기반 속도 피처가 채택된 경우
    FINAL_EXTRA.append((VEL_NAMES, VEL))  # 속도 피처를 추가 피처 목록에 등록
    FINAL_PARTS.append("과거 이력 기반 속도 피처")  # 구성 요소 설명에 추가
if adopted("time_add"):  # 상대 시각·요일 피처가 채택된 경우
    FINAL_EXTRA.append((TIME_NAMES, TIME_NEW))  # 시간 피처를 추가 피처 목록에 등록
    FINAL_PARTS.append("상대 시각·요일 피처")  # 구성 요소 설명에 추가
XGB_FINAL = dict(XGB_OFFICIAL, **TUNE_OVER)  # 공식 하이퍼파라미터에 튜닝에서 채택된 변경분을 덮어씀
if TUNE_OVER:  # 하이퍼파라미터 변경이 채택된 경우
    FINAL_PARTS.append(f"하이퍼파라미터 변경 {TUNE_OVER}")  # 변경 내용을 구성 요소 설명에 추가
USE_ENS = adopted("ensemble")  # XGBoost+LightGBM 앙상블 채택 여부
if USE_ENS:  # 앙상블이 채택된 경우
    FINAL_PARTS.append("XGBoost+LightGBM 평균 앙상블")  # 구성 요소 설명에 추가
print("최종 모델 구성:", FINAL_PARTS)  # 최종 모델 구성 출력

OOF = {}  # 모델별 전체 행 OOF 예측 저장 딕셔너리
def oof_from(tag):  # 결과 태그의 fold별 검증 예측을 전체 행 배열로 합치는 함수
    o = np.full(len(y_all), np.nan, dtype="float64")  # 전체 행을 NaN으로 초기화
    for r in RESULTS[tag]["folds"].values():  # 각 fold 결과를 순회
        o[r["va"]] = r["pred"]  # 검증 행 위치에 해당 fold 예측을 기록
    return o  # 전체 행 OOF 배열 반환

if RUN["final"]:  # 최종 모델 실행 스위치가 켜진 경우
    # 누수 제거 기준 모델: 학습 구간 피처 + 내부 검증 early stopping
    cv_run("final_clean", SPECS_ALL, XGB_OFFICIAL, ROUNDS_OFFICIAL, ES_OFFICIAL, BUILD_FS, es_mode="inner")
    if FINAL_EXTRA or TUNE_OVER:  # 추가 피처 또는 튜닝 변경이 있는 경우
        # 채택된 피처·하이퍼파라미터로 XGBoost 최종 학습
        cv_run("final_xgb", SPECS_ALL, XGB_FINAL, ROUNDS_OFFICIAL, ES_OFFICIAL,
               make_builder(True, extra=FINAL_EXTRA or None), es_mode="inner")
    else:  # 변경이 없는 경우
        RESULTS["final_xgb"] = RESULTS["final_clean"]  # 누수 제거 기준 모델 결과를 그대로 사용
    if USE_ENS:  # 앙상블이 채택된 경우
        # LightGBM 공식 설정으로 6-fold 학습
        cv_run("final_lgb", SPECS_ALL, None, None, None, make_builder(True, extra=FINAL_EXTRA or None),
               fit_fn=make_lgb_fit(LGB_OFFICIAL, ROUNDS_OFFICIAL, ES_OFFICIAL))
        blend_store("final_model", ["final_xgb", "final_lgb"])  # 두 모델 예측을 평균해 최종 결과로 저장
    else:  # 앙상블을 쓰지 않는 경우
        RESULTS["final_model"] = RESULTS["final_xgb"]  # XGBoost 결과를 최종 모델로 사용
    OOF["누수 제거 기준"] = oof_from("final_clean")  # 누수 제거 기준 모델의 OOF 예측
    OOF["최종 모델"] = oof_from("final_model")  # 최종 모델의 OOF 예측
    if OOF_TEAM is not None:  # 팀 OOF 파일이 있는 경우
        OOF["팀 OOF"] = OOF_TEAM  # 팀 OOF 예측 추가
    rows = []  # 모델별 지표 행 목록
    for nm, o in OOF.items():  # 각 모델 OOF 순회
        m = ~np.isnan(o)  # 예측이 존재하는 행 마스크
        rows.append(dict(모델=nm, ROC_AUC=roc_auc_score(y_all[m], o[m]),
                         PR_AUC=average_precision_score(y_all[m], o[m])))  # 모델별 전체 지표 계산
    res = pd.DataFrame(rows).set_index("모델")  # 모델별 지표 표
    display(res.round(4))  # 지표 표 출력
    mon = pd.DataFrame(
        {nm: [average_precision_score(y_all[(DT_M == m_) & ~np.isnan(o)], o[(DT_M == m_) & ~np.isnan(o)])
              for m_ in MONTHS] for nm, o in OOF.items()},
        index=MONTHS).rename_axis("월")  # 모델별·월별 PR-AUC 표
    display(mon.round(4))  # 월별 PR-AUC 출력
    d_h = res.loc["최종 모델", "PR_AUC"] - res.loc["누수 제거 기준", "PR_AUC"]  # 성능 개선분(PR-AUC 차이)
    print(f"누수 제거 기준 → 최종 모델 PR-AUC 변화: {d_h:+.4f} (수인이 추가한 성능 개선분)")  # 개선분 출력
    if "팀 OOF" in res.index:  # 팀 OOF가 있는 경우
        d_t = res.loc["누수 제거 기준", "PR_AUC"] - res.loc["팀 OOF", "PR_AUC"]  # 평가 방식 변경 효과
        print(f"팀 OOF → 누수 제거 기준 PR-AUC 변화: {d_t:+.4f} (평가 방식을 바로잡은 효과)")  # 변경 효과 출력
    if d_h < 0:  # 최종 모델이 기준보다 낮은 경우
        print("⚠ 최종 조합이 누수 제거 기준 모델보다 낮습니다. 스크리닝 채택 항목이 공식 설정에서 재현되지 않은 것이므로 "
              "판정표에 '공식 설정 미재현'으로 표시합니다.")  # 미재현 경고 출력
    oof_out = pd.DataFrame(OOF).assign(y=y_all, month=DT_M, amt=RAW["TransactionAmt"])  # 저장용 표(정답·월·금액 포함)
    oof_out.to_parquet(OUT / "oof_all.parquet")  # OOF 예측을 parquet 파일로 저장
    FINAL_OK = d_h >= 0  # 최종 모델이 기준 이상이면 True
else:  # 실행 스위치가 꺼진 경우
    FINAL_OK = True  # 미재현 표시를 하지 않음

최종 모델 구성: ['학습 구간 기준 피처', '내부 검증 기반 early stopping', 'XGBoost+LightGBM 평균 앙상블']
[final_clean] fold 0: ROC 0.8873 PR 0.5369 | 트리 710 | 피처 321 | 578s
[final_clean] fold 1: ROC 0.9437 PR 0.7014 | 트리 696 | 피처 321 | 1009s
[final_clean] fold 2: ROC 0.9447 PR 0.7104 | 트리 510 | 피처 321 | 813s
[final_clean] fold 3: ROC 0.9360 PR 0.6366 | 트리 631 | 피처 321 | 669s
[final_clean] fold 4: ROC 0.9469 PR 0.7138 | 트리 491 | 피처 321 | 809s
[final_clean] fold 5: ROC 0.9448 PR 0.6743 | 트리 445 | 피처 321 | 596s
[final_lgb] fold 0: ROC 0.9072 PR 0.4956 | 트리 471 | 피처 321 | 163s
[final_lgb] fold 1: ROC 0.9443 PR 0.6899 | 트리 544 | 피처 321 | 73s
[final_lgb] fold 2: ROC 0.9503 PR 0.7280 | 트리 458 | 피처 321 | 31s
[final_lgb] fold 3: ROC 0.9150 PR 0.5598 | 트리 587 | 피처 321 | 113s
[final_lgb] fold 4: ROC 0.9583 PR 0.7469 | 트리 548 | 피처 321 | 177s
[final_lgb] fold 5: ROC 0.9405 PR 0.6318 | 트리 331 | 피처 321 | 137s


,ROC_AUC,PR_AUC
모델,,
누수 제거 기준,0.9328,0.6571
최종 모델,0.9410,0.6620
팀 OOF,0.9492,0.7082


,누수 제거 기준,최종 모델,팀 OOF
월,,,
12,0.5369,0.5335,0.5674
13,0.7104,0.7333,0.7531
14,0.7138,0.7488,0.7742
15,0.7014,0.7072,0.7456
16,0.6743,0.6777,0.7603
17,0.6366,0.6156,0.6715


누수 제거 기준 → 최종 모델 PR-AUC 변화: +0.0049 (수인이 추가한 성능 개선분)
팀 OOF → 누수 제거 기준 PR-AUC 변화: -0.0511 (평가 방식을 바로잡은 효과)


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 임계치 통합표·권장 임계치]</b> 임계치 하나로 정리 — 심사 개선의견·질문 3·총평: "기준값이 0.0329 / 0.35 / 0.017로 다르게 제시, 하나로 정리"</div>

팀 문서에는 서로 다른 임계치가 **서로 다른 목적·평가표본**에서 나온 채로 나란히 있습니다.

| 값 | 팀 문서에서의 의미 (출처) |
|---|---|
| 0.0329 | OOF에서 Recall 80%를 달성하는 임계치 (`plot_threshold_analysis.py`) |
| 0.1867 | OOF에서 F1이 최대인 임계치 |
| 0.021 | OOF에서 비용이 최소인 임계치 (`SUMMARY_DRAFT.md`) |
| 0.05 | Tableau 대시보드 임계치 격자의 경계값 (0.05 아래에는 값이 없음) |
| 0.35 | 별도의 75/25 분할 표본에서 쓴 값 (`REPORT_REVISION_PLAN.md`) |
| 0.017 | 팀 파일에서 출처를 찾지 못함 |

**정리 방식**: ① **하나의 평가표본**(위 최종 OOF, 6개 fold 전체)과 ② **하나의 비용식**으로 모든 후보를 같은 표에 놓고, ③ 비용이 최소인 임계치를 **권장값 1개**로 확정합니다. 비용식은 팀 문서의 가정을 그대로 씁니다(아래 `ASSUMPTION`).
- 알림(심사) 1건당 비용 = **$5.43** (팀 가정), 놓친 사기의 손실 = **거래 금액** (팀 가정)
- 이 값들은 이 노트북이 확인한 사실이 아니라 **가정**이며, 비용 가정 민감도 분석에서 변화를 확인합니다.

In [27]:
# 🟩 [수인 개선 · 임계치 통합표·권장 임계치] 후보 통합표 + 권장 임계치 (비용식·평가표본 통일)
# 팀 가정(알림당 $5.43, 놓친 사기=거래금액). 민감도 분석에서 값을 바꿔 영향을 확인
ASSUMPTION = dict(c_alert=5.43, fn_mult=1.0, recover=1.0)
AMT = RAW["TransactionAmt"].astype("float64")  # 거래 금액 배열(손실 계산용)

def cost_curve(score, y, amt, c_alert, fn_mult=1.0, recover=1.0):  # 상위 k건 알림 시 k별 총비용 곡선 계산
    # 내림차순 정렬 후 상위 k건 알림. 반환: 정렬 점수, k별 비용. 비용 = 알림비용 + 놓친 사기 손실 + (1-회수율)*잡은 사기 손실
    o = np.argsort(-score, kind="stable")  # 점수 내림차순 정렬 순서(동점은 원래 순서 유지)
    s = score[o]  # 정렬된 점수
    yy = y[o]  # 정렬된 정답
    aa = amt[o] * yy  # 정렬된 사기 거래 금액(정상 거래는 0)
    k = np.arange(1, len(s) + 1)  # 알림 건수 1..N
    caught = np.cumsum(aa)  # 상위 k건에서 잡은 사기 금액 누적합
    tot = aa.sum()  # 전체 사기 금액
    cost = c_alert * k + fn_mult * (tot - caught) + fn_mult * (1 - recover) * caught  # k별 총비용
    cost = np.r_[fn_mult * tot, cost]  # 알림이 없는 경우(k=0) 비용을 맨 앞에 추가
    return s, cost, tot  # 정렬 점수, 비용 곡선, 전체 사기 금액 반환

def metrics_at(score, y, amt, t, c_alert=5.43):  # 임계치 t 적용 시 성능·비용 지표 계산
    a = score >= t  # 알림 대상 마스크
    tp = int((a & (y == 1)).sum())  # 알림 중 실제 사기 건수
    fp = int((a & (y == 0)).sum())  # 알림 중 정상 건수
    fn = int((~a & (y == 1)).sum())  # 놓친 사기 건수
    missed = float(amt[~a & (y == 1)].sum())  # 놓친 사기 금액 합계
    return dict(임계치=t, 알림건수=int(a.sum()), 알림비율=a.mean(), Recall=tp / max(tp + fn, 1),
                Precision=tp / max(tp + fp, 1), FP=fp, FN=fn, 총비용=c_alert * a.sum() + missed)  # 지표 딕셔너리 반환

if RUN["threshold_policy"] and "최종 모델" in OOF:  # 임계치 분석 스위치가 켜져 있고 최종 모델 OOF가 있는 경우
    S = OOF["최종 모델"]  # 최종 모델 OOF 점수
    m_ok = ~np.isnan(S)  # 예측이 있는 행 마스크
    S_, Y_, A_ = S[m_ok], y_all[m_ok], AMT[m_ok]  # 평가 대상 점수·정답·금액
    s_sorted, cost_k, tot_fraud_amt = cost_curve(S_, Y_, A_, **ASSUMPTION)  # 비용 곡선 계산
    kbest = int(np.argmin(cost_k))  # 비용이 최소인 알림 건수
    T_RAW = float(s_sorted[kbest - 1]) if kbest > 0 else 1.0  # 비용 최소 지점의 점수를 임계치로 확정
    TEAM_T = {"0.0329 (Recall80%)": 0.0329, "0.1867 (F1최대)": 0.1867, "0.021 (비용최소·팀)": 0.021,
              "0.05 (Tableau 경계)": 0.05, "0.35 (별도표본)": 0.35, "0.017 (출처불명)": 0.017}  # 팀 문서의 임계치 후보
    rows = [dict(후보=n, **metrics_at(S_, Y_, A_, t)) for n, t in TEAM_T.items()]  # 후보별 지표 행
    rows.append(dict(후보="★ 수인 권장 (비용최소, 최종모델 확률 기준)", **metrics_at(S_, Y_, A_, T_RAW)))  # 권장 임계치 행 추가
    tT = pd.DataFrame(rows).set_index("후보")  # 임계치 통합표
    display(tT.round(4))  # 통합표 출력
    base_none = float(tot_fraud_amt)  # 아무것도 막지 않을 때의 손실
    base_all = ASSUMPTION["c_alert"] * len(S_)  # 전건 심사 시 비용
    print(f"참고: 모델 없이 아무것도 안 막으면 손실 ${base_none:,.0f} / 모두 심사하면 ${base_all:,.0f} / "
          f"권장 임계치 {T_RAW:.4f} 적용 시 ${cost_k[kbest]:,.0f}")  # 기준 비용 비교 출력
    # 안정성 점검: 월별 비용최소 임계치와 비용곡선 평탄성
    per_m = []  # (월, 월별 비용최소 임계치) 목록
    for m_ in MONTHS:  # 월별 순회
        mm = DT_M[m_ok] == m_  # 해당 월 마스크
        if mm.sum() < 100 or Y_[mm].sum() < 5:  # 표본이나 사기 건수가 너무 적으면 건너뜀
            continue  # 다음 월로 이동
        ss, cc, _ = cost_curve(S_[mm], Y_[mm], A_[mm], **ASSUMPTION)  # 월별 비용 곡선
        kk = int(np.argmin(cc))  # 월별 비용 최소 알림 건수
        per_m.append((m_, float(ss[kk - 1]) if kk > 0 else np.nan))  # 월별 최적 임계치 기록
    print("월별 비용최소 임계치:", {m_: round(t, 4) for m_, t in per_m})  # 월별 임계치 출력
    tgrid = np.unique(np.round(np.quantile(S_, np.linspace(0.5, 0.999, 120)), 5))  # 임계치 탐색 격자(분위수 기반)
    flat = [(t, cost_k[int((S_ >= t).sum())]) for t in tgrid]  # 격자별 총비용
    best_c = min(c for _, c in flat)  # 격자 내 최소 비용
    within = [t for t, c in flat if c <= best_c * 1.02]  # 최소 비용의 +2% 이내 임계치
    print(f"비용이 최솟값의 +2% 이내인 임계치 구간: {min(within):.4f} ~ {max(within):.4f}  "
          "← 권장값 주변이 평탄할수록 임계치 변화에 대한 비용 민감도가 낮음")  # 평탄 구간 출력
    tT.to_csv(OUT / "임계치_통합표.csv", encoding="utf-8-sig")  # 통합표 CSV 저장
    add_verdict("threshold", "임계치 후보 통합표 + 권장 임계치 1개", "분석·보고", "수행 완료",
                f"권장 임계치 {T_RAW:.4f} (비용최소, 가정: 알림 $5.43·놓친 사기=거래금액). "
                f"Recall {tT.iloc[-1]['Recall']:.1%}, 알림비율 {tT.iloc[-1]['알림비율']:.2%}", t_raw=T_RAW)  # 판정 기록

,임계치,알림건수,알림비율,Recall,Precision,FP,FN,총비용
후보,,,,,,,,
0.0329 (Recall80%),0.0329,98953,0.1676,0.8646,0.1806,81087,2797,977762.284
0.1867 (F1최대),0.1867,19332,0.0327,0.5956,0.6366,7025,8356,1661416.824
0.021 (비용최소·팀),0.0210,155145,0.2627,0.9187,0.1224,136161,1679,1089112.919
0.05 (Tableau 경계),0.0500,63989,0.1084,0.8022,0.2590,47413,4087,1025402.291
0.35 (별도표본),0.3500,12537,0.0212,0.4942,0.8145,2325,10451,1938014.319
0.017 (출처불명),0.0170,187971,0.3183,0.9379,0.1031,168591,1283,1200581.120
"★ 수인 권장 (비용최소, 최종모델 확률 기준)",0.0359,90423,0.1531,0.8539,0.1951,72779,3019,964675.486


참고: 모델 없이 아무것도 안 막으면 손실 $3,083,845 / 모두 심사하면 $3,206,632 / 권장 임계치 0.0359 적용 시 $964,675
월별 비용최소 임계치: {12: 0.0461, 13: 0.036, 14: 0.0335, 15: 0.0243, 16: 0.0344, 17: 0.0302}
비용이 최솟값의 +2% 이내인 임계치 구간: 0.0319 ~ 0.0417  ← 권장값 주변이 평탄할수록 임계치 변화에 대한 비용 민감도가 낮음
▶ 임계치 후보 통합표 + 권장 임계치 1개: **수행 완료** — 권장 임계치 0.0359 (비용최소, 가정: 알림 $5.43·놓친 사기=거래금액). Recall 85.4%, 알림비율 15.31%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 예측 확률 교차 보정]</b> 확률 보정 — 비용 기반 임계치가 의미를 가지려면 예측값이 실제 확률에 가까워야 합니다</div>

**발견한 미흡한 점**: 팀의 비용 기반 임계치(0.021 등)는 모델 출력을 **실제 사기 확률로** 취급합니다. 그런데 최종 모델은 보정을 하지 않았고, 팀 `SUMMARY_DRAFT.md`도 이를 한계로 적고 있습니다. 월별 사기율이 달라 같은 점수가 월마다 다른 의미를 가질 수 있습니다.

**개선**: 교차 보정 — fold *f*의 점수는 **다른 fold들의 (점수, 정답)** 으로 학습한 isotonic 회귀로 보정합니다(자기 fold 정답을 쓰지 않음). 보정 전·후의 Brier score, ECE(구간 10개)를 비교합니다. 평가 타당성 수정 항목이므로 성능 수치와 무관하게 채택하되, 임계치에 미친 영향을 보고합니다.

In [28]:
# 🟩 [수인 개선 · 예측 확률 교차 보정] 교차 보정(isotonic) 및 보정 품질 평가
from sklearn.isotonic import IsotonicRegression  # 단조 증가 비모수 회귀(확률 보정용)

def ece(p, y, bins=10):  # Expected Calibration Error 계산(분위수 구간 기준)
    q = np.quantile(p, np.linspace(0, 1, bins + 1))  # 예측값 분위수 경계
    q[-1] += 1e-12  # 마지막 경계에 최댓값이 포함되도록 미세 보정
    idx = np.clip(np.searchsorted(q, p, side="right") - 1, 0, bins - 1)  # 각 예측값의 구간 번호
    return float(sum((idx == b).mean() * abs(p[idx == b].mean() - y[idx == b].mean())
                     for b in range(bins) if (idx == b).any()))  # 구간 비중 가중 |평균 예측 - 실제 비율| 합

if RUN["threshold_policy"] and "최종 모델" in OOF:  # 임계치 분석 스위치가 켜져 있고 최종 모델 OOF가 있는 경우
    S = OOF["최종 모델"]  # 최종 모델 OOF 점수
    CAL = np.full(len(S), np.nan)  # 보정된 확률 저장 배열
    for f in range(N_SPLITS):  # fold별 교차 보정
        te = FOLD_OF_ROW == f  # 현재 fold의 검증 행
        trn = (~te) & ~np.isnan(S)  # 보정기 학습에 쓸 다른 fold 행
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(S[trn], y_all[trn])  # 다른 fold로 보정기 학습
        CAL[te] = iso.predict(S[te])  # 현재 fold 점수를 보정
    CAL = np.clip(CAL, 1e-6, 1)  # 확률이 0이 되지 않도록 하한 적용
    ok = ~np.isnan(S)  # 예측이 있는 행 마스크
    before = [brier_score_loss(y_all[ok], S[ok]), ece(S[ok], y_all[ok]),
              roc_auc_score(y_all[ok], S[ok]), average_precision_score(y_all[ok], S[ok])]  # 보정 전 지표
    after = [brier_score_loss(y_all[ok], CAL[ok]), ece(CAL[ok], y_all[ok]),
             roc_auc_score(y_all[ok], CAL[ok]), average_precision_score(y_all[ok], CAL[ok])]  # 보정 후 지표
    tab = pd.DataFrame({"보정 전": before, "보정 후": after}, index=["Brier", "ECE(10구간)", "ROC-AUC", "PR-AUC"])  # 비교표
    display(tab.round(5))  # 비교표 출력
    # 보정 후 점수로 비용최소 임계치 재계산
    s2, c2, _ = cost_curve(CAL[ok], y_all[ok], AMT[ok], **ASSUMPTION)  # 보정 확률 기준 비용 곡선
    k2 = int(np.argmin(c2))  # 비용이 최소인 알림 건수
    T_CAL = float(s2[k2 - 1]) if k2 > 0 else 1.0  # 보정 확률 기준 권장 임계치
    print(f"보정 후 확률 기준 비용최소 임계치: {T_CAL:.4f} (보정 전 점수 기준 {T_RAW:.4f}) | "
          f"비용 ${c2[k2]:,.0f} vs 보정 전 ${cost_k[kbest]:,.0f}")  # 임계치·비용 비교 출력
    print("해석: ROC/PR-AUC는 순위 지표이므로 isotonic(단조변환)으로 거의 변하지 않고, "
          "Brier/ECE가 줄어들수록 점수가 실제 확률에 가까워집니다.")  # 해석 출력
    add_verdict("calibration", "교차 보정(isotonic) 적용", "평가 타당성 수정", "채택(평가 타당성 수정)",
                f"Brier {tab.loc['Brier','보정 전']:.5f}→{tab.loc['Brier','보정 후']:.5f}, "
                f"ECE {tab.loc['ECE(10구간)','보정 전']:.5f}→{tab.loc['ECE(10구간)','보정 후']:.5f}, "
                f"보정 후 비용최소 임계치 {T_CAL:.4f}", t_cal=T_CAL)  # 판정 기록

,보정 전,보정 후
Brier,0.01850,0.01853
ECE(10구간),0.00624,0.00088
ROC-AUC,0.94097,0.93961
PR-AUC,0.66202,0.65099


보정 후 확률 기준 비용최소 임계치: 0.0302 (보정 전 점수 기준 0.0359) | 비용 $967,664 vs 보정 전 $964,675
해석: ROC/PR-AUC는 순위 지표이므로 isotonic(단조변환)으로 거의 변하지 않고, Brier/ECE가 줄어들수록 점수가 실제 확률에 가까워집니다.
▶ 교차 보정(isotonic) 적용: **채택(평가 타당성 수정)** — Brier 0.01850→0.01853, ECE 0.00624→0.00088, 보정 후 비용최소 임계치 0.0302


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 비용 가정 민감도 분석]</b> 민감도 분석 — 심사 개선의견: "$5.43 가정값에 대한 민감도 분석(2배·0.5배)"</div>

세 가정을 각각 바꿔서 **비용최소 임계치와 절감액이 어떻게 달라지는지** 확인합니다: 알림 1건 비용(0.5배·1배·2배), 놓친 사기 손실 배수(0.5·1·2, 예: 챠지백 수수료·재방문 이탈 반영), 알림으로 잡은 사기를 실제로 막는 비율(100%·80%·50%).

In [29]:
# 🟩 [수인 개선 · 비용 가정 민감도 분석] 비용 가정 민감도 (27개 조합)
if RUN["threshold_policy"] and "최종 모델" in OOF:  # 임계치 분석 스위치가 켜져 있고 최종 모델 OOF가 있는 경우
    y_ok = y_all[ok]  # 평가 대상 정답
    order_ok = np.argsort(-CAL[ok], kind="stable")  # 보정 확률 내림차순 정렬 순서
    rows = []  # 조합별 결과 행 목록
    for cm in (0.5, 1.0, 2.0):  # 알림 비용 배수
        for fm in (0.5, 1.0, 2.0):  # 놓친 사기 손실 배수
            for rc in (1.0, 0.8, 0.5):  # 알림으로 잡은 사기의 방어 성공률
                s_, c_, tot_ = cost_curve(CAL[ok], y_ok, AMT[ok], ASSUMPTION["c_alert"] * cm, fm, rc)  # 조합별 비용 곡선
                k_ = int(np.argmin(c_))  # 비용 최소 알림 건수
                rows.append(dict(알림비용배수=cm, 놓친사기손실배수=fm, 방어성공률=rc,
                                 최적임계치=float(s_[k_ - 1]) if k_ > 0 else 1.0,
                                 알림비율=k_ / len(s_),
                                 Recall=float(y_ok[order_ok][:k_].sum() / y_ok.sum()),
                                 모델없을때비용=fm * tot_, 최소비용=c_[k_], 절감률=1 - c_[k_] / (fm * tot_)))  # 결과 행 추가
    sens = pd.DataFrame(rows)  # 민감도 결과 표
    display(sens.round(4))  # 결과 표 출력
    sens.to_csv(OUT / "민감도_27조합.csv", index=False, encoding="utf-8-sig")  # 결과 CSV 저장
    print(f"최적 임계치 범위: {sens['최적임계치'].min():.4f} ~ {sens['최적임계치'].max():.4f} | "
          f"알림비율 범위: {sens['알림비율'].min():.2%} ~ {sens['알림비율'].max():.2%} | "
          f"절감률 범위: {sens['절감률'].min():.1%} ~ {sens['절감률'].max():.1%}")  # 범위 요약 출력
    add_verdict("sensitivity", "비용 가정 민감도(27조합)", "분석·보고", "수행 완료",
                f"최적 임계치 {sens['최적임계치'].min():.3f}~{sens['최적임계치'].max():.3f}, "
                f"알림비율 {sens['알림비율'].min():.1%}~{sens['알림비율'].max():.1%}, "
                f"모델 도입 절감률 {sens['절감률'].min():.0%}~{sens['절감률'].max():.0%}")  # 판정 기록

,알림비용배수,놓친사기손실배수,방어성공률,최적임계치,알림비율,Recall,모델없을때비용,최소비용,절감률
0,0.5,0.5,1.0,0.0302,0.1530,0.8534,1541922.43,4.838320e+05,0.6862
1,0.5,0.5,0.8,0.0377,0.1415,0.8418,1541922.43,7.413059e+05,0.5192
2,0.5,0.5,0.5,0.0507,0.1082,0.8012,1541922.43,1.114446e+06,0.2772
3,0.5,1.0,1.0,0.0152,0.2405,0.9089,3083844.86,6.659003e+05,0.7841
4,0.5,1.0,0.8,0.0158,0.2312,0.9060,3083844.86,1.223637e+06,0.6032
5,0.5,1.0,0.5,0.0302,0.1530,0.8534,3083844.86,2.025754e+06,0.3431
6,0.5,2.0,1.0,0.0094,0.3734,0.9539,6167689.72,8.415551e+05,0.8636
7,0.5,2.0,0.8,0.0094,0.3734,0.9539,6167689.72,2.026519e+06,0.6714
8,0.5,2.0,0.5,0.0152,0.2405,0.9089,6167689.72,3.749745e+06,0.3920
9,1.0,0.5,1.0,0.0507,0.1082,0.8012,1541922.43,6.869697e+05,0.5545


최적 임계치 범위: 0.0094 ~ 0.2989 | 알림비율 범위: 2.83% ~ 37.34% | 절감률 범위: 11.2% ~ 86.4%
▶ 비용 가정 민감도(27조합): **수행 완료** — 최적 임계치 0.009~0.299, 알림비율 2.8%~37.3%, 모델 도입 절감률 11%~86%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 3단계 정책 시뮬레이션]</b> 중위험 구간 처리 절차 — 심사 현장 질문 1: "중위험 구간의 별도 확인 절차를 구체화"</div>

점수를 3단계로 나눠 **고위험은 수동 심사, 중위험은 추가 인증(예: 문자·앱 인증), 저위험은 자동 승인**하는 정책을 시뮬레이션하고, 2단계(심사/승인) 정책보다 비용이 줄어드는 조건을 찾습니다.

> **중요**: 아래 `c_auth`(추가 인증 1건 비용), `friction`(정상 고객이 인증 때문에 겪는 불편·이탈 비용), `s_auth`(추가 인증으로 사기를 막는 비율)은 **데이터에 없는 가정값**입니다. 실제 값은 운영 부서 확인이 필요하며, 결과는 "이 가정이면 이렇다"는 **조건부 계산**이지 예측이 아닙니다. 대신 **손익분기 추가인증 비용**(이 값보다 낮으면 3단계가 유리)을 구해 가정 의존도를 낮춥니다.

In [30]:
# 🟩 [수인 개선 · 3단계 정책 시뮬레이션] 3단계 정책 시뮬레이션 (가정값은 ASSUME_P에 명시)
# 가정값: 심사비용은 팀 가정, 나머지 3개는 수인이 둔 가정(운영 확인 필요)
ASSUME_P = dict(c_review=5.43, c_auth=0.50, friction=1.00, s_auth=0.70)

def three_tier(score, y, amt, k_hi_grid, k_lo_grid, P):  # 고위험·중위험 경계 격자 탐색으로 최소 비용 조합 반환
    o = np.argsort(-score, kind="stable")  # 점수 내림차순 정렬 순서
    yy = y[o]  # 정렬된 정답
    aa = amt[o] * yy  # 정렬된 사기 거래 금액
    legit = 1 - yy  # 정상 거래 표시(정상=1)
    cum_a = np.r_[0, np.cumsum(aa)]  # 사기 금액 누적합(앞에 0 포함)
    cum_leg = np.r_[0, np.cumsum(legit)]  # 정상 거래 건수 누적합(앞에 0 포함)
    tot = aa.sum()  # 전체 사기 금액
    best = None  # (비용, 고위험 경계 건수, 중위험 경계 건수) 최적 조합
    for kh in k_hi_grid:  # 고위험(수동 심사) 건수 후보
        for kl in k_lo_grid:  # 고위험+중위험 누적 건수 후보
            if kl < kh:  # 중위험 경계가 고위험 경계보다 앞서면 무효
                continue  # 다음 조합으로 이동
            review = P["c_review"] * kh  # 수동 심사 비용
            caught_hi = cum_a[kh]  # 수동 심사로 잡은 사기 금액
            mid_amt = cum_a[kl] - cum_a[kh]  # 중위험 구간의 사기 금액
            mid_leg = cum_leg[kl] - cum_leg[kh]  # 중위험 구간의 정상 거래 건수
            mid_n = kl - kh  # 중위험 구간 거래 건수
            cost = (review + P["c_auth"] * mid_n + P["friction"] * mid_leg
                    + (tot - caught_hi - P["s_auth"] * mid_amt))  # 심사+인증+불편+미방어 사기 손실 합계
            if best is None or cost < best[0]:  # 지금까지의 최소 비용보다 낮으면
                best = (cost, kh, kl)  # 최적 조합 갱신
    return best  # 최적 (비용, kh, kl) 반환

if RUN["threshold_policy"] and "최종 모델" in OOF:  # 임계치 분석 스위치가 켜져 있고 최종 모델 OOF가 있는 경우
    S_c, Y_c, A_c = CAL[ok], y_all[ok], AMT[ok]  # 보정 확률·정답·금액
    n = len(S_c)  # 평가 거래 수
    grid = np.unique(np.r_[0, np.round(np.geomspace(1, n * 0.3, 160)).astype(int)])  # 건수 후보 격자(로그 간격, 최대 30%)
    # 2단계 정책 = 중위험 구간 없음(kl = kh): 비용 곡선에서 직접 계산
    s2_, c2_, tot2 = cost_curve(S_c, Y_c, A_c, ASSUME_P["c_review"])  # 2단계 비용 곡선
    cost_two = float(c2_.min())  # 2단계 최소 비용
    k_two = int(np.argmin(c2_))  # 2단계 최적 알림 건수
    best3 = three_tier(S_c, Y_c, A_c, grid, grid, ASSUME_P)  # 3단계 최적 조합 탐색
    cost3, kh, kl = best3  # 3단계 최소 비용과 경계 건수
    o_ = np.argsort(-S_c, kind="stable")  # 점수 내림차순 정렬 순서
    sc_sorted = S_c[o_]  # 정렬된 점수
    t_hi = float(sc_sorted[kh - 1]) if kh > 0 else 1.0  # 고위험 경계 점수
    t_lo = float(sc_sorted[kl - 1]) if kl > 0 else 1.0  # 중위험 하한 경계 점수
    print(f"2단계 최적: 알림 {k_two:,}건, 비용 ${cost_two:,.0f}")  # 2단계 결과 출력
    print(f"3단계 최적: 수동심사(≥{t_hi:.3f}) {kh:,}건 + 추가인증({t_lo:.3f}~{t_hi:.3f}) {kl-kh:,}건, "
          f"비용 ${cost3:,.0f} → 2단계 대비 {1-cost3/cost_two:+.2%}")  # 3단계 결과 출력
    tier = np.where(S_c >= t_hi, "고위험(수동심사)", np.where(S_c >= t_lo, "중위험(추가인증)", "저위험(자동승인)"))  # 구간 분류
    seg = (pd.DataFrame({"구간": tier, "사기": Y_c, "금액": A_c}).groupby("구간")
           .agg(거래수=("사기", "size"), 사기건수=("사기", "sum"), 사기율=("사기", "mean"),
                사기금액=("금액", lambda s: float(s[Y_c[s.index] == 1].sum()))))  # 구간별 거래·사기 요약
    display(seg.round(4))  # 구간별 요약 출력
    # 손익분기: 추가인증 1건 비용 c_auth가 얼마 이하일 때 3단계가 2단계보다 낮은 비용인지(나머지 가정 고정)
    lo, hi = 0.0, 50.0  # 이분 탐색 구간(추가 인증 비용 USD)
    f = lambda ca: three_tier(S_c, Y_c, A_c, grid, grid, dict(ASSUME_P, c_auth=ca))[0] - cost_two  # 3단계-2단계 비용 차
    if f(hi) < 0:  # 상한에서도 3단계가 유리한 경우
        be = float("inf")  # 손익분기 없음(항상 유리)
    elif f(lo) >= 0:  # 비용 0에서도 3단계가 유리하지 않은 경우
        be = 0.0  # 손익분기 0
    else:  # 손익분기가 구간 내부에 있는 경우
        for _ in range(30):  # 이분 탐색 30회 반복
            mid = (lo + hi) / 2  # 구간 중앙값
            if f(mid) < 0:  # 중앙값에서 3단계가 유리하면
                lo = mid  # 하한을 올림
            else:  # 3단계가 유리하지 않으면
                hi = mid  # 상한을 내림
        be = (lo + hi) / 2  # 손익분기 비용 추정
    print(f"손익분기 추가인증 1건 비용(다른 가정 고정): ${be:.2f}  ← 이 값보다 추가인증 비용이 낮으면 3단계 정책이 유리")  # 손익분기 출력
    rows = []  # 가정 조합별 결과 행 목록
    for sa in (0.4, 0.7, 0.9):  # 방어 성공률 후보
        for fr in (0.0, 1.0, 3.0):  # 정상 고객 불편 비용 후보
            c3 = three_tier(S_c, Y_c, A_c, grid, grid, dict(ASSUME_P, s_auth=sa, friction=fr))[0]  # 조합별 3단계 최소 비용
            rows.append(dict(방어성공률=sa, 정상고객불편비용=fr, 비용_3단계=c3, 비용_2단계=cost_two,
                             변화=1 - c3 / cost_two))  # 결과 행 추가
    display(pd.DataFrame(rows).round(4))  # 조합별 결과 출력
    add_verdict("tier_policy", "3단계(자동 승인/추가 인증/수동 심사) 정책 시뮬레이션", "분석·보고", "수행 완료(가정 기반)",
                f"가정(c_auth ${ASSUME_P['c_auth']}, 불편 ${ASSUME_P['friction']}, 방어성공 {ASSUME_P['s_auth']:.0%}) 하에서 "
                f"2단계 대비 비용 {1-cost3/cost_two:+.2%}; 손익분기 추가인증 비용 ${be:.2f}. 가정값은 운영 확인 필요", be=be)  # 판정 기록

2단계 최적: 알림 90,362건, 비용 $967,664
3단계 최적: 수동심사(≥0.075) 45,104건 + 추가인증(0.013~0.075) 119,092건, 비용 $851,213 → 2단계 대비 +12.03%


,거래수,사기건수,사기율,사기금액
구간,,,,
고위험(수동심사),45699,15416,0.3373,2181488.499
저위험(자동승인),425489,1558,0.0037,227151.013
중위험(추가인증),119352,3689,0.0309,675205.348


손익분기 추가인증 1건 비용(다른 가정 고정): $2.46  ← 이 값보다 추가인증 비용이 낮으면 3단계 정책이 유리


,방어성공률,정상고객불편비용,비용_3단계,비용_2단계,변화
0,0.4,0.0,893507.2686,967664.077,0.0766
1,0.4,1.0,962606.8302,967664.077,0.0052
2,0.4,3.0,968814.9200,967664.077,-0.0012
3,0.7,0.0,724949.7286,967664.077,0.2508
4,0.7,1.0,851212.6833,967664.077,0.1203
5,0.7,3.0,967461.0485,967664.077,0.0002
6,0.9,0.0,523627.4019,967664.077,0.4589
7,0.9,1.0,676421.4019,967664.077,0.3010
8,0.9,3.0,909362.0450,967664.077,0.0603


▶ 3단계(자동 승인/추가 인증/수동 심사) 정책 시뮬레이션: **수행 완료(가정 기반)** — 가정(c_auth $0.5, 불편 $1.0, 방어성공 70%) 하에서 2단계 대비 비용 +12.03%; 손익분기 추가인증 비용 $2.46. 가정값은 운영 확인 필요


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 룰 선처리 + 모델 후처리]</b> 룰 선처리 + AI 후처리 — 심사 현장 질문 5: "룰로 먼저 처리하고 나머지를 AI로 넘기는 시도는?"</div>

**방법**: 각 fold에서 **학습 구간 월만으로** 얕은 의사결정나무(깊이 4, 리프 최소 500건)를 학습하고, 학습 구간에서 **정밀도 90% 이상·사기 20건 이상**인 리프를 "룰"로 채택합니다. 검증 월에서는 룰에 걸린 거래를 즉시 알림으로 보내고, 나머지만 모델 점수로 판단합니다. (트리 입력은 UID 집계·빈도 같은 전역 통계 피처를 뺀 변수만 사용 — 누수 방지)

**평가**: 전체 Recall을 같은 수준(80%)으로 맞췄을 때 **필요한 알림 건수**가 줄어드는지 비교합니다. 룰 선처리 판정 기준: 알림 3% 이상 감소 & Recall 손실 없음 → 채택.

In [31]:
# 🟩 [수인 개선 · 룰 선처리 + 모델 후처리] 룰(얕은 트리 리프) 선처리 + 모델 후처리
from sklearn.tree import DecisionTreeClassifier  # 얕은 의사결정나무(룰 추출용)
if RUN["rule_prefilter"] and "최종 모델" in OOF:  # 룰 실험 스위치가 켜져 있고 최종 모델 OOF가 있는 경우
    TREE_COLS = [n for n in TEAM_FEATURES if n not in set(CROSS_NAMES)]  # 전역 통계 피처를 제외한 트리 입력 변수
    ti = [COL[n] for n in TREE_COLS]  # 트리 입력 변수의 열 위치
    Xr = np.nan_to_num(X_TEAM[:, ti], nan=-1.0)  # 결측을 -1로 대체한 트리 입력 행렬
    S = OOF["최종 모델"]  # 최종 모델 OOF 점수
    hit = np.zeros(len(S), dtype=bool)  # 검증 월에서 룰에 적중한 거래 표시
    rule_list = []  # fold별 룰 요약 목록
    for f in range(N_SPLITS):  # fold별 룰 학습·적용
        tr = np.where(FOLD_OF_ROW != f)[0]  # 학습 구간 행 위치
        va = FOLD_IDX[f]  # 검증 월 행 위치
        dtc = DecisionTreeClassifier(max_depth=4, min_samples_leaf=500, random_state=0)  # 얕은 트리 설정
        dtc.fit(Xr[tr], y_all[tr])  # 학습 구간으로 트리 학습
        leaf_tr = dtc.apply(Xr[tr])  # 학습 행의 리프 번호
        df_l = pd.DataFrame({"leaf": leaf_tr, "y": y_all[tr]})  # 학습 행의 리프 번호와 정답
        df_l = df_l.groupby("leaf")["y"].agg(["sum", "mean", "size"])  # 리프별 사기 건수·정밀도·크기
        good = df_l[(df_l["mean"] >= 0.90) & (df_l["sum"] >= 20)].index.values  # 정밀도 90%·사기 20건 이상 리프
        hit[va] = np.isin(dtc.apply(Xr[va]), good)  # 검증 월 거래 중 룰 리프에 들어간 거래 표시
        fraud_rate = float(y_all[va][hit[va]].mean()) if hit[va].any() else np.nan  # 적중 건의 실제 사기율
        rule_list.append((f, len(good), int(hit[va].sum()), fraud_rate))  # fold별 요약 추가
    rule_cols = ["fold", "채택된 룰(리프) 수", "검증월 룰 적중 건수", "적중 건의 실제 사기율"]  # 요약표 열 이름
    display(pd.DataFrame(rule_list, columns=rule_cols).round(4))  # fold별 룰 요약 출력
    R = 0.80  # 목표 전체 Recall
    Fz = y_all[ok].sum()  # 전체 사기 건수
    def alerts_for_recall(score_mask_valid, pre_flag):  # 목표 Recall 달성에 필요한 알림 건수 계산
        # pre_flag 거래는 항상 알림. 나머지는 점수 내림차순으로 전체 Recall이 R이 될 때까지 추가
        tp0 = int(y_all[pre_flag & ok].sum())  # 룰로 먼저 잡은 사기 건수
        need = int(math.ceil(R * Fz)) - tp0  # 모델로 추가 확보해야 할 사기 건수
        rest = np.where(ok & ~pre_flag)[0]  # 룰에 걸리지 않은 평가 행
        if need <= 0:  # 룰만으로 목표 Recall을 달성한 경우
            return int((pre_flag & ok).sum())  # 룰 알림 건수만 반환
        o = rest[np.argsort(-S[rest], kind="stable")]  # 남은 행을 모델 점수 내림차순으로 정렬
        cs = np.cumsum(y_all[o])  # 정렬 순서별 사기 누적 건수
        k = int(np.searchsorted(cs, need)) + 1  # need건을 확보하는 데 필요한 모델 알림 건수
        return int((pre_flag & ok).sum()) + k  # 룰 알림 + 모델 알림 합계 반환
    n0 = alerts_for_recall(None, np.zeros(len(S), dtype=bool))  # 모델만 사용할 때 필요한 알림 건수
    n1 = alerts_for_recall(None, hit)  # 룰 선처리 + 모델일 때 필요한 알림 건수
    red = 1 - n1 / n0  # 알림 건수 감소율
    print(f"Recall {R:.0%} 달성에 필요한 알림: 모델만 {n0:,}건 → 룰 선처리+모델 {n1:,}건 ({red:+.2%} 변화)")  # 비교 출력
    prec_rule = y_all[hit & ok].mean() if (hit & ok).any() else float("nan")  # 룰 적중 거래의 실제 사기율
    print(f"룰 적중 전체: {int((hit&ok).sum()):,}건, 실제 사기율 {prec_rule:.1%}")  # 룰 적중 요약 출력
    add_verdict("rule_prefilter", "룰 선처리 + 모델 후처리", "룰 선처리", "채택" if red >= 0.03 else "기각",
                f"Recall 80% 기준 알림 {n0:,}→{n1:,} ({red:+.2%}); 룰 적중 {int((hit&ok).sum()):,}건 사기율 {prec_rule:.1%}",
                red=red)  # 판정 기록

,fold,채택된 룰(리프) 수,검증월 룰 적중 건수,적중 건의 실제 사기율
0,0,1,313,0.8658
1,1,1,367,0.9019
2,2,1,347,0.9510
3,3,1,326,0.9172
4,4,1,333,0.9730
5,5,1,164,0.9146


Recall 80% 달성에 필요한 알림: 모델만 63,316건 → 룰 선처리+모델 63,326건 (-0.02% 변화)
룰 적중 전체: 1,850건, 실제 사기율 92.2%
▶ 룰 선처리 + 모델 후처리: **기각** — Recall 80% 기준 알림 63,316→63,326 (-0.02%); 룰 적중 1,850건 사기율 92.2%


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 변수 선택 근거표]</b> 변수 선택 근거 — 심사 현장 질문 2: "변수 선택에 대한 충분한 설명"</div>

팀의 변수 선택은 4단계였지만 한 곳에 정리되어 있지 않았습니다. 이 노트북은 **각 단계에서 몇 개가 어떤 이유로 남고/빠졌는지**를 실행 결과로 자동 집계합니다(앞서 팀 피처 엔지니어링을 실행하며 기록한 `LOG`와 팀의 Time Consistency 결과 파일 사용).

In [32]:
# 🟩 [수인 개선 · 변수 선택 근거표] 변수 선택 단계별 근거표 (팀 단계의 수치는 위 팀 코드 실행 로그에서 가져옴)
steps = [("1. 원본 병합", LOG.get("입력 컬럼 수"), "transaction + identity 병합(isFraud 제외)"),
         ("2. 파생·집계 추가 후", LOG.get("UID·빈도 집계 후 컬럼 수"),
          "결측 지표·D 정규화·cents·범주 인코딩·빈도·UID 집계 추가 (🟦 팀)"),
         ("3. V 컬럼 중복 제거", LOG.get("V 컬럼 유지 수"),
          f"V {LOG.get('V 컬럼 수(전)')}개 중 |상관|>0.95 인 {LOG.get('V 상관 0.95+ 제거 수')}개 제거 (🟦 팀)"),
         ("4. Time Consistency 탈락 제거", len(TEAM_FEATURES),
          f"{TC_FAILED} 제거 → 최종 {len(TEAM_FEATURES)}개 (🟦 팀)")]  # 단계별 (단계, 남은 변수 수, 근거) 목록
display(pd.DataFrame(steps, columns=["단계", "남은 변수 수(참고)", "근거"]))  # 단계별 근거표 출력
if TEAM_TC.exists():  # 팀 Time Consistency 결과 파일이 있는 경우
    tc = pd.read_csv(TEAM_TC)  # 결과 파일 읽기
    print("팀 Time Consistency 결과 파일(상위 일부):")  # 안내 출력
    display(tc.head(12))  # 상위 12행 출력
print(f"🟩 수인 추가 후보: 속도 피처 {len(VEL_NAMES)}개, 시간 피처 {len(TIME_NAMES)}개 — 채택 여부는 판정표 참조")  # 추가 후보 수 출력

,단계,남은 변수 수(참고),근거
0,1. 원본 병합,432,transaction + identity 병합(isFraud 제외)
1,2. 파생·집계 추가 후,484,결측 지표·D 정규화·cents·범주 인코딩·빈도·UID 집계 추가 (🟦 팀)
2,3. V 컬럼 중복 제거,181,V 339개 중 |상관|>0.95 인 158개 제거 (🟦 팀)
3,4. Time Consistency 탈락 제거,321,"['D1_was_missing', 'V286', 'V107', 'V305'] 제거 → 최종 321개 (🟦 팀)"


팀 Time Consistency 결과 파일(상위 일부):


,feature,train_auc,val_auc,pass
0,cents,0.702352,0.709406,True
1,V10,0.625522,0.693536,True
2,C12_uid_mean,0.665494,0.693163,True
3,C4_uid_mean,0.693774,0.692872,True
4,C8_uid_mean,0.688912,0.690677,True
5,R_emaildomain,0.676415,0.690252,True
6,C4,0.687769,0.689804,True
7,V79,0.678150,0.689767,True
8,V200,0.674326,0.689596,True
9,C10_uid_mean,0.686997,0.688220,True


🟩 수인 추가 후보: 속도 피처 12개, 시간 피처 2개 — 채택 여부는 판정표 참조


<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · Test set 사용 방식]</b> Test set 사용 방식 — 심사 현장 질문 3: "Test set을 어떻게 분류해 사용했는가"</div>

아래 셀이 **실제 폴더의 파일을 열어** 확인한 사실과 분할 구조를 보여 줍니다.

In [33]:
# 🟩 [수인 개선 · Test set 사용 방식] 분할 구조 표 + test 파일 정보 (직접 확인)
tt = DATA / "test_transaction.csv"  # 대회 test 파일 경로
if tt.exists():  # test 파일이 존재하는 경우
    cols = pd.read_csv(tt, nrows=2).columns  # 헤더 확인을 위해 2행만 읽음
    print("test_transaction.csv 존재 | 'isFraud' 열 포함 여부:", "isFraud" in cols,
          "→ 정답이 없으면 성능 평가에 쓸 수 없음")  # 정답 열 유무 출력
else:  # test 파일이 없는 경우
    print("test_transaction.csv 없음")  # 파일 없음 출력
sp_tbl = pd.DataFrame({"fold": list(range(N_SPLITS)),
                       "검증 월": [fold_to_month[f] for f in range(N_SPLITS)],
                       "검증 거래 수": [len(FOLD_IDX[f]) for f in range(N_SPLITS)],
                       "검증 사기율": [y_all[FOLD_IDX[f]].mean() for f in range(N_SPLITS)],
                       "학습 거래 수": [int((FOLD_OF_ROW != f).sum()) for f in range(N_SPLITS)]})  # fold별 분할 요약 표
sp_tbl = sp_tbl.set_index("fold")  # fold 번호를 인덱스로 지정
display(sp_tbl.round(4))  # 분할 요약 출력

test_transaction.csv 존재 | 'isFraud' 열 포함 여부: False → 정답이 없으면 성능 평가에 쓸 수 없음


,검증 월,검증 거래 수,검증 사기율,학습 거래 수
fold,,,,
0,12,137321,0.0259,453219
1,15,101632,0.0395,488908
2,13,92585,0.0400,497955
3,17,89326,0.0349,501214
4,14,86021,0.0401,504519
5,16,83655,0.0338,506885


<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> Test set 사용 방식 설명</div>

- 이 프로젝트는 **정답(`isFraud`)이 있는 `train_*.csv`만** 평가에 사용합니다. 위 셀의 확인 결과처럼 대회 test 파일에 정답이 없으면 성능을 계산할 수 없습니다.
- 대신 **월 단위 GroupKFold(6)** 로 데이터를 나눠, 각 fold의 **검증 월이 "test 역할"** 을 합니다. 같은 월의 거래는 학습과 검증에 동시에 들어가지 않습니다.
- **한계(🟩 이 노트북이 바로잡은 부분)**: 팀 코드에서는 검증 월이 early stopping에도 쓰였고(내부 검증 기반 early stopping 항목), 통계 피처가 전체 데이터로 계산되었으며(학습 구간 기준 피처 생성 항목), 임계치도 같은 OOF에서 골랐습니다(임계치 통합표 항목). 따라서 팀 방식의 검증 월은 **완전한 test가 아니었습니다**. 이 노트북은 early stopping을 학습 구간 안 내부 검증으로, 피처 통계를 학습 구간으로 제한해 검증 월을 평가에만 사용합니다. 단, 임계치는 여전히 전체 OOF에서 정하므로 **운영 성능의 완전한 독립 추정은 아니며**(월별 안정성 점검으로 보완), 이를 한계로 명시합니다.

<div style="border-left:6px solid #2ca02c;background:#eef8ee;padding:8px 14px;margin:4px 0"><b>🟩 [수인 개선 · 최종 판정표]</b> 모든 개선 항목의 채택/기각 결과 — 위에서 실행한 숫자를 사전 기준에 적용해 자동 생성</div>

판정 기준은 이 노트북 앞부분(채택/기각 기준)에서 결과를 보기 전에 고정한 것입니다. **평가 타당성 수정** 항목은 성능과 무관하게 채택하고, **성능 개선 후보·경량화·룰 선처리** 항목은 수치 기준으로 채택/기각하며, **분석·보고** 항목은 분석 결과를 보고합니다.
**"개선 시도, 기각"** 으로 표시된 항목은 성능을 올리지 못한 것이며 그대로 기록합니다.

In [35]:
# 🟩 [수인 개선 · 최종 판정표] 자동 생성
# 판정 id → 표에 표시할 항목 이름
DISPLAY = {"fold_features": "미래 정보 혼입 제거(학습 구간 기준 피처 생성)",
           "inner_early_stopping": "내부 검증 기반 early stopping",
           "calibration": "예측 확률 교차 보정",
           "velocity": "과거 이력 기반 속도 피처",
           "time_add": "상대 시각·요일 피처 추가",
           "time_drop": "시간 의존 피처 제거 영향 분석",
           "lgb": "LightGBM 단독",
           "ensemble": "XGBoost+LightGBM 평균 앙상블",
           "tuning": "하이퍼파라미터 재검토",
           "explainability": "변수 의미 사전·사유 코드·V 변수 제외 모델",
           "region_dependence": "지역·결제환경 의존 피처 제거 실험",
           "hypothesis_link": "가설과 모델 근거 연결(접속·기기 변수군 제거 실험)",
           "latency_topk": "추론 지연시간·피처 축소(Top-K)",
           "threshold": "임계치 통합표·권장 임계치",
           "sensitivity": "비용 가정 민감도 분석",
           "tier_policy": "3단계(자동 승인/추가 인증/수동 심사) 정책 시뮬레이션",
           "rule_prefilter": "룰 선처리 + 모델 후처리"}
ORDER = list(DISPLAY)  # 표 출력 순서(판정 id 목록)
def label(v):  # 판정 문자열을 화면 표시용 라벨로 변환
    return "개선 시도, 기각" if v == "기각" else v  # 기각은 '개선 시도, 기각'으로 표시
def kind_of(v):  # 판정 기록에서 판정 종류 문자열을 꺼냄
    return v.get("판정 종류", v.get("유형"))  # 저장 키 이름(판정 종류/유형) 모두 허용
vt = pd.DataFrame([{"항목": DISPLAY[k], "판정 종류": kind_of(VERDICTS[k]), "판정": VERDICTS[k]["판정"], "근거": VERDICTS[k]["근거"]}
                   for k in ORDER if k in VERDICTS], columns=["항목", "판정 종류", "판정", "근거"])  # 판정 id로 조회해 표 구성
if len(vt):  # 기록된 판정이 있는 경우
    vt["판정"] = vt["판정"].map(label)  # 판정 라벨 변환
if "FINAL_OK" in globals() and not FINAL_OK:  # 최종 모델이 누수 제거 기준보다 낮아 미재현인 경우
    is_cand = vt["판정 종류"] == "성능 개선 후보"  # 성능 개선 후보 행 마스크
    note = " (단, 공식 설정 6-fold에서는 누수 제거 기준 모델보다 낮아 미재현)"  # 미재현 표시 문구
    vt.loc[is_cand, "판정"] = vt.loc[is_cand, "판정"].map(lambda s: s + note if s == "채택" else s)  # 채택 항목에 표시 추가
pd.set_option("display.max_colwidth", 200)  # 근거 열이 잘리지 않도록 출력 폭 확장
display(vt)  # 판정표 출력
vt.to_csv(OUT / "최종_판정표.csv", index=False, encoding="utf-8-sig")  # 판정표 CSV 저장
def to_md(df):  # tabulate 없이 DataFrame을 markdown 표 문자열로 변환
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")  # 셀 안의 | 와 줄바꿈이 표 문법과 충돌하지 않게 처리
    head = "| " + " | ".join(map(esc, df.columns)) + " |"  # 머리글 행
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"  # 구분선 행
    rows = ["| " + " | ".join(map(esc, r)) + " |" for r in df.itertuples(index=False)]  # 데이터 행
    return "\n".join([head, sep, *rows])  # 줄 단위로 결합
vt_text = to_md(vt)  # 외부 패키지 없이 markdown 변환
(OUT / "최종_판정표.md").write_text(vt_text, encoding="utf-8")  # 판정표 markdown 파일 저장
miss = [DISPLAY[k] for k in ORDER if k not in VERDICTS]  # 아직 실행되지 않은 항목 이름
print("아직 실행하지 않았거나 RUN 스위치가 꺼진 항목:", miss or "없음")  # 미실행 항목 출력

,항목,판정 종류,판정,근거
0,미래 정보 혼입 제거(학습 구간 기준 피처 생성),평가 타당성 수정,채택(평가 타당성 수정),"GroupKFold 스크리닝: PR 0.7065→0.6493 (Δ-0.0572), ROC Δ-0.0146 | 순방향: PR 0.5684→0.5081 (Δ-0.0603)"
1,내부 검증 기반 early stopping,평가 타당성 수정,채택(평가 타당성 수정),"스크리닝 3 fold: PR 0.6493→0.6153 (Δ-0.0340), ROC Δ-0.0111. (내부 검증 모델은 학습 구간 마지막 1개월을 학습에 덜 사용하므로 하락분에 이 효과도 일부 포함됩니다.)"
2,예측 확률 교차 보정,평가 타당성 수정,채택(평가 타당성 수정),"Brier 0.01850→0.01853, ECE 0.00624→0.00088, 보정 후 비용최소 임계치 0.0302"
3,과거 이력 기반 속도 피처,성능 개선 후보,"개선 시도, 기각","PR 0.6153→0.5903 (Δ-0.0250), ROC Δ-0.0075, PR 개선 fold 0% (기준: Δ≥+0.003 & fold 2/3)"
4,상대 시각·요일 피처 추가,성능 개선 후보,"개선 시도, 기각","PR Δ-0.0194, ROC Δ-0.0026, PR 개선 fold 0%"
5,시간 의존 피처 제거 영향 분석,분석·보고,수행 완료,"제거 피처 ['D4', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15']: PR Δ-0.0364, ROC Δ-0.0138 → 시간 정보를 유지한 근거로 사용"
6,LightGBM 단독,성능 개선 후보,"개선 시도, 기각","PR Δ-0.0104, ROC Δ+0.0015, PR 개선 fold 33%"
7,XGBoost+LightGBM 평균 앙상블,성능 개선 후보,채택,"PR Δ+0.0178, ROC Δ+0.0084, PR 개선 fold 100%"
8,하이퍼파라미터 재검토,성능 개선 후보,"개선 시도, 기각","최선 후보 regularized: PR Δ+0.0029, ROC Δ+0.0000, PR 개선 fold 33% (후보 5개 중 선택 → 최종 6-fold에서 재확인)"
9,변수 의미 사전·사유 코드·V 변수 제외 모델,분석·보고,수행 완료,"V 178개 제외 시 PR Δ-0.0467, ROC Δ-0.0046. 상위 300건 중 해석 가능한 사유 제시 비율 100%"


아직 실행하지 않았거나 RUN 스위치가 꺼진 항목: 없음


<div style="border-left:6px solid #888;background:#f5f5f5;padding:8px 14px;margin:4px 0"><b>⬜ [안내]</b> 팀 기존 작업과 수인 개선의 경계 (요약)</div>

| 구분 | 내용 |
|---|---|
| 🟦 **팀 기존 (그대로 유지, 수인의 성과로 주장하지 않음)** | 피처 엔지니어링 전 단계(결측 지표·D 정규화·UID 집계·빈도 인코딩·V 상관 제거) · Time Consistency Test · GroupKFold(월) · XGBoost 설정(깊이 12, eta 0.02, subsample 0.8, colsample 0.4) · 최종 OOF · 임계치·비용 그림 |
| 🟨 **이 노트북 범위 밖** | Tableau 대시보드 D1~D6는 모델링이 아니라 시각화 산출물이므로 이 노트북에서 다루지 않습니다. D1~D6의 화면 기획과 제작은 수인이 전담했고, 모델 성능·임계치·비용 수치는 팀 모델링 결과를 사용했습니다. |
| 🟩 **수인 개선** | 미래 정보 혼입 제거(학습 구간 기준 피처 생성)·순방향 검증 · 내부 검증 기반 early stopping · 예측 확률 교차 보정 · 과거 이력 기반 속도 피처 · 상대 시각·요일 피처 실험과 시간 의존 피처 제거 영향 분석 · LightGBM 단독·앙상블 · 하이퍼파라미터 재검토 · 변수 의미 사전·사유 코드 · 추론 지연시간·피처 축소(Top-K) · 지역·결제환경 의존 피처 제거 실험 · 룰 선처리 + 모델 후처리 · 임계치 통합표·비용 가정 민감도 분석·3단계 정책 시뮬레이션 · 변수 선택 근거표·Test set 사용 방식·가설과 모델 근거 연결 |

**출처 표기**: UID 기반 "magic" 피처 아이디어는 Kaggle IEEE-CIS 대회 1위 솔루션(이 프로젝트 `notebooks/xgb-fraud-with-magic-0-9600`의 헤더에 그렇게 적혀 있음)에서 온 것이며, 팀이 이를 참고해 적용했습니다. 이 노트북의 🟩 항목은 그 아이디어 자체가 아니라 **그 위의 검증 방식 수정과 추가 실험**입니다.